# **Time Series Forecasting using Autoformer, Case Study in Pelabuhan Ratu**

#### **Import Libraries**

In [ ]:
# Library dasar untuk pengolahan data dan manipulasi array
import math
import numpy as np
import pandas as pd

# Library untuk visualisasi
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import cm

# Library untuk data science dan evaluasi model
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Library untuk implementasi model dengan PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.cuda.amp import GradScaler, autocast

#### **Data Preparation/Pre-Processing**

In [ ]:
# Step 1: Load and preprocess the data
data = pd.read_csv('/Users/mac/Desktop/.../Wave_Forecasting/data/TOTAL_WAVE_PR-C1_HsTpDir.txt', sep='\t', header=None)
data.columns = ['Year', 'Month', 'Day', 'Hour', 'Minute', 'Second', 'Hs', 'Tp', 'Dir']

In [ ]:
# Step 2: Combine date columns and set the index
data['Timestamp'] = pd.to_datetime(data[['Year', 'Month', 'Day', 'Hour', 'Minute']])
data.set_index('Timestamp', inplace=True)

In [ ]:
# Pratinjau data
data.head()

In [ ]:
#Eksplorasi data (memeriksa tipe data dan nilai kosong)
print(data.info())

In [ ]:
# Step 3: Handle missing values
data = data.dropna()

In [ ]:
# Scale the data for 'Hs' only
# Menggunakan MinMaxScaler untuk normalisasi
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data[['Hs']])
scaled_data = pd.DataFrame(scaled_data, columns=['Hs'], index=data.index)

In [ ]:
# Impor gaya tambahan untuk matplotlib
plt.style.use('ggplot')  # Menggunakan gaya "ggplot" untuk tampilan yang lebih profesional

# Warna pink untuk plot
color = '#ff69b4'  # Warna pink hex code

# Visualisasi Series: Plotting Scaled Wave Height saja
fig, ax = plt.subplots(figsize=(15, 6))

# Plotting Scaled Wave Height dengan gaya pink
ax.plot(scaled_data.index, scaled_data['Hs'], label='Scaled Wave Height', color=color, linewidth=1.5, linestyle='-')
ax.fill_between(scaled_data.index, scaled_data['Hs'], color=color, alpha=0.1)  # Transparansi di bawah kurva
ax.set_title('Scaled Wave Height Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Time (Year)', fontsize=12)
ax.set_ylabel('Scaled Height', fontsize=12)
ax.legend()
ax.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.7)  # Grid dengan transparansi

# Format sumbu x sebagai tahun
ax.xaxis.set_major_locator(mdates.YearLocator())  # Menampilkan label tahun di sumbu x
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))  # Format tahun di sumbu x

# Menampilkan grafik
plt.show()



In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta

# Fungsi untuk membagi data menjadi train, validation, dan test set berdasarkan proporsi
def split_data(data, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15):
    total_data = len(data)
    
    # Hitung ukuran train, val, dan test set berdasarkan rasio yang diberikan
    train_size = int(train_ratio * total_data)
    val_size = int(val_ratio * total_data)
    test_size = int(test_ratio * total_data)
    
    # Buat indeks untuk train, val, dan test
    indices = np.arange(total_data)
    train_indices = indices[:train_size]
    val_indices = indices[train_size:train_size + val_size]
    test_indices = indices[train_size + val_size:train_size + val_size + test_size]
    
    return train_indices, val_indices, test_indices

# Mendefinisikan skenario berdasarkan tanggal dinamis
def get_scenarios(end_date='2024-01-15'):
    end_date = datetime.strptime(end_date, '%Y-%m-%d')
    
    return {
        '3 months': (end_date - relativedelta(months=4)),  # Tanggal mulai untuk skenario 3 bulan
        '6 months': (end_date - relativedelta(months=7)),  # Tanggal mulai untuk skenario 6 bulan
        '8 months': (end_date - relativedelta(months=9))   # Tanggal mulai untuk skenario 8 bulan
    }

# Step 4: Apply the split for each scenario and return the data (using scaled_data)
def split_by_scenario(scaled_data, scenarios, end_date='2024-01-15'):
    results = {}
    
    for scenario, start_date in scenarios.items():
        # Filter scaled_data berdasarkan periode waktu
        data_period = scaled_data[(scaled_data.index >= start_date) & (scaled_data.index < end_date)]
        
        # Bagi data menggunakan proporsi 75% train, 12.5% validation, dan 12.5% test
        train_indices, val_indices, test_indices = split_data(data_period)
        
        # Mengambil data berdasarkan indeks
        train_data = data_period.iloc[train_indices]
        val_data = data_period.iloc[val_indices]
        test_data = data_period.iloc[test_indices]
        
        # Simpan hasilnya dalam dictionary
        results[scenario] = {
            'train': train_data,
            'val': val_data,
            'test': test_data
        }
        
        # Cetak ukuran setiap set
        print(f"{scenario} - Train: {len(train_data)}, Validation: {len(val_data)}, Test: {len(test_data)}")
    
    return results

# Contoh penggunaan
# Pastikan untuk mengganti 'scaled_data' dengan DataFrame Anda
# scaled_data = pd.DataFrame(...)  # Misalnya, isi DataFrame Anda di sini
scenarios = get_scenarios()
split_results = split_by_scenario(scaled_data, scenarios)

# Akses hasil pembagian data untuk setiap skenario
train_data_3_months = split_results['3 months']['train']
val_data_3_months = split_results['3 months']['val']
test_data_3_months = split_results['3 months']['test']

train_data_6_months = split_results['6 months']['train']
val_data_6_months = split_results['6 months']['val']
test_data_6_months = split_results['6 months']['test']

train_data_8_months = split_results['8 months']['train']
val_data_8_months = split_results['8 months']['val']
test_data_8_months = split_results['8 months']['test']


In [ ]:
import matplotlib.pyplot as plt

def plot_data_with_raw(ax, raw_data, train_data, val_data, test_data, title, ylabel='Hs (Wave Height)', xlabel='Date'):
    """
    Memplot data raw, training, validation, dan test pada sumbu yang diberikan.
    
    :param ax: Axes untuk plot
    :param raw_data: Data keseluruhan (raw data)
    :param train_data: Data training
    :param val_data: Data validasi
    :param test_data: Data test
    :param title: Judul plot
    :param ylabel: Label untuk sumbu y
    :param xlabel: Label untuk sumbu x
    """
    # Plot raw data (seluruh data)
    ax.plot(raw_data.index, raw_data['Hs'], color='gray', label='Raw Data', alpha=0.4)

    # Plot data training
    ax.plot(train_data.index, train_data['Hs'], color='blue', label='Training Set')

    # Plot data validasi
    ax.plot(val_data.index, val_data['Hs'], color='orange', label='Validation Set')

    # Plot data test
    ax.plot(test_data.index, test_data['Hs'], color='green', label='Test Set')

    # Tambahkan detail plot
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel(xlabel)
    ax.legend()
    ax.grid(True)

# Misalkan `raw_data` adalah dataset asli sebelum dibagi
raw_data = scaled_data  # Gantilah sesuai data yang Anda gunakan

# Plot data untuk setiap skenario
fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)

# Plot untuk 3-Month Training Split
plot_data_with_raw(axes[0], raw_data, train_data_3_months, val_data_3_months, test_data_3_months, '3-Month Training Split')

# Plot untuk 6-Month Training Split
plot_data_with_raw(axes[1], raw_data, train_data_6_months, val_data_6_months, test_data_6_months, '6-Month Training Split')

# Plot untuk 8-Month Training Split
plot_data_with_raw(axes[2], raw_data, train_data_8_months, val_data_8_months, test_data_8_months, '8-Month Training Split')

plt.xlabel('Date')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

def plot_data(ax, train_data, val_data, test_data, title, ylabel='Wave Height (Hs)', xlabel='Date'):
    """
    Memplot data training, validasi, dan test pada sumbu yang diberikan.
    
    :param ax: Axes untuk plot
    :param train_data: Data training
    :param val_data: Data validasi
    :param test_data: Data test
    :param title: Judul plot
    :param ylabel: Label untuk sumbu y
    :param xlabel: Label untuk sumbu x
    """
    # Plot data training
    ax.plot(train_data.index, train_data['Hs'], color='blue', label='Train')
    
    # Plot data validasi
    ax.plot(val_data.index, val_data['Hs'], color='orange', label='Validation')
    
    # Plot data test
    ax.plot(test_data.index, test_data['Hs'], color='green', label='Test')
    
    # Tambahkan detail plot
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel(xlabel)
    ax.legend()
    ax.grid(True)

# Plot data untuk setiap skenario
fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)

# Plot untuk 3-Month Training Split
plot_data(axes[0], train_data_3_months, val_data_3_months, test_data_3_months, '3-Month Training Split')

# Plot untuk 6-Month Training Split
plot_data(axes[1], train_data_6_months, val_data_6_months, test_data_6_months, '6-Month Training Split')

# Plot untuk 8-Month Training Split
plot_data(axes[2], train_data_8_months, val_data_8_months, test_data_8_months, '8-Month Training Split')

plt.xlabel('Date')
plt.tight_layout()
plt.show()



In [ ]:
# Plot setiap bagian data dengan warna berbeda
plt.figure(figsize=(12, 6))

# Plot data 8-month training set
plt.plot(train_data_8_months, color='blue', label='Train Data (8 months)')

# Plot data 6-month training set
plt.plot(train_data_6_months, color='cyan', linestyle='--', label='Train Data (6 months)')

# Plot data 3-month training set
plt.plot(train_data_3_months, color='black', linestyle=':', label='Train Data (3 months)')

# Plot validation set (3 months)
plt.plot(val_data_3_months, color='orange', label='Validation Data (3 months)')

# Plot validation set (6 months)
plt.plot(val_data_6_months, color='green', linestyle='-.', label='Validation Data (6 months)')

# Plot validation set (8 months)
plt.plot(val_data_8_months, color='purple', linestyle='-', label='Validation Data (8 months)')

# Plot test set (3 months)
plt.plot(test_data_3_months, color='red', label='Test Data (3 months)')

# Plot test set (6 months)
plt.plot(test_data_6_months, color='magenta', linestyle='--', label='Test Data (6 months)')

# Plot test set (8 months)
plt.plot(test_data_8_months, color='brown', linestyle=':', label='Test Data (8 months)')

# Tambahkan label dan judul
plt.xlabel('Index')
plt.ylabel('Scaled Data (Hs)')
plt.title('Train, Validation, and Test Split')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Fungsi untuk membuat dan menampilkan plot
def plot_data(data, title, color, ylabel='Wave Height (Hs)', xlabel='Date'):
    plt.figure(figsize=(10, 6))
    plt.plot(data.index, data['Hs'], color=color)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(True)
    plt.show()

# Plot data untuk 8-Month Training, Validation, dan Test Data
plot_data(train_data_8_months, '8-Month Training Data', color='blue')
plot_data(val_data_8_months, '8-Month Validation Data', color='orange')
plot_data(test_data_8_months, '8-Month Test Data', color='red')

# Plot data untuk 6-Month Training, Validation, dan Test Data
plot_data(train_data_6_months, '6-Month Training Data', color='cyan')
plot_data(val_data_6_months, '6-Month Validation Data', color='green')
plot_data(test_data_6_months, '6-Month Test Data', color='magenta')

# Plot data untuk 3-Month Training, Validation, dan Test Data
plot_data(train_data_3_months, '3-Month Training Data', color='black')
plot_data(val_data_3_months, '3-Month Validation Data', color='purple')
plot_data(test_data_3_months, '3-Month Test Data', color='brown')



#### 24 Hours 

In [ ]:
# Step 5: Create a window-based dataset using scaled_data (3 Months - 24 jam)
def create_windows(data, window_size=16, forecast_horizon=224):
    X, y = [], []
    for i in range(len(data) - window_size - forecast_horizon + 1):
        X.append(data.iloc[i: i + window_size].values)
        y.append(data.iloc[i + window_size: i + window_size + forecast_horizon].values)
    return np.array(X), np.array(y)

In [ ]:
# Buat dataset berbasis window untuk setiap skenario (24 jam)
X_train_3, y_train_3 = create_windows(train_data_3_months)
X_train_6, y_train_6 = create_windows(train_data_6_months)
X_train_8, y_train_8 = create_windows(train_data_8_months)

X_val_3, y_val_3 = create_windows(val_data_3_months)
X_val_6, y_val_6 = create_windows(val_data_6_months)
X_val_8, y_val_8 = create_windows(val_data_8_months)

X_test_3, y_test_3 = create_windows(test_data_3_months)
X_test_6, y_test_6 = create_windows(test_data_6_months)
X_test_8, y_test_8 = create_windows(test_data_8_months)

# Cek hasilnya untuk melihat ukuran setiap dataset
print(f"Shape of X_train_3: {X_train_3.shape}, y_train_3: {y_train_3.shape}")
print(f"Shape of X_train_6: {X_train_6.shape}, y_train_6: {y_train_6.shape}")
print(f"Shape of X_train_8: {X_train_8.shape}, y_train_8: {y_train_8.shape}")
print(f"Shape of X_val_3: {X_val_3.shape}, y_val_3: {y_val_3.shape}")
print(f"Shape of X_val_6: {X_val_6.shape}, y_val_6: {y_val_6.shape}")
print(f"Shape of X_val_8: {X_val_8.shape}, y_val_8: {y_val_8.shape}")
print(f"Shape of X_test_3: {X_test_3.shape}, y_test_3: {y_test_3.shape}")
print(f"Shape of X_test_6: {X_test_6.shape}, y_test_6: {y_test_6.shape}")
print(f"Shape of X_test_8: {X_test_8.shape}, y_test_8: {y_test_8.shape}")


In [ ]:
# Step 6: Create DataLoaders
batch_size = 64
train_loader_3 = DataLoader(TensorDataset(torch.Tensor(X_train_3), torch.Tensor(y_train_3)), batch_size=batch_size, shuffle=True)
val_loader_3 = DataLoader(TensorDataset(torch.Tensor(X_val_3), torch.Tensor(y_val_3)), batch_size=batch_size, shuffle=False)
test_loader_3 = DataLoader(TensorDataset(torch.Tensor(X_test_3), torch.Tensor(y_test_3)), batch_size=batch_size, shuffle=False)

train_loader_6 = DataLoader(TensorDataset(torch.Tensor(X_train_6), torch.Tensor(y_train_6)), batch_size=batch_size, shuffle=True)
val_loader_6 = DataLoader(TensorDataset(torch.Tensor(X_val_6), torch.Tensor(y_val_6)), batch_size=batch_size, shuffle=False)
test_loader_6 = DataLoader(TensorDataset(torch.Tensor(X_test_6), torch.Tensor(y_test_6)), batch_size=batch_size, shuffle=False)

train_loader_8 = DataLoader(TensorDataset(torch.Tensor(X_train_8), torch.Tensor(y_train_8)), batch_size=batch_size, shuffle=True)
val_loader_8 = DataLoader(TensorDataset(torch.Tensor(X_val_8), torch.Tensor(y_val_8)), batch_size=batch_size, shuffle=False)
test_loader_8 = DataLoader(TensorDataset(torch.Tensor(X_test_8), torch.Tensor(y_test_8)), batch_size=batch_size, shuffle=False)

#### **AUTOFORMER**

#### ARSITEKTUR AUTOFORMER

In [ ]:
# Step 7: Define the Autoformer architecture
class DecoderLayer(nn.Module):
    """
    Autoformer decoder layer with the progressive decomposition architecture
    """
    def __init__(self, self_attention, cross_attention, d_model, c_out, d_ff=None,
                 moving_avg=25, dropout=0.3, activation="relu"):
        super(DecoderLayer, self).__init__()
        d_ff = d_ff or 4 * d_model
        self.self_attention = self_attention
        self.cross_attention = cross_attention
        self.conv1 = nn.Conv1d(in_channels=d_model, out_channels=d_ff, kernel_size=1, bias=False)
        self.conv2 = nn.Conv1d(in_channels=d_ff, out_channels=d_model, kernel_size=1, bias=False)
        self.decomp1 = series_decomp(moving_avg)
        self.decomp2 = series_decomp(moving_avg)
        self.decomp3 = series_decomp(moving_avg)
        self.dropout = nn.Dropout(dropout)
        self.projection = nn.Conv1d(in_channels=d_model, out_channels=c_out, kernel_size=3, stride=1, padding=1,
                                    padding_mode='circular', bias=False)
        self.activation = F.relu if activation == "relu" else F.gelu

    def forward(self, x, cross, x_mask=None, cross_mask=None):
        x = x + self.dropout(self.self_attention(
            x, x, x,
            attn_mask=x_mask
        )[0])
        x, trend1 = self.decomp1(x)
        x = x + self.dropout(self.cross_attention(
            x, cross, cross,
            attn_mask=cross_mask
        )[0])
        x, trend2 = self.decomp2(x)
        y = x
        y = self.dropout(self.activation(self.conv1(y.transpose(-1, 1))))
        y = self.dropout(self.conv2(y).transpose(-1, 1))
        x, trend3 = self.decomp3(x + y)

        residual_trend = trend1 + trend2 + trend3
        residual_trend = self.projection(residual_trend.permute(0, 2, 1)).transpose(1, 2)
        return x, residual_trend

class Decoder(nn.Module):
    """
    Autoformer encoder
    """
    def __init__(self, layers, norm_layer=None, projection=None):
        super(Decoder, self).__init__()
        self.layers = nn.ModuleList(layers)
        self.norm = norm_layer
        self.projection = projection

    def forward(self, x, cross, x_mask=None, cross_mask=None, trend=None):
        for layer in self.layers:
            x, residual_trend = layer(
                x, cross, x_mask=x_mask, cross_mask=cross_mask)
            trend = trend + residual_trend

        if self.norm is not None:
            x = self.norm(x)

        if self.projection is not None:
            x = self.projection(x)
        return x, trend


class EncoderLayer(nn.Module):
    """
    Autoformer encoder layer with the progressive decomposition architecture
    """
    def __init__(self, attention, d_model, d_ff=None, moving_avg=25, dropout=0.2, activation="relu"):
        super(EncoderLayer, self).__init__()
        d_ff = d_ff or 4 * d_model
        self.attention = attention
        self.conv1 = nn.Conv1d(in_channels=d_model, out_channels=d_ff, kernel_size=1, bias=False)
        self.conv2 = nn.Conv1d(in_channels=d_ff, out_channels=d_model, kernel_size=1, bias=False)
        self.decomp1 = series_decomp(moving_avg)
        self.decomp2 = series_decomp(moving_avg)
        self.dropout = nn.Dropout(dropout)
        self.activation = F.relu if activation == "relu" else F.gelu

    def forward(self, x, attn_mask=None):
        new_x, attn = self.attention(
            x, x, x,
            attn_mask=attn_mask
        )
        x = x + self.dropout(new_x)
        x, _ = self.decomp1(x)
        y = x
        y = self.dropout(self.activation(self.conv1(y.transpose(-1, 1))))
        y = self.dropout(self.conv2(y).transpose(-1, 1))
        res, _ = self.decomp2(x + y)
        return res, attn

class Encoder(nn.Module):
    """
    Autoformer encoder
    """
    def __init__(self, attn_layers, conv_layers=None, norm_layer=None):
        super(Encoder, self).__init__()
        self.attn_layers = nn.ModuleList(attn_layers)
        self.conv_layers = nn.ModuleList(conv_layers) if conv_layers is not None else None
        self.norm = norm_layer

    def forward(self, x, attn_mask=None):
        attns = []
        if self.conv_layers is not None:
            for attn_layer, conv_layer in zip(self.attn_layers, self.conv_layers):
                x, attn = attn_layer(x, attn_mask=attn_mask)
                x = conv_layer(x)
                attns.append(attn)
            x, attn = self.attn_layers[-1](x)
            attns.append(attn)
        else:
            for attn_layer in self.attn_layers:
                x, attn = attn_layer(x, attn_mask=attn_mask)
                attns.append(attn)

        if self.norm is not None:
            x = self.norm(x)

        return x, attns


class my_Layernorm(nn.Module):
    """
    Special designed layernorm for the seasonal part
    """
    def __init__(self, channels):
        super(my_Layernorm, self).__init__()
        self.layernorm = nn.LayerNorm(channels)

    def forward(self, x):
        x_hat = self.layernorm(x)
        bias = torch.mean(x_hat, dim=1).unsqueeze(1).repeat(1, x.shape[1], 1)
        return x_hat - bias

class TokenEmbedding(nn.Module):
    def __init__(self, c_in, d_model):
        super(TokenEmbedding, self).__init__()
        padding = 1 if torch.__version__ >= '1.5.0' else 2
        self.tokenConv = nn.Conv1d(in_channels=c_in, out_channels=d_model,
                                   kernel_size=3, padding=padding, padding_mode='circular', bias=False)
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='leaky_relu')

    def forward(self, x):
        x = self.tokenConv(x.permute(0, 2, 1)).transpose(1, 2)
        return x


class moving_avg(nn.Module):
    """
    Moving average block to highlight the trend of time series
    """
    def __init__(self, kernel_size, stride):
        super(moving_avg, self).__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=stride, padding=0)

    def forward(self, x):
        # padding on the both ends of time series
        front = x[:, 0:1, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        end = x[:, -1:, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        x = torch.cat([front, x, end], dim=1)
        x = self.avg(x.permute(0, 2, 1))
        x = x.permute(0, 2, 1)
        return x

class series_decomp(nn.Module):
    """
    Series decomposition block
    """
    def __init__(self, kernel_size):
        super(series_decomp, self).__init__()
        self.moving_avg = moving_avg(kernel_size, stride=1)

    def forward(self, x):
        moving_mean = self.moving_avg(x)
        res = x - moving_mean
        return res, moving_mean


class AutoCorrelation(nn.Module):
    """
    AutoCorrelation Mechanism with the following two phases:
    (1) period-based dependencies discovery
    (2) time delay aggregation
    This block can replace the self-attention family mechanism seamlessly.
    """
    def __init__(self, mask_flag=True, factor=1, scale=None, attention_dropout=0.2, output_attention=False):
        super(AutoCorrelation, self).__init__()
        self.factor = factor
        self.scale = scale
        self.mask_flag = mask_flag
        self.output_attention = output_attention
        self.dropout = nn.Dropout(attention_dropout)

    def time_delay_agg_training(self, values, corr):
        """
        SpeedUp version of Autocorrelation (a batch-normalization style design)
        This is for the training phase.
        """
        head = values.shape[1]
        channel = values.shape[2]
        length = values.shape[3]
        # find top k
        top_k = int(self.factor * math.log(length))
        mean_value = torch.mean(torch.mean(corr, dim=1), dim=1)
        index = torch.topk(torch.mean(mean_value, dim=0), top_k, dim=-1)[1]
        weights = torch.stack([mean_value[:, index[i]] for i in range(top_k)], dim=-1)
        # update corr
        tmp_corr = torch.softmax(weights, dim=-1)
        # aggregation
        tmp_values = values
        delays_agg = torch.zeros_like(values).float()
        for i in range(top_k):
            pattern = torch.roll(tmp_values, -int(index[i]), -1)
            delays_agg = delays_agg + pattern * \
                         (tmp_corr[:, i].unsqueeze(1).unsqueeze(1).unsqueeze(1).repeat(1, head, channel, length))
        return delays_agg

    def time_delay_agg_inference(self, values, corr):
        """
        SpeedUp version of Autocorrelation (a batch-normalization style design)
        This is for the inference phase.
        """
        batch = values.shape[0]
        head = values.shape[1]
        channel = values.shape[2]
        length = values.shape[3]
        # index init
        init_index = torch.arange(length).unsqueeze(0).unsqueeze(0).unsqueeze(0)\
            .repeat(batch, head, channel, 1).to(values.device)
        # find top k
        top_k = int(self.factor * math.log(length))
        mean_value = torch.mean(torch.mean(corr, dim=1), dim=1)
        weights, delay = torch.topk(mean_value, top_k, dim=-1)
        # update corr
        tmp_corr = torch.softmax(weights, dim=-1)
        # aggregation
        tmp_values = values.repeat(1, 1, 1, 2)
        delays_agg = torch.zeros_like(values).float()
        for i in range(top_k):
            tmp_delay = init_index + delay[:, i].unsqueeze(1).unsqueeze(1).unsqueeze(1).repeat(1, head, channel, length)
            pattern = torch.gather(tmp_values, dim=-1, index=tmp_delay)
            delays_agg = delays_agg + pattern * \
                         (tmp_corr[:, i].unsqueeze(1).unsqueeze(1).unsqueeze(1).repeat(1, head, channel, length))
        return delays_agg

    def time_delay_agg_full(self, values, corr):
        """
        Standard version of Autocorrelation
        """
        batch = values.shape[0]
        head = values.shape[1]
        channel = values.shape[2]
        length = values.shape[3]
        # index init
        init_index = torch.arange(length).unsqueeze(0).unsqueeze(0).unsqueeze(0)\
            .repeat(batch, head, channel, 1).to(values.device)
        # find top k
        top_k = int(self.factor * math.log(length))
        weights, delay = torch.topk(corr, top_k, dim=-1)
        # update corr
        tmp_corr = torch.softmax(weights, dim=-1)
        # aggregation
        tmp_values = values.repeat(1, 1, 1, 2)
        delays_agg = torch.zeros_like(values).float()
        for i in range(top_k):
            tmp_delay = init_index + delay[..., i].unsqueeze(-1)
            pattern = torch.gather(tmp_values, dim=-1, index=tmp_delay)
            delays_agg = delays_agg + pattern * (tmp_corr[..., i].unsqueeze(-1))
        return delays_agg

    def forward(self, queries, keys, values, attn_mask):
        # attn_mask - we dont actually use it
        B, L, H, E = queries.shape
        _, S, _, D = values.shape
        if L > S:
            zeros = torch.zeros_like(queries[:, :(L - S), :]).float()
            values = torch.cat([values, zeros], dim=1)
            keys = torch.cat([keys, zeros], dim=1)
        else:
            values = values[:, :L, :, :]
            keys = keys[:, :L, :, :]

        # period-based dependencies
        q_fft = torch.fft.rfft(queries.permute(0, 2, 3, 1).contiguous(), dim=-1)
        k_fft = torch.fft.rfft(keys.permute(0, 2, 3, 1).contiguous(), dim=-1)
        res = q_fft * torch.conj(k_fft)
        corr = torch.fft.irfft(res, dim=-1)

        # time delay agg
        if self.training:
            V = self.time_delay_agg_training(values.permute(0, 2, 3, 1).contiguous(), corr).permute(0, 3, 1, 2)
        else:
            V = self.time_delay_agg_inference(values.permute(0, 2, 3, 1).contiguous(), corr).permute(0, 3, 1, 2)

        if self.output_attention:
            return (V.contiguous(), corr.permute(0, 3, 1, 2))
        else:
            return (V.contiguous(), None)

class AutoCorrelationLayer(nn.Module):
    def __init__(self, correlation, d_model, n_heads, d_keys=None,
                 d_values=None):
        super(AutoCorrelationLayer, self).__init__()

        d_keys = d_keys or (d_model // n_heads)
        d_values = d_values or (d_model // n_heads)

        self.inner_correlation = correlation
        self.query_projection = nn.Linear(d_model, d_keys * n_heads)
        self.key_projection = nn.Linear(d_model, d_keys * n_heads)
        self.value_projection = nn.Linear(d_model, d_values * n_heads)
        self.out_projection = nn.Linear(d_values * n_heads, d_model)
        self.n_heads = n_heads

    def forward(self, queries, keys, values, attn_mask):
        B, L, _ = queries.shape
        _, S, _ = keys.shape
        H = self.n_heads

        queries = self.query_projection(queries).view(B, L, H, -1)
        keys = self.key_projection(keys).view(B, S, H, -1)
        values = self.value_projection(values).view(B, S, H, -1)

        out, attn = self.inner_correlation(
            queries,
            keys,
            values,
            attn_mask
        )
        out = out.view(B, L, -1)

        return self.out_projection(out), attn

In [ ]:
class Autoformer(nn.Module):
    def __init__(self, enc_in, dec_in, c_out, seq_len, label_len, out_len, 
                 d_model, n_heads, e_layers, d_layers, 
                 d_ff, moving_avg, dropout, factor, 
                 activation, output_attention=False):
        super(Autoformer, self).__init__()
        
        self.seq_len = seq_len
        self.label_len = label_len
        self.pred_len = out_len  # Menyimpan panjang output untuk prediksi
        self.output_attention = output_attention

        # Embedding layers
        self.enc_embedding = TokenEmbedding(enc_in, d_model)
        self.dec_embedding = TokenEmbedding(dec_in, d_model)

        # Encoder
        enc_layers = [EncoderLayer(
            AutoCorrelationLayer(
                AutoCorrelation(True, factor, attention_dropout=dropout, output_attention=output_attention),
                d_model, n_heads),
            d_model,
            d_ff,
            moving_avg=moving_avg,
            dropout=dropout,
            activation=activation
        ) for _ in range(e_layers)]
        self.encoder = Encoder(enc_layers, norm_layer=my_Layernorm(d_model))

        # Decoder
        dec_layers = [DecoderLayer(
            AutoCorrelationLayer(
                AutoCorrelation(False, factor, attention_dropout=dropout, output_attention=False),
                d_model, n_heads),
            AutoCorrelationLayer(
                AutoCorrelation(False, factor, attention_dropout=dropout, output_attention=False),
                d_model, n_heads),
            d_model,
            c_out,
            d_ff,
            moving_avg=moving_avg,
            dropout=dropout,
            activation=activation,
        ) for _ in range(d_layers)]
        self.decoder = Decoder(dec_layers, norm_layer=my_Layernorm(d_model), projection=nn.Linear(d_model, c_out, bias=True))

    def forward(self, x_enc, x_dec, enc_self_mask=None, dec_self_mask=None, dec_enc_mask=None):
        # Embedding
        enc_out = self.enc_embedding(x_enc)
        enc_out, attns = self.encoder(enc_out, attn_mask=enc_self_mask)
        dec_out = self.dec_embedding(x_dec)
        dec_out, trend = self.decoder(dec_out, enc_out, x_mask=dec_self_mask, cross_mask=dec_enc_mask, trend=enc_out[:, -self.label_len:, :])

        if self.output_attention:
            return dec_out[:, -self.pred_len:, :], attns
        else:
            return dec_out[:, -self.pred_len:, :]  # [B, L, D]


In [ ]:
import torch
import os
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from torch import nn

# Fungsi untuk menghitung RMSE dan MAPE dengan inverse scaling
def calculate_metrics(y_true, y_pred, scaler=None):
    if scaler is not None:
        y_true = scaler.inverse_transform(y_true)
        y_pred = scaler.inverse_transform(y_pred)
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    return rmse, mape


# Fungsi untuk menyimpan checkpoint
def save_checkpoint(model, optimizer, epoch, val_loss, filename):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_loss': val_loss,
    }
    torch.save(checkpoint, filename)

# Fungsi untuk memuat checkpoint
def load_checkpoint(filename):
    checkpoint = torch.load(filename)
    model_state_dict = checkpoint['model_state_dict']
    optimizer_state_dict = checkpoint['optimizer_state_dict']
    epoch = checkpoint['epoch']
    val_loss = checkpoint['val_loss']
    
    return epoch, model_state_dict, optimizer_state_dict, val_loss

#### Train 3 Months - 24 Hours

In [ ]:
import matplotlib.pyplot as plt
 # 3 Months- 24 hours
def train_and_evaluate_auto3_24(params, save_best_only=True, checkpoint_filename="autoformer3m24.pth.tar", patience=5, scaler=None):
    checkpoint_dir = "./checkpoints"
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)

    checkpoint_filepath = os.path.join(checkpoint_dir, checkpoint_filename)

    # Definisikan model
    model = Autoformer(
        enc_in=1,
        dec_in=1,
        c_out=1,
        seq_len=16,
        label_len=15,
        out_len=224,  # Sesuaikan dengan panjang output yang diharapkan
        d_model=params['d_model'],
        n_heads=params['n_heads'],
        e_layers=params['e_layers'],
        d_layers=params['d_layers'],
        d_ff=params['d_ff'],
        moving_avg=params['moving_avg'],
        dropout=params['dropout'],
        factor=params['factor'],
        activation=params['activation']
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])
    criterion = nn.MSELoss()  # Loss function
    
    best_val_loss = float('inf')  # Untuk menyimpan loss terbaik
    best_rmse = float('inf')  # Untuk menyimpan RMSE terbaik
    best_mape = float('inf')  # Untuk menyimpan MAPE terbaik
    num_epochs = 50  # Jumlah maksimum epoch
    epochs_no_improve = 0  # Untuk melacak jumlah epoch tanpa perbaikan
    early_stop = False  # Status untuk early stopping

    # Inisialisasi list untuk menyimpan train dan validation loss
    train_losses = []
    val_losses = []
    
    for epoch in range(num_epochs):
        if early_stop:
            print("Early stopping")
            break
            
        model.train()
        train_loss = 0.0
        for batch_X, batch_y in train_loader_3:
            optimizer.zero_grad()
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            x_enc = batch_X
            x_dec = batch_X[:, -15:, :]
            output = model(x_enc, x_dec)

            # Sesuaikan batch_y agar memiliki ukuran yang sama dengan output model
            batch_y = batch_y[:, :output.shape[1]]  # Potong atau sesuaikan batch_y

            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader_3)
        train_losses.append(train_loss)  # Simpan train loss

        # Evaluasi pada set validasi
        model.eval()
        val_loss = 0.0
        val_true, val_pred = [], []
        with torch.no_grad():
            for batch_X, batch_y in val_loader_3:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                x_enc = batch_X
                x_dec = batch_X[:, -15:, :]
                output = model(x_enc, x_dec)

                # Sesuaikan batch_y di set validasi
                batch_y = batch_y[:, :output.shape[1]]  # Sesuaikan ukuran batch_y

                val_true.append(batch_y.cpu().numpy())
                val_pred.append(output.cpu().numpy())
                loss = criterion(output, batch_y)
                val_loss += loss.item()

        val_loss /= len(val_loader_3)
        val_losses.append(val_loss)  # Simpan val loss

        # Inverse scaling untuk metrik jika scaler diberikan
        val_true = np.concatenate(val_true, axis=0).reshape(-1, 1)  # Sesuaikan bentuk data
        val_pred = np.concatenate(val_pred, axis=0).reshape(-1, 1)
        
        if scaler is not None:
            val_true = scaler.inverse_transform(val_true)  # Mengembalikan ke skala asli
            val_pred = scaler.inverse_transform(val_pred)

        # Hitung RMSE, MAPE
        rmse, mape = calculate_metrics(val_true, val_pred)
        
        print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.4f}")

        # Cek apakah ada perbaikan dalam validation loss tanpa min_delta
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse
            best_mape = mape
            epochs_no_improve = 0
            if save_best_only:
                save_checkpoint(model, optimizer, epoch, val_loss, filename=checkpoint_filepath)
                print(f"Saving checkpoint at epoch {epoch} with validation loss {val_loss:.4f}")
        else:
            epochs_no_improve += 1  # Tidak ada perbaikan, tambahkan hitungan
        
        # Cek apakah harus early stop
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            early_stop = True
    
    # Visualisasi train dan val loss setelah selesai pelatihan
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Train Loss and Validation Loss Over Epochs')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Kembalikan best_val_loss, rmse, dan mape
    return best_val_loss, best_rmse, best_mape


In [ ]:
# 3 Months - 24 Hours
def random_search_with_checkpoint3auto24(param_grid, num_iter=10):
    best_val_loss = float('inf')  # Untuk melacak loss validasi terbaik
    best_rmse = float('inf')  # Untuk melacak RMSE terbaik
    best_mape = float('inf')  # Untuk melacak MAPE terbaik
    best_params = None

    # Daftar hyperparameter dan pilih secara acak
    keys = list(param_grid.keys())
    evaluated_params = set()  # Untuk melacak kombinasi parameter yang dievaluasi

    for i in range(num_iter):  # Menjalankan pencarian acak sebanyak num_iter kali
        # Pilih hyperparameter secara acak
        params = {key: random.choice(param_grid[key]) for key in keys}

        # Buat representasi parameter yang dapat di-hash
        params_tuple = tuple(sorted(params.items()))
        if params_tuple in evaluated_params:
            continue  # Lewati parameter yang sudah dievaluasi

        evaluated_params.add(params_tuple)  # Tambahkan ke set yang dievaluasi
        print(f"Evaluating with params: {params}")

        # Jalankan pelatihan untuk kombinasi parameter dan dapatkan val_loss, rmse, mape
        val_loss, rmse, mape = train_and_evaluate_auto3_24(params, checkpoint_filename="autoformer3m24.pth.tar")

        # Simpan parameter terbaik jika val_loss lebih baik
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse  # Simpan RMSE terbaik
            best_mape = mape  # Simpan MAPE terbaik
            best_params = params

    # Cetak hasil terbaik setelah pencarian hyperparameter selesai
    print(f"\nBest Validation Loss: {best_val_loss:.4f}")
    print(f"Best RMSE: {best_rmse:.4f}")
    print(f"Best MAPE: {best_mape:.4f}")
    print(f"Best Hyperparameters: {best_params}")

    return best_params


In [ ]:
best_params = {
    'd_model': [256],          # Dimensi model
    'n_heads': [4],            # Jumlah attention heads
    'e_layers': [3],           # Jumlah layer di encoder
    'd_layers': [2],           # Jumlah layer di decoder
    'd_ff': [512],             # Dimensi feed-forward (disarankan 2x atau 4x dari d_model)
    'moving_avg': [25],        # Ukuran rata-rata bergerak (bisa disesuaikan)
    'dropout': [0.3],          # Dropout rate
    'factor': [1],             # Faktor autocorrelation (1 adalah default yang umum)
    'activation': ['relu'],      # Aktivasi, bisa 'relu' atau 'gelu'
    'learning_rate': [0.001]   # Laju pembelajaran
}

# Jalankan Grid Search dengan checkpoint untuk model 3 bulan
best_params = random_search_with_checkpoint3auto(best_params)


In [ ]:
best_params = {
    'd_model': [256],          # Dimensi model
    'n_heads': [4],            # Jumlah attention heads
    'e_layers': [3],           # Jumlah layer di encoder
    'd_layers': [2],           # Jumlah layer di decoder
    'd_ff': [512],             # Dimensi feed-forward (disarankan 2x atau 4x dari d_model)
    'moving_avg': [25],        # Ukuran rata-rata bergerak (bisa disesuaikan)
    'dropout': [0.3],          # Dropout rate
    'factor': [1],             # Faktor autocorrelation (1 adalah default yang umum)
    'activation': ['relu'],      # Aktivasi, bisa 'relu' atau 'gelu'
    'learning_rate': [0.001]   # Laju pembelajaran
}

# Jalankan Grid Search dengan checkpoint untuk model 3 bulan
best_params = random_search_with_checkpoint3auto24(best_params)


#### Actual vs Prediction 3 Months - 24 Hours

In [ ]:
# Define your model (Autoformer) and optimizer
# Adjust the parameters according to your model architecture
model = Autoformer(
    enc_in=1,
    dec_in=1,
    c_out=1,
    seq_len=16,
    label_len=15,
    out_len=224,  # Expected output length
    d_model=256,  # Example configuration
    n_heads=4,
    e_layers=3,
    d_layers=2,
    d_ff=512,
    moving_avg=25,
    dropout=0.3,
    factor=1,
    activation='relu'
).to(device)

# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Step 1: Load model checkpoint
checkpoint_path = './checkpoints/autoformer3mm.pth.tar'  # Path to your checkpoint file

def load_checkpoint(model, optimizer, checkpoint_path):
    if checkpoint_path is not None:
        print(f"Loading checkpoint from '{checkpoint_path}'...")
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        epoch = checkpoint.get('epoch', None)
        best_val_loss = checkpoint.get('best_val_loss', None)  # Handle missing best_val_loss
        print(f"Checkpoint loaded. Resuming from epoch {epoch}")
        return epoch, best_val_loss
    else:
        print("No checkpoint provided, starting from scratch.")
        return None, None

# Load the checkpoint
epoch, best_val_loss = load_checkpoint(model, optimizer, checkpoint_path)

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Define your model (Autoformer) and optimizer
# Adjust the parameters according to your model architecture
model = Autoformer(
    enc_in=1,
    dec_in=1,
    c_out=1,
    seq_len=16,
    label_len=15,
    out_len=224,  # Expected output length
    d_model=256,  # Example configuration
    n_heads=4,
    e_layers=3,
    d_layers=2,
    d_ff=512,
    moving_avg=25,
    dropout=0.3,
    factor=1,
    activation='relu'
).to(device)

# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Step 1: Load model checkpoint
checkpoint_path = './checkpoints/autoformer3mm.pth.tar'  # Path to your checkpoint file

def load_checkpoint(model, optimizer, checkpoint_path):
    if checkpoint_path is not None:
        print(f"Loading checkpoint from '{checkpoint_path}'...")
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        epoch = checkpoint.get('epoch', None)
        best_val_loss = checkpoint.get('best_val_loss', None)  # Handle missing best_val_loss
        print(f"Checkpoint loaded. Resuming from epoch {epoch}")
        return epoch, best_val_loss
    else:
        print("No checkpoint provided, starting from scratch.")
        return None, None

# Load the checkpoint
epoch, best_val_loss = load_checkpoint(model, optimizer, checkpoint_path)

# Step 2: Generate predictions using the test data
model.eval()  # Switch the model to evaluation mode

predictions = []
with torch.no_grad():
    for batch_X, batch_y in test_loader_3:  # test_loader_3 is your DataLoader for the test set
        batch_X = batch_X.to(device)  # Move data to the same device as the model (GPU/CPU)
        x_enc = batch_X  # Encoder input
        x_dec = batch_X[:, -15:, :]  # Decoder input (using the last 15 time steps)
        
        # Get model predictions
        pred = model(x_enc, x_dec)
        
        # Take only the last time step prediction for each batch
        predictions.append(pred[:, -1, :].cpu().numpy())

# Step 3: Concatenate predictions into a single array
predictions = np.concatenate(predictions, axis=0)

# Step 4: Ensure predictions match the size of the test set (if necessary)
pred_len = len(predictions)
test_data_sliced = test_data_3_months.iloc[:pred_len]  # Slicing test data to match predictions length

# Step 5: Convert predictions to DataFrame for plotting
predictions_df = pd.DataFrame(predictions, index=test_data_sliced.index, columns=['predictions'])

# Step 6: Filter the data to start from December
start_date = '2023-12-01'  # Specify the start date for the plot

train_data_filtered = train_data_3_months[train_data_3_months.index >= start_date]
val_data_filtered = val_data_3_months[val_data_3_months.index >= start_date]
test_data_filtered = test_data_sliced[test_data_sliced.index >= start_date]
predictions_filtered = predictions_df[predictions_df.index >= start_date]

# Step 7: Calculate RMSE, MAPE, and residuals
actual_values = test_data_filtered['Hs'].values  # Actual test data values
predicted_values = predictions_filtered['predictions'].values  # Predicted values

# Calculate RMSE and MAPE
rmse = mean_squared_error(actual_values, predicted_values, squared=False)
mape = mean_absolute_percentage_error(actual_values, predicted_values)

# Calculate residuals and standard deviation for confidence intervals
residuals = actual_values - predicted_values
std_dev = np.std(residuals)
conf_interval = 1.96 * std_dev  # 95% confidence interval

# Print RMSE and MAPE
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape * 100:.2f}%")

# Step 8: Plot predictions along with train, validation, test data, and confidence intervals
fig, ax = plt.subplots(figsize=(10, 5))

# Plot train data with a blue solid line
train_data_filtered.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=2)

# Plot validation data with a purple dashed line
val_data_filtered.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=2)

# Plot actual test data with an orange dotted line
test_data_filtered.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=2)

# Plot model predictions with a green solid line
predictions_filtered.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Plot confidence intervals
upper_bound = predictions_filtered['predictions'] + conf_interval
lower_bound = predictions_filtered['predictions'] - conf_interval
ax.fill_between(predictions_filtered.index, lower_bound, upper_bound, color='green', alpha=0.3, label="95% Confidence Interval")

# Add labels and legend
ax.set_xlabel('Datetime')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Train, Validation, Test, and Predictions from December Onwards\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()

# Show the plot
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Assuming predictions, actual test data, and confidence interval calculations are already in place

# Step 1: Define the zoom-in start date to focus on the test data and predictions
zoom_start_date = '2023-12-20'  # Adjust this date to focus on the test and prediction period

# Step 2: Filter data to zoom-in on the specified range
train_data_zoomed = train_data_3_months[train_data_3_months.index >= zoom_start_date]
val_data_zoomed = val_data_3_months[val_data_3_months.index >= zoom_start_date]
test_data_zoomed = test_data_sliced[test_data_sliced.index >= zoom_start_date]
predictions_zoomed = predictions_df[predictions_df.index >= zoom_start_date]

# Step 3: Plot zoomed-in view
fig, ax = plt.subplots(figsize=(12, 6))

# Plot zoomed train data
train_data_zoomed.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=1.5)

# Plot zoomed validation data
val_data_zoomed.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=1.5)

# Plot zoomed test data
test_data_zoomed.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=1.5)

# Plot zoomed model predictions
predictions_zoomed.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Plot confidence intervals for the predictions
upper_bound_zoomed = predictions_zoomed['predictions'] + conf_interval
lower_bound_zoomed = predictions_zoomed['predictions'] - conf_interval
ax.fill_between(predictions_zoomed.index, lower_bound_zoomed, upper_bound_zoomed, color='green', alpha=0.2, label="95% Confidence Interval")

# Step 4: Add labels, title, and legend
ax.set_xlabel('Date')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Zoomed-in View of Test Data and Predictions with Confidence Interval\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()

# Show grid and plot
plt.grid(True)
plt.show()


In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Step 1: Load model checkpoint
checkpoint_path = './checkpoints/autoformer3mm.pth.tar'  # Path to your checkpoint file

def load_checkpoint(model, optimizer, checkpoint_path):
    if checkpoint_path is not None:
        print(f"Loading checkpoint from '{checkpoint_path}'...")
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        epoch = checkpoint.get('epoch', None)
        best_val_loss = checkpoint.get('best_val_loss', None)  # Handle missing best_val_loss
        print(f"Checkpoint loaded. Resuming from epoch {epoch}")
        return epoch, best_val_loss
    else:
        print("No checkpoint provided, starting from scratch.")
        return None, None

# Assume your model and optimizer are already defined (you should have them ready)
# For example:
# model = MyModelClass().to(device)
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Load the checkpoint
epoch, best_val_loss = load_checkpoint(model, optimizer, checkpoint_path)

# Step 2: Generate predictions using the test data
model.eval()  # Switch the model to evaluation mode

predictions = []
with torch.no_grad():
    for batch_X, batch_y in test_loader_3:  # test_loader_3 is your DataLoader for the test set
        batch_X = batch_X.to(device)  # Move data to the same device as the model (GPU/CPU)
        x_enc = batch_X  # Encoder input
        x_dec = batch_X[:, -15:, :]  # Decoder input (using the last 15 time steps)
        
        # Get model predictions
        pred = model(x_enc, x_dec)
        
        # Take only the last time step prediction for each batch
        predictions.append(pred[:, -1, :].cpu().numpy())

# Step 3: Concatenate predictions into a single array
predictions = np.concatenate(predictions, axis=0)

# Step 4: Ensure predictions match the size of the test set (if necessary)
pred_len = len(predictions)
test_data_sliced = test_data_3_months.iloc[:pred_len]  # Slicing test data to match predictions length

# Step 5: Convert predictions to DataFrame for plotting
predictions_df = pd.DataFrame(predictions, index=test_data_sliced.index, columns=['predictions'])

# Step 6: Filter the data to start from December
start_date = '2023-12-01'  # Specify the start date for the plot

train_data_filtered = train_data_3_months[train_data_3_months.index >= start_date]
val_data_filtered = val_data_3_months[val_data_3_months.index >= start_date]
test_data_filtered = test_data_sliced[test_data_sliced.index >= start_date]
predictions_filtered = predictions_df[predictions_df.index >= start_date]

# Step 7: Calculate RMSE and MAPE
actual_values = test_data_filtered['Hs'].values  # Actual test data values
predicted_values = predictions_filtered['predictions'].values  # Predicted values

# Calculate RMSE and MAPE
rmse = mean_squared_error(actual_values, predicted_values, squared=False)
mape = mean_absolute_percentage_error(actual_values, predicted_values)

# Print RMSE and MAPE
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape * 100:.2f}%")

# Step 8: Plot predictions along with train, validation, and test data
fig, ax = plt.subplots(figsize=(10, 5))

# Plot train data with a blue solid line
train_data_filtered.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=2)

# Plot validation data with a purple dashed line
val_data_filtered.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=2)

# Plot actual test data with an orange dotted line
test_data_filtered.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=2)

# Plot model predictions with a green solid line
predictions_filtered.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Add labels and legend
ax.set_xlabel('Datetime')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Train, Validation, Test, and Predictions from December Onwards\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()

# Show the plot
plt.grid(True)
plt.show()


#### Forecast 14 Days

In [ ]:
import matplotlib.pyplot as plt
import torch

def forecast_14_days(model, last_data, scaler=None):
    """
    Fungsi untuk memprediksi 14 hari ke depan menggunakan model yang telah dilatih.
    
    :param model: Model yang telah dilatih.
    :param last_data: Data terakhir yang akan digunakan sebagai input untuk prediksi (sequential input).
    :param scaler: Scaler yang digunakan pada data, untuk mengembalikan hasil prediksi ke skala asli jika perlu.
    """
    model.eval()  # Set model ke mode evaluasi
    
    # Prediksi 14 hari ke depan
    with torch.no_grad():
        # Ubah data terakhir menjadi tensor dan pindahkan ke device
        last_data_tensor = torch.Tensor(last_data).unsqueeze(0).to(device)  # Tambahkan batch dimension
        
        # Gunakan model untuk melakukan prediksi 14 langkah ke depan
        x_enc = last_data_tensor  # Input terakhir untuk encoder
        x_dec = last_data_tensor[:, -15:, :]  # Ambil 15 langkah terakhir sebagai input untuk decoder
        
        forecast = model(x_enc, x_dec)
    
    # Hasil prediksi
    forecast = forecast.cpu().numpy().reshape(-1, 1)  # Ubah bentuk menjadi array (14, 1)
    
    # Slice the first 14 predictions if forecast length is longer
    forecast = forecast[:14]  # Ambil hanya 14 hari prediksi pertama
    
    # Inverse scaling jika diperlukan
    if scaler is not None:
        forecast = scaler.inverse_transform(forecast)
    
    return forecast


def plot_forecast_14_days(forecast, title='14-Day Forecast'):
    """
    Fungsi untuk memplot prediksi 14 hari ke depan.
    
    :param forecast: Array hasil prediksi (14 hari ke depan).
    :param title: Judul plot.
    """
    days = list(range(1, 15))  # Hari 1 hingga 14
    
    plt.figure(figsize=(10, 6))
    plt.plot(days, forecast, label='Forecast', color='green', marker='o')
    plt.title(title, fontsize=14)
    plt.xlabel('Days')
    plt.ylabel('Forecasted Value')
    plt.grid(True)
    plt.legend()
    plt.show()


# Contoh penggunaan forecast 14 hari setelah pelatihan model
# Pastikan scaler telah di-fit ke data training sebelumnya
scaler.fit(train_data_3_months)

# Ambil data terbaru dari dataset validasi atau test set sebagai input
# last_data bisa berasal dari data validasi atau test set
last_data = test_data_3_months.values[-16:]  # Ambil 16 titik data terakhir sebagai input untuk prediksi

# Lakukan forecast 14 hari ke depan
forecast_14 = forecast_14_days(model, last_data, scaler=scaler)

# Visualisasikan prediksi 14 hari ke depan
plot_forecast_14_days(forecast_14, title='14-Day Forecast Using Trained Model')


#### 48 Hours

In [ ]:
# Step 5: Create a window-based dataset using scaled_data (24 jam)
def create_windows_48(data, window_size=32, forecast_horizon=224):
    X, y = [], []
    for i in range(len(data) - window_size - forecast_horizon + 1):
        X.append(data.iloc[i: i + window_size].values)
        y.append(data.iloc[i + window_size: i + window_size + forecast_horizon].values)
    return np.array(X), np.array(y)

In [ ]:
 # Buat dataset berbasis window untuk setiap skenario (24 jam)
X_train_3, y_train_3 = create_windows_48(train_data_3_months)
X_train_6, y_train_6 = create_windows_48(train_data_6_months)
X_train_8, y_train_8 = create_windows_48(train_data_8_months)

X_val_3, y_val_3 = create_windows_48(val_data_3_months)
X_val_6, y_val_6 = create_windows_48(val_data_6_months)
X_val_8, y_val_8 = create_windows_48(val_data_8_months)

X_test_3, y_test_3 = create_windows_48(test_data_3_months)
X_test_6, y_test_6 = create_windows_48(test_data_6_months)
X_test_8, y_test_8 = create_windows_48(test_data_8_months)

# Cek hasilnya untuk melihat ukuran setiap dataset
print(f"Shape of X_train_3: {X_train_3.shape}, y_train_3: {y_train_3.shape}")
print(f"Shape of X_train_6: {X_train_6.shape}, y_train_6: {y_train_6.shape}")
print(f"Shape of X_train_8: {X_train_8.shape}, y_train_8: {y_train_8.shape}")
print(f"Shape of X_val_3: {X_val_3.shape}, y_val_3: {y_val_3.shape}")
print(f"Shape of X_val_6: {X_val_6.shape}, y_val_6: {y_val_6.shape}")
print(f"Shape of X_val_8: {X_val_8.shape}, y_val_8: {y_val_8.shape}")
print(f"Shape of X_test_3: {X_test_3.shape}, y_test_3: {y_test_3.shape}")
print(f"Shape of X_test_6: {X_test_6.shape}, y_test_6: {y_test_6.shape}")
print(f"Shape of X_test_8: {X_test_8.shape}, y_test_8: {y_test_8.shape}")


#### Train 3 Months - 48 Hours

In [ ]:

import matplotlib.pyplot as plt

def train_and_evaluate_auto3_48(params, save_best_only=True, checkpoint_filename="autoformer3m48.pth.tar", patience=5, scaler=None):
    checkpoint_dir = "./checkpoints"
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)

    checkpoint_filepath = os.path.join(checkpoint_dir, checkpoint_filename)

    # Definisikan model
    model = Autoformer(
        enc_in=1,
        dec_in=1,
        c_out=1,
        seq_len=32,
        label_len=15,
        out_len=224,  # Sesuaikan dengan panjang output yang diharapkan
        d_model=params['d_model'],
        n_heads=params['n_heads'],
        e_layers=params['e_layers'],
        d_layers=params['d_layers'],
        d_ff=params['d_ff'],
        moving_avg=params['moving_avg'],
        dropout=params['dropout'],
        factor=params['factor'],
        activation=params['activation']
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])
    criterion = nn.MSELoss()  # Loss function
    
    best_val_loss = float('inf')  # Untuk menyimpan loss terbaik
    best_rmse = float('inf')  # Untuk menyimpan RMSE terbaik
    best_mape = float('inf')  # Untuk menyimpan MAPE terbaik
    num_epochs = 50  # Jumlah maksimum epoch
    epochs_no_improve = 0  # Untuk melacak jumlah epoch tanpa perbaikan
    early_stop = False  # Status untuk early stopping

    # Inisialisasi list untuk menyimpan train dan validation loss
    train_losses = []
    val_losses = []
    
    for epoch in range(num_epochs):
        if early_stop:
            print("Early stopping")
            break
            
        model.train()
        train_loss = 0.0
        for batch_X, batch_y in train_loader_3:
            optimizer.zero_grad()
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            x_enc = batch_X
            x_dec = batch_X[:, -15:, :]
            output = model(x_enc, x_dec)

            # Sesuaikan batch_y agar memiliki ukuran yang sama dengan output model
            batch_y = batch_y[:, :output.shape[1]]  # Potong atau sesuaikan batch_y

            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader_3)
        train_losses.append(train_loss)  # Simpan train loss

        # Evaluasi pada set validasi
        model.eval()
        val_loss = 0.0
        val_true, val_pred = [], []
        with torch.no_grad():
            for batch_X, batch_y in val_loader_3:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                x_enc = batch_X
                x_dec = batch_X[:, -15:, :]
                output = model(x_enc, x_dec)

                # Sesuaikan batch_y di set validasi
                batch_y = batch_y[:, :output.shape[1]]  # Sesuaikan ukuran batch_y

                val_true.append(batch_y.cpu().numpy())
                val_pred.append(output.cpu().numpy())
                loss = criterion(output, batch_y)
                val_loss += loss.item()

        val_loss /= len(val_loader_3)
        val_losses.append(val_loss)  # Simpan val loss

        # Inverse scaling untuk metrik jika scaler diberikan
        val_true = np.concatenate(val_true, axis=0).reshape(-1, 1)  # Sesuaikan bentuk data
        val_pred = np.concatenate(val_pred, axis=0).reshape(-1, 1)
        
        if scaler is not None:
            val_true = scaler.inverse_transform(val_true)  # Mengembalikan ke skala asli
            val_pred = scaler.inverse_transform(val_pred)

        # Hitung RMSE, MAPE
        rmse, mape = calculate_metrics(val_true, val_pred)
        
        print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.4f}")

        # Cek apakah ada perbaikan dalam validation loss tanpa min_delta
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse
            best_mape = mape
            epochs_no_improve = 0
            if save_best_only:
                save_checkpoint(model, optimizer, epoch, val_loss, filename=checkpoint_filepath)
                print(f"Saving checkpoint at epoch {epoch} with validation loss {val_loss:.4f}")
        else:
            epochs_no_improve += 1  # Tidak ada perbaikan, tambahkan hitungan
        
        # Cek apakah harus early stop
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            early_stop = True
    
    # Visualisasi train dan val loss setelah selesai pelatihan
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Train Loss and Validation Loss Over Epochs')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Kembalikan best_val_loss, rmse, dan mape
    return best_val_loss, best_rmse, best_mape


In [ ]:
def random_search_with_checkpoint3auto48(param_grid, num_iter=10):
    best_val_loss = float('inf')  # Untuk melacak loss validasi terbaik
    best_rmse = float('inf')  # Untuk melacak RMSE terbaik
    best_mape = float('inf')  # Untuk melacak MAPE terbaik
    best_params = None

    # Daftar hyperparameter dan pilih secara acak
    keys = list(param_grid.keys())
    evaluated_params = set()  # Untuk melacak kombinasi parameter yang dievaluasi

    for i in range(num_iter):  # Menjalankan pencarian acak sebanyak num_iter kali
        # Pilih hyperparameter secara acak
        params = {key: random.choice(param_grid[key]) for key in keys}

        # Buat representasi parameter yang dapat di-hash
        params_tuple = tuple(sorted(params.items()))
        if params_tuple in evaluated_params:
            continue  # Lewati parameter yang sudah dievaluasi

        evaluated_params.add(params_tuple)  # Tambahkan ke set yang dievaluasi
        print(f"Evaluating with params: {params}")

        # Jalankan pelatihan untuk kombinasi parameter dan dapatkan val_loss, rmse, mape
        val_loss, rmse, mape = train_and_evaluate_auto3_48(params, checkpoint_filename="autoformer3m48.pth.tar")

        # Simpan parameter terbaik jika val_loss lebih baik
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse  # Simpan RMSE terbaik
            best_mape = mape  # Simpan MAPE terbaik
            best_params = params

    # Cetak hasil terbaik setelah pencarian hyperparameter selesai
    print(f"\nBest Validation Loss: {best_val_loss:.4f}")
    print(f"Best RMSE: {best_rmse:.4f}")
    print(f"Best MAPE: {best_mape:.4f}")
    print(f"Best Hyperparameters: {best_params}")

    return best_params


In [ ]:
best_params = {
    'd_model': [256],          # Dimensi model
    'n_heads': [4],            # Jumlah attention heads
    'e_layers': [3],           # Jumlah layer di encoder
    'd_layers': [2],           # Jumlah layer di decoder
    'd_ff': [512],             # Dimensi feed-forward (disarankan 2x atau 4x dari d_model)
    'moving_avg': [25],        # Ukuran rata-rata bergerak (bisa disesuaikan)
    'dropout': [0.3],          # Dropout rate
    'factor': [1],             # Faktor autocorrelation (1 adalah default yang umum)
    'activation': ['relu'],      # Aktivasi, bisa 'relu' atau 'gelu'
    'learning_rate': [0.001]   # Laju pembelajaran
}

# Jalankan Grid Search dengan checkpoint untuk model 3 bulan
best_params = random_search_with_checkpoint3auto48(best_params)


In [ ]:
best_params = {
    'd_model': [256],          # Dimensi model
    'n_heads': [4],            # Jumlah attention heads
    'e_layers': [3],           # Jumlah layer di encoder
    'd_layers': [2],           # Jumlah layer di decoder
    'd_ff': [512],             # Dimensi feed-forward (disarankan 2x atau 4x dari d_model)
    'moving_avg': [25],        # Ukuran rata-rata bergerak (bisa disesuaikan)
    'dropout': [0.3],          # Dropout rate
    'factor': [1],             # Faktor autocorrelation (1 adalah default yang umum)
    'activation': ['relu'],      # Aktivasi, bisa 'relu' atau 'gelu'
    'learning_rate': [0.001]   # Laju pembelajaran
}

# Jalankan Grid Search dengan checkpoint untuk model 3 bulan
best_params = random_search_with_checkpoint3auto48(best_params)


In [ ]:
import torch
import os

# Function to save checkpoint
def save_checkpoint(state, is_best, filename="./checkpoints/autoformer3m48.pth.tar"):
    torch.save(state, filename)
    if is_best:
        torch.save(state, "./checkpoints/autoformer3m48.pth.tar")  # Save as 'best_model.pth.tar'

# Function to load checkpoint
def load_checkpoint(filename="./checkpoints/autoformer3m48.pth.tar"):
    if os.path.isfile(filename):
        checkpoint = torch.load(filename)
        print(f"Loaded checkpoint from {filename} (epoch {checkpoint['epoch']})")
        return checkpoint
    else:
        print(f"No checkpoint found at {filename}")
        return None


In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Define your model (Autoformer) and optimizer
# Adjust the parameters according to your model architecture
model = Autoformer(
    enc_in=1,
    dec_in=1,
    c_out=1,
    seq_len=32,
    label_len=15,
    out_len=224,  # Expected output length
    d_model=256,  # Example configuration
    n_heads=4,
    e_layers=3,
    d_layers=2,
    d_ff=512,
    moving_avg=25,
    dropout=0.3,
    factor=1,
    activation='relu'
).to(device)

# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Step 1: Load model checkpoint
checkpoint_path = './checkpoints/autoformer3m48.pth.tar'  # Path to your checkpoint file

def load_checkpoint(model, optimizer, checkpoint_path):
    if checkpoint_path is not None:
        print(f"Loading checkpoint from '{checkpoint_path}'...")
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        epoch = checkpoint.get('epoch', None)
        best_val_loss = checkpoint.get('best_val_loss', None)  # Handle missing best_val_loss
        print(f"Checkpoint loaded. Resuming from epoch {epoch}")
        return epoch, best_val_loss
    else:
        print("No checkpoint provided, starting from scratch.")
        return None, None

# Load the checkpoint
epoch, best_val_loss = load_checkpoint(model, optimizer, checkpoint_path)

# Step 2: Generate predictions using the test data
model.eval()  # Switch the model to evaluation mode

predictions = []
with torch.no_grad():
    for batch_X, batch_y in test_loader_3:  # test_loader_3 is your DataLoader for the test set
        batch_X = batch_X.to(device)  # Move data to the same device as the model (GPU/CPU)
        x_enc = batch_X  # Encoder input
        x_dec = batch_X[:, -15:, :]  # Decoder input (using the last 15 time steps)
        
        # Get model predictions
        pred = model(x_enc, x_dec)
        
        # Take only the last time step prediction for each batch
        predictions.append(pred[:, -1, :].cpu().numpy())

# Step 3: Concatenate predictions into a single array
predictions = np.concatenate(predictions, axis=0)

# Step 4: Ensure predictions match the size of the test set (if necessary)
pred_len = len(predictions)
test_data_sliced = test_data_3_months.iloc[:pred_len]  # Slicing test data to match predictions length

# Step 5: Convert predictions to DataFrame for plotting
predictions_df = pd.DataFrame(predictions, index=test_data_sliced.index, columns=['predictions'])

# Step 6: Filter the data to start from December
start_date = '2023-12-01'  # Specify the start date for the plot

train_data_filtered = train_data_3_months[train_data_3_months.index >= start_date]
val_data_filtered = val_data_3_months[val_data_3_months.index >= start_date]
test_data_filtered = test_data_sliced[test_data_sliced.index >= start_date]
predictions_filtered = predictions_df[predictions_df.index >= start_date]

# Step 7: Calculate RMSE, MAPE, and residuals
actual_values = test_data_filtered['Hs'].values  # Actual test data values
predicted_values = predictions_filtered['predictions'].values  # Predicted values

# Calculate RMSE and MAPE
rmse = mean_squared_error(actual_values, predicted_values, squared=False)
mape = mean_absolute_percentage_error(actual_values, predicted_values)

# Calculate residuals and standard deviation for confidence intervals
residuals = actual_values - predicted_values
std_dev = np.std(residuals)
conf_interval = 1.96 * std_dev  # 95% confidence interval

# Print RMSE and MAPE
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape * 100:.2f}%")

# Step 8: Plot predictions along with train, validation, test data, and confidence intervals
fig, ax = plt.subplots(figsize=(10, 5))

# Plot train data with a blue solid line
train_data_filtered.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=2)

# Plot validation data with a purple dashed line
val_data_filtered.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=2)

# Plot actual test data with an orange dotted line
test_data_filtered.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=2)

# Plot model predictions with a green solid line
predictions_filtered.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Plot confidence intervals
upper_bound = predictions_filtered['predictions'] + conf_interval
lower_bound = predictions_filtered['predictions'] - conf_interval
ax.fill_between(predictions_filtered.index, lower_bound, upper_bound, color='green', alpha=0.3, label="95% Confidence Interval")

# Add labels and legend
ax.set_xlabel('Datetime')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Train, Validation, Test, and Predictions from December Onwards\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()

# Show the plot
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Assuming predictions, actual test data, and confidence interval calculations are already in place

# Step 1: Define the zoom-in start date to focus on the test data and predictions
zoom_start_date = '2023-12-20'  # Adjust this date to focus on the test and prediction period

# Step 2: Filter data to zoom-in on the specified range
train_data_zoomed = train_data_3_months[train_data_3_months.index >= zoom_start_date]
val_data_zoomed = val_data_3_months[val_data_3_months.index >= zoom_start_date]
test_data_zoomed = test_data_sliced[test_data_sliced.index >= zoom_start_date]
predictions_zoomed = predictions_df[predictions_df.index >= zoom_start_date]

# Step 3: Plot zoomed-in view
fig, ax = plt.subplots(figsize=(12, 6))

# Plot zoomed train data
train_data_zoomed.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=1.5)

# Plot zoomed validation data
val_data_zoomed.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=1.5)

# Plot zoomed test data
test_data_zoomed.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=1.5)

# Plot zoomed model predictions
predictions_zoomed.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Plot confidence intervals for the predictions
upper_bound_zoomed = predictions_zoomed['predictions'] + conf_interval
lower_bound_zoomed = predictions_zoomed['predictions'] - conf_interval
ax.fill_between(predictions_zoomed.index, lower_bound_zoomed, upper_bound_zoomed, color='green', alpha=0.2, label="95% Confidence Interval")

# Step 4: Add labels, title, and legend
ax.set_xlabel('Date')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Zoomed-in View of Test Data and Predictions with Confidence Interval\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()

# Show grid and plot
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import torch

def forecast_14_days(model, last_data, scaler=None):
    """
    Fungsi untuk memprediksi 14 hari ke depan menggunakan model yang telah dilatih.
    
    :param model: Model yang telah dilatih.
    :param last_data: Data terakhir yang akan digunakan sebagai input untuk prediksi (sequential input).
    :param scaler: Scaler yang digunakan pada data, untuk mengembalikan hasil prediksi ke skala asli jika perlu.
    """
    model.eval()  # Set model ke mode evaluasi
    
    # Prediksi 14 hari ke depan
    with torch.no_grad():
        # Ubah data terakhir menjadi tensor dan pindahkan ke device
        last_data_tensor = torch.Tensor(last_data).unsqueeze(0).to(device)  # Tambahkan batch dimension
        
        # Gunakan model untuk melakukan prediksi 14 langkah ke depan
        x_enc = last_data_tensor  # Input terakhir untuk encoder
        x_dec = last_data_tensor[:, -15:, :]  # Ambil 15 langkah terakhir sebagai input untuk decoder
        
        forecast = model(x_enc, x_dec)
    
    # Hasil prediksi
    forecast = forecast.cpu().numpy().reshape(-1, 1)  # Ubah bentuk menjadi array (14, 1)
    
    # Slice the first 14 predictions if forecast length is longer
    forecast = forecast[:14]  # Ambil hanya 14 hari prediksi pertama
    
    # Inverse scaling jika diperlukan
    if scaler is not None:
        forecast = scaler.inverse_transform(forecast)
    
    return forecast


def plot_forecast_14_days(forecast, title='14-Day Forecast'):
    """
    Fungsi untuk memplot prediksi 14 hari ke depan.
    
    :param forecast: Array hasil prediksi (14 hari ke depan).
    :param title: Judul plot.
    """
    days = list(range(1, 15))  # Hari 1 hingga 14
    
    plt.figure(figsize=(10, 6))
    plt.plot(days, forecast, label='Forecast', color='green', marker='o')
    plt.title(title, fontsize=14)
    plt.xlabel('Days')
    plt.ylabel('Forecasted Value')
    plt.grid(True)
    plt.legend()
    plt.show()


# Contoh penggunaan forecast 14 hari setelah pelatihan model
# Pastikan scaler telah di-fit ke data training sebelumnya
scaler.fit(train_data_3_months)

# Ambil data terbaru dari dataset validasi atau test set sebagai input
# last_data bisa berasal dari data validasi atau test set
last_data = test_data_3_months.values[-16:]  # Ambil 16 titik data terakhir sebagai input untuk prediksi

# Lakukan forecast 14 hari ke depan
forecast_14 = forecast_14_days(model, last_data, scaler=scaler)

# Visualisasikan prediksi 14 hari ke depan
plot_forecast_14_days(forecast_14, title='14-Day Forecast Using Trained Model')


#### 96 Hours

In [ ]:
# Step 5: Create a window-based dataset using scaled_data (24 jam)
def create_windows_96(data, window_size=64, forecast_horizon=224):
    X, y = [], []
    for i in range(len(data) - window_size - forecast_horizon + 1):
        X.append(data.iloc[i: i + window_size].values)
        y.append(data.iloc[i + window_size: i + window_size + forecast_horizon].values)
    return np.array(X), np.array(y)

In [ ]:
 # Buat dataset berbasis window untuk setiap skenario (24 jam)
X_train_3, y_train_3 = create_windows_96(train_data_3_months)
X_train_6, y_train_6 = create_windows_96(train_data_6_months)
X_train_8, y_train_8 = create_windows_96(train_data_8_months)

X_val_3, y_val_3 = create_windows_96(val_data_3_months)
X_val_6, y_val_6 = create_windows_96(val_data_6_months)
X_val_8, y_val_8 = create_windows_96(val_data_8_months)

X_test_3, y_test_3 = create_windows_96(test_data_3_months)
X_test_6, y_test_6 = create_windows_96(test_data_6_months)
X_test_8, y_test_8 = create_windows_96(test_data_8_months)

# Cek hasilnya untuk melihat ukuran setiap dataset
print(f"Shape of X_train_3: {X_train_3.shape}, y_train_3: {y_train_3.shape}")
print(f"Shape of X_train_6: {X_train_6.shape}, y_train_6: {y_train_6.shape}")
print(f"Shape of X_train_8: {X_train_8.shape}, y_train_8: {y_train_8.shape}")
print(f"Shape of X_val_3: {X_val_3.shape}, y_val_3: {y_val_3.shape}")
print(f"Shape of X_val_6: {X_val_6.shape}, y_val_6: {y_val_6.shape}")
print(f"Shape of X_val_8: {X_val_8.shape}, y_val_8: {y_val_8.shape}")
print(f"Shape of X_test_3: {X_test_3.shape}, y_test_3: {y_test_3.shape}")
print(f"Shape of X_test_6: {X_test_6.shape}, y_test_6: {y_test_6.shape}")
print(f"Shape of X_test_8: {X_test_8.shape}, y_test_8: {y_test_8.shape}")


In [ ]:

import matplotlib.pyplot as plt

def train_and_evaluate_auto3_96(params, save_best_only=True, checkpoint_filename="autoformer3m96.pth.tar", patience=5, scaler=None):
    checkpoint_dir = "./checkpoints"
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)

    checkpoint_filepath = os.path.join(checkpoint_dir, checkpoint_filename)

    # Definisikan model
    model = Autoformer(
        enc_in=1,
        dec_in=1,
        c_out=1,
        seq_len=64,
        label_len=15,
        out_len=224,  # Sesuaikan dengan panjang output yang diharapkan
        d_model=params['d_model'],
        n_heads=params['n_heads'],
        e_layers=params['e_layers'],
        d_layers=params['d_layers'],
        d_ff=params['d_ff'],
        moving_avg=params['moving_avg'],
        dropout=params['dropout'],
        factor=params['factor'],
        activation=params['activation']
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])
    criterion = nn.MSELoss()  # Loss function
    
    best_val_loss = float('inf')  # Untuk menyimpan loss terbaik
    best_rmse = float('inf')  # Untuk menyimpan RMSE terbaik
    best_mape = float('inf')  # Untuk menyimpan MAPE terbaik
    num_epochs = 50  # Jumlah maksimum epoch
    epochs_no_improve = 0  # Untuk melacak jumlah epoch tanpa perbaikan
    early_stop = False  # Status untuk early stopping

    # Inisialisasi list untuk menyimpan train dan validation loss
    train_losses = []
    val_losses = []
    
    for epoch in range(num_epochs):
        if early_stop:
            print("Early stopping")
            break
            
        model.train()
        train_loss = 0.0
        for batch_X, batch_y in train_loader_3:
            optimizer.zero_grad()
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            x_enc = batch_X
            x_dec = batch_X[:, -15:, :]
            output = model(x_enc, x_dec)

            # Sesuaikan batch_y agar memiliki ukuran yang sama dengan output model
            batch_y = batch_y[:, :output.shape[1]]  # Potong atau sesuaikan batch_y

            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader_3)
        train_losses.append(train_loss)  # Simpan train loss

        # Evaluasi pada set validasi
        model.eval()
        val_loss = 0.0
        val_true, val_pred = [], []
        with torch.no_grad():
            for batch_X, batch_y in val_loader_3:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                x_enc = batch_X
                x_dec = batch_X[:, -15:, :]
                output = model(x_enc, x_dec)

                # Sesuaikan batch_y di set validasi
                batch_y = batch_y[:, :output.shape[1]]  # Sesuaikan ukuran batch_y

                val_true.append(batch_y.cpu().numpy())
                val_pred.append(output.cpu().numpy())
                loss = criterion(output, batch_y)
                val_loss += loss.item()

        val_loss /= len(val_loader_3)
        val_losses.append(val_loss)  # Simpan val loss

        # Inverse scaling untuk metrik jika scaler diberikan
        val_true = np.concatenate(val_true, axis=0).reshape(-1, 1)  # Sesuaikan bentuk data
        val_pred = np.concatenate(val_pred, axis=0).reshape(-1, 1)
        
        if scaler is not None:
            val_true = scaler.inverse_transform(val_true)  # Mengembalikan ke skala asli
            val_pred = scaler.inverse_transform(val_pred)

        # Hitung RMSE, MAPE
        rmse, mape = calculate_metrics(val_true, val_pred)
        
        print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.4f}")

        # Cek apakah ada perbaikan dalam validation loss tanpa min_delta
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse
            best_mape = mape
            epochs_no_improve = 0
            if save_best_only:
                save_checkpoint(model, optimizer, epoch, val_loss, filename=checkpoint_filepath)
                print(f"Saving checkpoint at epoch {epoch} with validation loss {val_loss:.4f}")
        else:
            epochs_no_improve += 1  # Tidak ada perbaikan, tambahkan hitungan
        
        # Cek apakah harus early stop
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            early_stop = True
    
    # Visualisasi train dan val loss setelah selesai pelatihan
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Train Loss and Validation Loss Over Epochs')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Kembalikan best_val_loss, rmse, dan mape
    return best_val_loss, best_rmse, best_mape


In [ ]:
def random_search_with_checkpoint3auto96(param_grid, num_iter=10):
    best_val_loss = float('inf')  # Untuk melacak loss validasi terbaik
    best_rmse = float('inf')  # Untuk melacak RMSE terbaik
    best_mape = float('inf')  # Untuk melacak MAPE terbaik
    best_params = None

    # Daftar hyperparameter dan pilih secara acak
    keys = list(param_grid.keys())
    evaluated_params = set()  # Untuk melacak kombinasi parameter yang dievaluasi

    for i in range(num_iter):  # Menjalankan pencarian acak sebanyak num_iter kali
        # Pilih hyperparameter secara acak
        params = {key: random.choice(param_grid[key]) for key in keys}

        # Buat representasi parameter yang dapat di-hash
        params_tuple = tuple(sorted(params.items()))
        if params_tuple in evaluated_params:
            continue  # Lewati parameter yang sudah dievaluasi

        evaluated_params.add(params_tuple)  # Tambahkan ke set yang dievaluasi
        print(f"Evaluating with params: {params}")

        # Jalankan pelatihan untuk kombinasi parameter dan dapatkan val_loss, rmse, mape
        val_loss, rmse, mape = train_and_evaluate_auto3_96(params, checkpoint_filename="autoformer3m96.pth.tar")

        # Simpan parameter terbaik jika val_loss lebih baik
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse  # Simpan RMSE terbaik
            best_mape = mape  # Simpan MAPE terbaik
            best_params = params

    # Cetak hasil terbaik setelah pencarian hyperparameter selesai
    print(f"\nBest Validation Loss: {best_val_loss:.4f}")
    print(f"Best RMSE: {best_rmse:.4f}")
    print(f"Best MAPE: {best_mape:.4f}")
    print(f"Best Hyperparameters: {best_params}")

    return best_params


In [ ]:
best_params = {
    'd_model': [256],          # Dimensi model
    'n_heads': [4],            # Jumlah attention heads
    'e_layers': [3],           # Jumlah layer di encoder
    'd_layers': [2],           # Jumlah layer di decoder
    'd_ff': [512],             # Dimensi feed-forward (disarankan 2x atau 4x dari d_model)
    'moving_avg': [25],        # Ukuran rata-rata bergerak (bisa disesuaikan)
    'dropout': [0.3],          # Dropout rate
    'factor': [1],             # Faktor autocorrelation (1 adalah default yang umum)
    'activation': ['relu'],      # Aktivasi, bisa 'relu' atau 'gelu'
    'learning_rate': [0.001]   # Laju pembelajaran
}

# Jalankan Grid Search dengan checkpoint untuk model 3 bulan
best_params = random_search_with_checkpoint3auto96(best_params)


In [ ]:
best_params = {
    'd_model': [256],          # Dimensi model
    'n_heads': [4],            # Jumlah attention heads
    'e_layers': [3],           # Jumlah layer di encoder
    'd_layers': [2],           # Jumlah layer di decoder
    'd_ff': [512],             # Dimensi feed-forward (disarankan 2x atau 4x dari d_model)
    'moving_avg': [25],        # Ukuran rata-rata bergerak (bisa disesuaikan)
    'dropout': [0.3],          # Dropout rate
    'factor': [1],             # Faktor autocorrelation (1 adalah default yang umum)
    'activation': ['relu'],      # Aktivasi, bisa 'relu' atau 'gelu'
    'learning_rate': [0.001]   # Laju pembelajaran
}

# Jalankan Grid Search dengan checkpoint untuk model 3 bulan
best_params = random_search_with_checkpoint3auto96(best_params)


In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Define your model (Autoformer) and optimizer
# Adjust the parameters according to your model architecture
model = Autoformer(
    enc_in=1,
    dec_in=1,
    c_out=1,
    seq_len=64,
    label_len=15,
    out_len=224,  # Expected output length
    d_model=256,  # Example configuration
    n_heads=4,
    e_layers=3,
    d_layers=2,
    d_ff=512,
    moving_avg=25,
    dropout=0.3,
    factor=1,
    activation='relu'
).to(device)

# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Step 1: Load model checkpoint
checkpoint_path = './checkpoints/autoformer3m96.pth.tar'  # Path to your checkpoint file

def load_checkpoint(model, optimizer, checkpoint_path):
    if checkpoint_path is not None:
        print(f"Loading checkpoint from '{checkpoint_path}'...")
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        epoch = checkpoint.get('epoch', None)
        best_val_loss = checkpoint.get('best_val_loss', None)  # Handle missing best_val_loss
        print(f"Checkpoint loaded. Resuming from epoch {epoch}")
        return epoch, best_val_loss
    else:
        print("No checkpoint provided, starting from scratch.")
        return None, None

# Load the checkpoint
epoch, best_val_loss = load_checkpoint(model, optimizer, checkpoint_path)

# Step 2: Generate predictions using the test data
model.eval()  # Switch the model to evaluation mode

predictions = []
with torch.no_grad():
    for batch_X, batch_y in test_loader_3:  # test_loader_3 is your DataLoader for the test set
        batch_X = batch_X.to(device)  # Move data to the same device as the model (GPU/CPU)
        x_enc = batch_X  # Encoder input
        x_dec = batch_X[:, -15:, :]  # Decoder input (using the last 15 time steps)
        
        # Get model predictions
        pred = model(x_enc, x_dec)
        
        # Take only the last time step prediction for each batch
        predictions.append(pred[:, -1, :].cpu().numpy())

# Step 3: Concatenate predictions into a single array
predictions = np.concatenate(predictions, axis=0)

# Step 4: Ensure predictions match the size of the test set (if necessary)
pred_len = len(predictions)
test_data_sliced = test_data_3_months.iloc[:pred_len]  # Slicing test data to match predictions length

# Step 5: Convert predictions to DataFrame for plotting
predictions_df = pd.DataFrame(predictions, index=test_data_sliced.index, columns=['predictions'])

# Step 6: Filter the data to start from December
start_date = '2023-12-01'  # Specify the start date for the plot

train_data_filtered = train_data_3_months[train_data_3_months.index >= start_date]
val_data_filtered = val_data_3_months[val_data_3_months.index >= start_date]
test_data_filtered = test_data_sliced[test_data_sliced.index >= start_date]
predictions_filtered = predictions_df[predictions_df.index >= start_date]

# Step 7: Calculate RMSE, MAPE, and residuals
actual_values = test_data_filtered['Hs'].values  # Actual test data values
predicted_values = predictions_filtered['predictions'].values  # Predicted values

# Calculate RMSE and MAPE
rmse = mean_squared_error(actual_values, predicted_values, squared=False)
mape = mean_absolute_percentage_error(actual_values, predicted_values)

# Calculate residuals and standard deviation for confidence intervals
residuals = actual_values - predicted_values
std_dev = np.std(residuals)
conf_interval = 1.96 * std_dev  # 95% confidence interval

# Print RMSE and MAPE
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape * 100:.2f}%")

# Step 8: Plot predictions along with train, validation, test data, and confidence intervals
fig, ax = plt.subplots(figsize=(10, 5))

# Plot train data with a blue solid line
train_data_filtered.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=2)

# Plot validation data with a purple dashed line
val_data_filtered.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=2)

# Plot actual test data with an orange dotted line
test_data_filtered.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=2)

# Plot model predictions with a green solid line
predictions_filtered.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Plot confidence intervals
upper_bound = predictions_filtered['predictions'] + conf_interval
lower_bound = predictions_filtered['predictions'] - conf_interval
ax.fill_between(predictions_filtered.index, lower_bound, upper_bound, color='green', alpha=0.3, label="95% Confidence Interval")

# Add labels and legend
ax.set_xlabel('Datetime')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Train, Validation, Test, and Predictions from December Onwards\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()

# Show the plot
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Assuming predictions, actual test data, and confidence interval calculations are already in place

# Step 1: Define the zoom-in start date to focus on the test data and predictions
zoom_start_date = '2023-12-20'  # Adjust this date to focus on the test and prediction period

# Step 2: Filter data to zoom-in on the specified range
train_data_zoomed = train_data_3_months[train_data_3_months.index >= zoom_start_date]
val_data_zoomed = val_data_3_months[val_data_3_months.index >= zoom_start_date]
test_data_zoomed = test_data_sliced[test_data_sliced.index >= zoom_start_date]
predictions_zoomed = predictions_df[predictions_df.index >= zoom_start_date]

# Step 3: Plot zoomed-in view
fig, ax = plt.subplots(figsize=(12, 6))

# Plot zoomed train data
train_data_zoomed.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=1.5)

# Plot zoomed validation data
val_data_zoomed.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=1.5)

# Plot zoomed test data
test_data_zoomed.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=1.5)

# Plot zoomed model predictions
predictions_zoomed.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Plot confidence intervals for the predictions
upper_bound_zoomed = predictions_zoomed['predictions'] + conf_interval
lower_bound_zoomed = predictions_zoomed['predictions'] - conf_interval
ax.fill_between(predictions_zoomed.index, lower_bound_zoomed, upper_bound_zoomed, color='green', alpha=0.2, label="95% Confidence Interval")

# Step 4: Add labels, title, and legend
ax.set_xlabel('Date')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Zoomed-in View of Test Data and Predictions with Confidence Interval\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()

# Show grid and plot
plt.grid(True)
plt.show()

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Step 1: Load model checkpoint
checkpoint_path = './checkpoints/autoformer3m96.pth.tar'  # Path to your checkpoint file

def load_checkpoint(model, optimizer, checkpoint_path):
    if checkpoint_path is not None:
        print(f"Loading checkpoint from '{checkpoint_path}'...")
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        epoch = checkpoint.get('epoch', None)
        best_val_loss = checkpoint.get('best_val_loss', None)  # Handle missing best_val_loss
        print(f"Checkpoint loaded. Resuming from epoch {epoch}")
        return epoch, best_val_loss
    else:
        print("No checkpoint provided, starting from scratch.")
        return None, None

# Assume your model and optimizer are already defined (you should have them ready)
# For example:
# model = MyModelClass().to(device)
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Load the checkpoint
epoch, best_val_loss = load_checkpoint(model, optimizer, checkpoint_path)

# Step 2: Generate predictions using the test data
model.eval()  # Switch the model to evaluation mode

predictions = []
with torch.no_grad():
    for batch_X, batch_y in test_loader_3:  # test_loader_3 is your DataLoader for the test set
        batch_X = batch_X.to(device)  # Move data to the same device as the model (GPU/CPU)
        x_enc = batch_X  # Encoder input
        x_dec = batch_X[:, -15:, :]  # Decoder input (using the last 15 time steps)
        
        # Get model predictions
        pred = model(x_enc, x_dec)
        
        # Take only the last time step prediction for each batch
        predictions.append(pred[:, -1, :].cpu().numpy())

# Step 3: Concatenate predictions into a single array
predictions = np.concatenate(predictions, axis=0)

# Step 4: Ensure predictions match the size of the test set (if necessary)
pred_len = len(predictions)
test_data_sliced = test_data_3_months.iloc[:pred_len]  # Slicing test data to match predictions length

# Step 5: Convert predictions to DataFrame for plotting
predictions_df = pd.DataFrame(predictions, index=test_data_sliced.index, columns=['predictions'])

# Step 6: Filter the data to start from December
start_date = '2023-12-01'  # Specify the start date for the plot

train_data_filtered = train_data_3_months[train_data_3_months.index >= start_date]
val_data_filtered = val_data_3_months[val_data_3_months.index >= start_date]
test_data_filtered = test_data_sliced[test_data_sliced.index >= start_date]
predictions_filtered = predictions_df[predictions_df.index >= start_date]

# Step 7: Calculate RMSE and MAPE
actual_values = test_data_filtered['Hs'].values  # Actual test data values
predicted_values = predictions_filtered['predictions'].values  # Predicted values

# Calculate RMSE and MAPE
rmse = mean_squared_error(actual_values, predicted_values, squared=False)
mape = mean_absolute_percentage_error(actual_values, predicted_values)

# Print RMSE and MAPE
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape * 100:.2f}%")

# Step 8: Plot predictions along with train, validation, and test data
fig, ax = plt.subplots(figsize=(10, 5))

# Plot train data with a blue solid line
train_data_filtered.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=2)

# Plot validation data with a purple dashed line
val_data_filtered.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=2)

# Plot actual test data with an orange dotted line
test_data_filtered.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=2)

# Plot model predictions with a green solid line
predictions_filtered.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Add labels and legend
ax.set_xlabel('Datetime')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Train, Validation, Test, and Predictions from December Onwards\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()

# Show the plot
plt.grid(True)
plt.show()


#### 6 Months

#### 24 hours

In [ ]:
# Step 5: Create a window-based dataset using scaled_data (24 jam)
def create_windows_6_24(data, window_size=16, forecast_horizon=224):
    X, y = [], []
    for i in range(len(data) - window_size - forecast_horizon + 1):
        X.append(data.iloc[i: i + window_size].values)
        y.append(data.iloc[i + window_size: i + window_size + forecast_horizon].values)
    return np.array(X), np.array(y)

In [ ]:
 # Buat dataset berbasis window untuk setiap skenario (24 jam)
X_train_3, y_train_3 = create_windows_6_24(train_data_3_months)
X_train_6, y_train_6 = create_windows_6_24(train_data_6_months)
X_train_8, y_train_8 = create_windows_6_24(train_data_8_months)

X_val_3, y_val_3 = create_windows_6_24(val_data_3_months)
X_val_6, y_val_6 = create_windows_6_24(val_data_6_months)
X_val_8, y_val_8 = create_windows_6_24(val_data_8_months)

X_test_3, y_test_3 = create_windows_6_24(test_data_3_months)
X_test_6, y_test_6 = create_windows_6_24(test_data_6_months)
X_test_8, y_test_8 = create_windows_6_24(test_data_8_months)

# Cek hasilnya untuk melihat ukuran setiap dataset
print(f"Shape of X_train_3: {X_train_3.shape}, y_train_3: {y_train_3.shape}")
print(f"Shape of X_train_6: {X_train_6.shape}, y_train_6: {y_train_6.shape}")
print(f"Shape of X_train_8: {X_train_8.shape}, y_train_8: {y_train_8.shape}")
print(f"Shape of X_val_3: {X_val_3.shape}, y_val_3: {y_val_3.shape}")
print(f"Shape of X_val_6: {X_val_6.shape}, y_val_6: {y_val_6.shape}")
print(f"Shape of X_val_8: {X_val_8.shape}, y_val_8: {y_val_8.shape}")
print(f"Shape of X_test_3: {X_test_3.shape}, y_test_3: {y_test_3.shape}")
print(f"Shape of X_test_6: {X_test_6.shape}, y_test_6: {y_test_6.shape}")
print(f"Shape of X_test_8: {X_test_8.shape}, y_test_8: {y_test_8.shape}")


In [ ]:

import matplotlib.pyplot as plt

def train_and_evaluate_auto6_24(params, save_best_only=True, checkpoint_filename="autoformer6m24.pth.tar", patience=5, scaler=None):
    checkpoint_dir = "./checkpoints"
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)

    checkpoint_filepath = os.path.join(checkpoint_dir, checkpoint_filename)

    # Definisikan model
    model = Autoformer(
        enc_in=1,
        dec_in=1,
        c_out=1,
        seq_len=16,
        label_len=15,
        out_len=224,  # Sesuaikan dengan panjang output yang diharapkan
        d_model=params['d_model'],
        n_heads=params['n_heads'],
        e_layers=params['e_layers'],
        d_layers=params['d_layers'],
        d_ff=params['d_ff'],
        moving_avg=params['moving_avg'],
        dropout=params['dropout'],
        factor=params['factor'],
        activation=params['activation']
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])
    criterion = nn.MSELoss()  # Loss function
    
    best_val_loss = float('inf')  # Untuk menyimpan loss terbaik
    best_rmse = float('inf')  # Untuk menyimpan RMSE terbaik
    best_mape = float('inf')  # Untuk menyimpan MAPE terbaik
    num_epochs = 50  # Jumlah maksimum epoch
    epochs_no_improve = 0  # Untuk melacak jumlah epoch tanpa perbaikan
    early_stop = False  # Status untuk early stopping

    # Inisialisasi list untuk menyimpan train dan validation loss
    train_losses = []
    val_losses = []
    
    for epoch in range(num_epochs):
        if early_stop:
            print("Early stopping")
            break
            
        model.train()
        train_loss = 0.0
        for batch_X, batch_y in train_loader_6:
            optimizer.zero_grad()
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            x_enc = batch_X
            x_dec = batch_X[:, -15:, :]
            output = model(x_enc, x_dec)

            # Sesuaikan batch_y agar memiliki ukuran yang sama dengan output model
            batch_y = batch_y[:, :output.shape[1]]  # Potong atau sesuaikan batch_y

            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader_6)
        train_losses.append(train_loss)  # Simpan train loss

        # Evaluasi pada set validasi
        model.eval()
        val_loss = 0.0
        val_true, val_pred = [], []
        with torch.no_grad():
            for batch_X, batch_y in val_loader_6:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                x_enc = batch_X
                x_dec = batch_X[:, -15:, :]
                output = model(x_enc, x_dec)

                # Sesuaikan batch_y di set validasi
                batch_y = batch_y[:, :output.shape[1]]  # Sesuaikan ukuran batch_y

                val_true.append(batch_y.cpu().numpy())
                val_pred.append(output.cpu().numpy())
                loss = criterion(output, batch_y)
                val_loss += loss.item()

        val_loss /= len(val_loader_6)
        val_losses.append(val_loss)  # Simpan val loss

        # Inverse scaling untuk metrik jika scaler diberikan
        val_true = np.concatenate(val_true, axis=0).reshape(-1, 1)  # Sesuaikan bentuk data
        val_pred = np.concatenate(val_pred, axis=0).reshape(-1, 1)
        
        if scaler is not None:
            val_true = scaler.inverse_transform(val_true)  # Mengembalikan ke skala asli
            val_pred = scaler.inverse_transform(val_pred)

        # Hitung RMSE, MAPE
        rmse, mape = calculate_metrics(val_true, val_pred)
        
        print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.4f}")

        # Cek apakah ada perbaikan dalam validation loss tanpa min_delta
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse
            best_mape = mape
            epochs_no_improve = 0
            if save_best_only:
                save_checkpoint(model, optimizer, epoch, val_loss, filename=checkpoint_filepath)
                print(f"Saving checkpoint at epoch {epoch} with validation loss {val_loss:.4f}")
        else:
            epochs_no_improve += 1  # Tidak ada perbaikan, tambahkan hitungan
        
        # Cek apakah harus early stop
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            early_stop = True
    
    # Visualisasi train dan val loss setelah selesai pelatihan
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Train Loss and Validation Loss Over Epochs')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Kembalikan best_val_loss, rmse, dan mape
    return best_val_loss, best_rmse, best_mape


In [ ]:
def random_search_with_checkpoint6auto24(param_grid, num_iter=10):
    best_val_loss = float('inf')  # Untuk melacak loss validasi terbaik
    best_rmse = float('inf')  # Untuk melacak RMSE terbaik
    best_mape = float('inf')  # Untuk melacak MAPE terbaik
    best_params = None

    # Daftar hyperparameter dan pilih secara acak
    keys = list(param_grid.keys())
    evaluated_params = set()  # Untuk melacak kombinasi parameter yang dievaluasi

    for i in range(num_iter):  # Menjalankan pencarian acak sebanyak num_iter kali
        # Pilih hyperparameter secara acak
        params = {key: random.choice(param_grid[key]) for key in keys}

        # Buat representasi parameter yang dapat di-hash
        params_tuple = tuple(sorted(params.items()))
        if params_tuple in evaluated_params:
            continue  # Lewati parameter yang sudah dievaluasi

        evaluated_params.add(params_tuple)  # Tambahkan ke set yang dievaluasi
        print(f"Evaluating with params: {params}")

        # Jalankan pelatihan untuk kombinasi parameter dan dapatkan val_loss, rmse, mape
        val_loss, rmse, mape = train_and_evaluate_auto6_24(params, checkpoint_filename="autoformer6m24.pth.tar")

        # Simpan parameter terbaik jika val_loss lebih baik
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse  # Simpan RMSE terbaik
            best_mape = mape  # Simpan MAPE terbaik
            best_params = params

    # Cetak hasil terbaik setelah pencarian hyperparameter selesai
    print(f"\nBest Validation Loss: {best_val_loss:.4f}")
    print(f"Best RMSE: {best_rmse:.4f}")
    print(f"Best MAPE: {best_mape:.4f}")
    print(f"Best Hyperparameters: {best_params}")

    return best_params


In [ ]:
best_params = {
    'd_model': [256],          # Dimensi model
    'n_heads': [4],            # Jumlah attention heads
    'e_layers': [3],           # Jumlah layer di encoder
    'd_layers': [2],           # Jumlah layer di decoder
    'd_ff': [512],             # Dimensi feed-forward (disarankan 2x atau 4x dari d_model)
    'moving_avg': [25],        # Ukuran rata-rata bergerak (bisa disesuaikan)
    'dropout': [0.3],          # Dropout rate
    'factor': [1],             # Faktor autocorrelation (1 adalah default yang umum)
    'activation': ['relu'],      # Aktivasi, bisa 'relu' atau 'gelu'
    'learning_rate': [0.001]   # Laju pembelajaran
}

# Jalankan Grid Search dengan checkpoint untuk model 3 bulan
best_params = random_search_with_checkpoint6auto24(best_params)

In [ ]:
best_params = {
    'd_model': [256],          # Dimensi model
    'n_heads': [4],            # Jumlah attention heads
    'e_layers': [3],           # Jumlah layer di encoder
    'd_layers': [2],           # Jumlah layer di decoder
    'd_ff': [512],             # Dimensi feed-forward (disarankan 2x atau 4x dari d_model)
    'moving_avg': [25],        # Ukuran rata-rata bergerak (bisa disesuaikan)
    'dropout': [0.3],          # Dropout rate
    'factor': [1],             # Faktor autocorrelation (1 adalah default yang umum)
    'activation': ['relu'],      # Aktivasi, bisa 'relu' atau 'gelu'
    'learning_rate': [0.001]   # Laju pembelajaran
}

# Jalankan Grid Search dengan checkpoint untuk model 3 bulan
best_params = random_search_with_checkpoint6auto24(best_params)


In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Define your model (Autoformer) and optimizer
# Adjust the parameters according to your model architecture
model = Autoformer(
    enc_in=1,
    dec_in=1,
    c_out=1,
    seq_len=16,
    label_len=15,
    out_len=224,  # Expected output length
    d_model=256,  # Example configuration
    n_heads=4,
    e_layers=3,
    d_layers=2,
    d_ff=512,
    moving_avg=25,
    dropout=0.3,
    factor=1,
    activation='relu'
).to(device)

# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Step 1: Load model checkpoint
checkpoint_path = './checkpoints/autoformer6m24.pth.tar'  # Path to your checkpoint file

def load_checkpoint(model, optimizer, checkpoint_path):
    if checkpoint_path is not None:
        print(f"Loading checkpoint from '{checkpoint_path}'...")
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        epoch = checkpoint.get('epoch', None)
        best_val_loss = checkpoint.get('best_val_loss', None)  # Handle missing best_val_loss
        print(f"Checkpoint loaded. Resuming from epoch {epoch}")
        return epoch, best_val_loss
    else:
        print("No checkpoint provided, starting from scratch.")
        return None, None

# Load the checkpoint
epoch, best_val_loss = load_checkpoint(model, optimizer, checkpoint_path)

# Step 2: Generate predictions using the test data
model.eval()  # Switch the model to evaluation mode

predictions = []
with torch.no_grad():
    for batch_X, batch_y in test_loader_3:  # test_loader_3 is your DataLoader for the test set
        batch_X = batch_X.to(device)  # Move data to the same device as the model (GPU/CPU)
        x_enc = batch_X  # Encoder input
        x_dec = batch_X[:, -15:, :]  # Decoder input (using the last 15 time steps)
        
        # Get model predictions
        pred = model(x_enc, x_dec)
        
        # Take only the last time step prediction for each batch
        predictions.append(pred[:, -1, :].cpu().numpy())

# Step 3: Concatenate predictions into a single array
predictions = np.concatenate(predictions, axis=0)

# Step 4: Ensure predictions match the size of the test set (if necessary)
pred_len = len(predictions)
test_data_sliced = test_data_3_months.iloc[:pred_len]  # Slicing test data to match predictions length

# Step 5: Convert predictions to DataFrame for plotting
predictions_df = pd.DataFrame(predictions, index=test_data_sliced.index, columns=['predictions'])

# Step 6: Filter the data to start from December
start_date = '2023-12-01'  # Specify the start date for the plot

train_data_filtered = train_data_3_months[train_data_3_months.index >= start_date]
val_data_filtered = val_data_3_months[val_data_3_months.index >= start_date]
test_data_filtered = test_data_sliced[test_data_sliced.index >= start_date]
predictions_filtered = predictions_df[predictions_df.index >= start_date]

# Step 7: Calculate RMSE, MAPE, and residuals
actual_values = test_data_filtered['Hs'].values  # Actual test data values
predicted_values = predictions_filtered['predictions'].values  # Predicted values

# Calculate RMSE and MAPE
rmse = mean_squared_error(actual_values, predicted_values, squared=False)
mape = mean_absolute_percentage_error(actual_values, predicted_values)

# Calculate residuals and standard deviation for confidence intervals
residuals = actual_values - predicted_values
std_dev = np.std(residuals)
conf_interval = 1.96 * std_dev  # 95% confidence interval

# Print RMSE and MAPE
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape * 100:.2f}%")

# Step 8: Plot predictions along with train, validation, test data, and confidence intervals
fig, ax = plt.subplots(figsize=(10, 5))

# Plot train data with a blue solid line
train_data_filtered.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=2)

# Plot validation data with a purple dashed line
val_data_filtered.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=2)

# Plot actual test data with an orange dotted line
test_data_filtered.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=2)

# Plot model predictions with a green solid line
predictions_filtered.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Plot confidence intervals
upper_bound = predictions_filtered['predictions'] + conf_interval
lower_bound = predictions_filtered['predictions'] - conf_interval
ax.fill_between(predictions_filtered.index, lower_bound, upper_bound, color='green', alpha=0.3, label="95% Confidence Interval")

# Add labels and legend
ax.set_xlabel('Datetime')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Train, Validation, Test, and Predictions from December Onwards\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()

# Show the plot
plt.grid(True)
plt.show()



In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Assuming predictions, actual test data, and confidence interval calculations are already in place

# Step 1: Define the zoom-in start date to focus on the test data and predictions
zoom_start_date = '2023-12-20'  # Adjust this date to focus on the test and prediction period

# Step 2: Filter data to zoom-in on the specified range
train_data_zoomed = train_data_3_months[train_data_3_months.index >= zoom_start_date]
val_data_zoomed = val_data_3_months[val_data_3_months.index >= zoom_start_date]
test_data_zoomed = test_data_sliced[test_data_sliced.index >= zoom_start_date]
predictions_zoomed = predictions_df[predictions_df.index >= zoom_start_date]

# Step 3: Plot zoomed-in view
fig, ax = plt.subplots(figsize=(12, 6))

# Plot zoomed train data
train_data_zoomed.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=1.5)

# Plot zoomed validation data
val_data_zoomed.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=1.5)

# Plot zoomed test data
test_data_zoomed.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=1.5)

# Plot zoomed model predictions
predictions_zoomed.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Plot confidence intervals for the predictions
upper_bound_zoomed = predictions_zoomed['predictions'] + conf_interval
lower_bound_zoomed = predictions_zoomed['predictions'] - conf_interval
ax.fill_between(predictions_zoomed.index, lower_bound_zoomed, upper_bound_zoomed, color='green', alpha=0.2, label="95% Confidence Interval")

# Step 4: Add labels, title, and legend
ax.set_xlabel('Date')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Zoomed-in View of Test Data and Predictions with Confidence Interval\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()

# Show grid and plot
plt.grid(True)
plt.show()

#### 48 hours

In [ ]:
# Step 5: Create a window-based dataset using scaled_data (24 jam)
def create_windows_6_48(data, window_size=32, forecast_horizon=224):
    X, y = [], []
    for i in range(len(data) - window_size - forecast_horizon + 1):
        X.append(data.iloc[i: i + window_size].values)
        y.append(data.iloc[i + window_size: i + window_size + forecast_horizon].values)
    return np.array(X), np.array(y)

In [ ]:
# Buat dataset berbasis window untuk setiap skenario (24 jam)
X_train_3, y_train_3 = create_windows_6_48(train_data_3_months)
X_train_6, y_train_6 = create_windows_6_48(train_data_6_months)
X_train_8, y_train_8 = create_windows_6_48(train_data_8_months)

X_val_3, y_val_3 = create_windows_6_48(val_data_3_months)
X_val_6, y_val_6 = create_windows_6_48(val_data_6_months)
X_val_8, y_val_8 = create_windows_6_48(val_data_8_months)

X_test_3, y_test_3 = create_windows_6_48(test_data_3_months)
X_test_6, y_test_6 = create_windows_6_48(test_data_6_months)
X_test_8, y_test_8 = create_windows_6_48(test_data_8_months)

# Cek hasilnya untuk melihat ukuran setiap dataset
print(f"Shape of X_train_3: {X_train_3.shape}, y_train_3: {y_train_3.shape}")
print(f"Shape of X_train_6: {X_train_6.shape}, y_train_6: {y_train_6.shape}")
print(f"Shape of X_train_8: {X_train_8.shape}, y_train_8: {y_train_8.shape}")
print(f"Shape of X_val_3: {X_val_3.shape}, y_val_3: {y_val_3.shape}")
print(f"Shape of X_val_6: {X_val_6.shape}, y_val_6: {y_val_6.shape}")
print(f"Shape of X_val_8: {X_val_8.shape}, y_val_8: {y_val_8.shape}")
print(f"Shape of X_test_3: {X_test_3.shape}, y_test_3: {y_test_3.shape}")
print(f"Shape of X_test_6: {X_test_6.shape}, y_test_6: {y_test_6.shape}")
print(f"Shape of X_test_8: {X_test_8.shape}, y_test_8: {y_test_8.shape}")


In [ ]:

import matplotlib.pyplot as plt

def train_and_evaluate_auto6_48(params, save_best_only=True, checkpoint_filename="autoformer6m48.pth.tar", patience=5, scaler=None):
    checkpoint_dir = "./checkpoints"
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)

    checkpoint_filepath = os.path.join(checkpoint_dir, checkpoint_filename)

    # Definisikan model
    model = Autoformer(
        enc_in=1,
        dec_in=1,
        c_out=1,
        seq_len=32,
        label_len=15,
        out_len=224,  # Sesuaikan dengan panjang output yang diharapkan
        d_model=params['d_model'],
        n_heads=params['n_heads'],
        e_layers=params['e_layers'],
        d_layers=params['d_layers'],
        d_ff=params['d_ff'],
        moving_avg=params['moving_avg'],
        dropout=params['dropout'],
        factor=params['factor'],
        activation=params['activation']
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])
    criterion = nn.MSELoss()  # Loss function
    
    best_val_loss = float('inf')  # Untuk menyimpan loss terbaik
    best_rmse = float('inf')  # Untuk menyimpan RMSE terbaik
    best_mape = float('inf')  # Untuk menyimpan MAPE terbaik
    num_epochs = 50  # Jumlah maksimum epoch
    epochs_no_improve = 0  # Untuk melacak jumlah epoch tanpa perbaikan
    early_stop = False  # Status untuk early stopping

    # Inisialisasi list untuk menyimpan train dan validation loss
    train_losses = []
    val_losses = []
    
    for epoch in range(num_epochs):
        if early_stop:
            print("Early stopping")
            break
            
        model.train()
        train_loss = 0.0
        for batch_X, batch_y in train_loader_6:
            optimizer.zero_grad()
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            x_enc = batch_X
            x_dec = batch_X[:, -15:, :]
            output = model(x_enc, x_dec)

            # Sesuaikan batch_y agar memiliki ukuran yang sama dengan output model
            batch_y = batch_y[:, :output.shape[1]]  # Potong atau sesuaikan batch_y

            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader_6)
        train_losses.append(train_loss)  # Simpan train loss

        # Evaluasi pada set validasi
        model.eval()
        val_loss = 0.0
        val_true, val_pred = [], []
        with torch.no_grad():
            for batch_X, batch_y in val_loader_6:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                x_enc = batch_X
                x_dec = batch_X[:, -15:, :]
                output = model(x_enc, x_dec)

                # Sesuaikan batch_y di set validasi
                batch_y = batch_y[:, :output.shape[1]]  # Sesuaikan ukuran batch_y

                val_true.append(batch_y.cpu().numpy())
                val_pred.append(output.cpu().numpy())
                loss = criterion(output, batch_y)
                val_loss += loss.item()

        val_loss /= len(val_loader_6)
        val_losses.append(val_loss)  # Simpan val loss

        # Inverse scaling untuk metrik jika scaler diberikan
        val_true = np.concatenate(val_true, axis=0).reshape(-1, 1)  # Sesuaikan bentuk data
        val_pred = np.concatenate(val_pred, axis=0).reshape(-1, 1)
        
        if scaler is not None:
            val_true = scaler.inverse_transform(val_true)  # Mengembalikan ke skala asli
            val_pred = scaler.inverse_transform(val_pred)

        # Hitung RMSE, MAPE
        rmse, mape = calculate_metrics(val_true, val_pred)
        
        print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.4f}")

        # Cek apakah ada perbaikan dalam validation loss tanpa min_delta
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse
            best_mape = mape
            epochs_no_improve = 0
            if save_best_only:
                save_checkpoint(model, optimizer, epoch, val_loss, filename=checkpoint_filepath)
                print(f"Saving checkpoint at epoch {epoch} with validation loss {val_loss:.4f}")
        else:
            epochs_no_improve += 1  # Tidak ada perbaikan, tambahkan hitungan
        
        # Cek apakah harus early stop
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            early_stop = True
    
    # Visualisasi train dan val loss setelah selesai pelatihan
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Train Loss and Validation Loss Over Epochs')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Kembalikan best_val_loss, rmse, dan mape
    return best_val_loss, best_rmse, best_mape


In [ ]:
def random_search_with_checkpoint6auto48(param_grid, num_iter=10):
    best_val_loss = float('inf')  # Untuk melacak loss validasi terbaik
    best_rmse = float('inf')  # Untuk melacak RMSE terbaik
    best_mape = float('inf')  # Untuk melacak MAPE terbaik
    best_params = None

    # Daftar hyperparameter dan pilih secara acak
    keys = list(param_grid.keys())
    evaluated_params = set()  # Untuk melacak kombinasi parameter yang dievaluasi

    for i in range(num_iter):  # Menjalankan pencarian acak sebanyak num_iter kali
        # Pilih hyperparameter secara acak
        params = {key: random.choice(param_grid[key]) for key in keys}

        # Buat representasi parameter yang dapat di-hash
        params_tuple = tuple(sorted(params.items()))
        if params_tuple in evaluated_params:
            continue  # Lewati parameter yang sudah dievaluasi

        evaluated_params.add(params_tuple)  # Tambahkan ke set yang dievaluasi
        print(f"Evaluating with params: {params}")

        # Jalankan pelatihan untuk kombinasi parameter dan dapatkan val_loss, rmse, mape
        val_loss, rmse, mape = train_and_evaluate_auto6_48(params, checkpoint_filename="autoformer6m48.pth.tar")

        # Simpan parameter terbaik jika val_loss lebih baik
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse  # Simpan RMSE terbaik
            best_mape = mape  # Simpan MAPE terbaik
            best_params = params

    # Cetak hasil terbaik setelah pencarian hyperparameter selesai
    print(f"\nBest Validation Loss: {best_val_loss:.4f}")
    print(f"Best RMSE: {best_rmse:.4f}")
    print(f"Best MAPE: {best_mape:.4f}")
    print(f"Best Hyperparameters: {best_params}")

    return best_params


In [ ]:
best_params = {
    'd_model': [256],          # Dimensi model
    'n_heads': [4],            # Jumlah attention heads
    'e_layers': [3],           # Jumlah layer di encoder
    'd_layers': [2],           # Jumlah layer di decoder
    'd_ff': [512],             # Dimensi feed-forward (disarankan 2x atau 4x dari d_model)
    'moving_avg': [25],        # Ukuran rata-rata bergerak (bisa disesuaikan)
    'dropout': [0.3],          # Dropout rate
    'factor': [1],             # Faktor autocorrelation (1 adalah default yang umum)
    'activation': ['relu'],      # Aktivasi, bisa 'relu' atau 'gelu'
    'learning_rate': [0.001]   # Laju pembelajaran
}

# Jalankan Grid Search dengan checkpoint untuk model 3 bulan
best_params = random_search_with_checkpoint6auto48(best_params)


In [ ]:
best_params = {
    'd_model': [256],          # Dimensi model
    'n_heads': [4],            # Jumlah attention heads
    'e_layers': [3],           # Jumlah layer di encoder
    'd_layers': [2],           # Jumlah layer di decoder
    'd_ff': [512],             # Dimensi feed-forward (disarankan 2x atau 4x dari d_model)
    'moving_avg': [25],        # Ukuran rata-rata bergerak (bisa disesuaikan)
    'dropout': [0.3],          # Dropout rate
    'factor': [1],             # Faktor autocorrelation (1 adalah default yang umum)
    'activation': ['relu'],      # Aktivasi, bisa 'relu' atau 'gelu'
    'learning_rate': [0.001]   # Laju pembelajaran
}

# Jalankan Grid Search dengan checkpoint untuk model 3 bulan
best_params = random_search_with_checkpoint6auto48(best_params)


In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Define your model (Autoformer) and optimizer
# Adjust the parameters according to your model architecture
model = Autoformer(
    enc_in=1,
    dec_in=1,
    c_out=1,
    seq_len=32,
    label_len=15,
    out_len=224,  # Expected output length
    d_model=256,  # Example configuration
    n_heads=4,
    e_layers=3,
    d_layers=2,
    d_ff=512,
    moving_avg=25,
    dropout=0.3,
    factor=1,
    activation='relu'
).to(device)

# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Step 1: Load model checkpoint
checkpoint_path = './checkpoints/autoformer6m48.pth.tar'  # Path to your checkpoint file

def load_checkpoint(model, optimizer, checkpoint_path):
    if checkpoint_path is not None:
        print(f"Loading checkpoint from '{checkpoint_path}'...")
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        epoch = checkpoint.get('epoch', None)
        best_val_loss = checkpoint.get('best_val_loss', None)  # Handle missing best_val_loss
        print(f"Checkpoint loaded. Resuming from epoch {epoch}")
        return epoch, best_val_loss
    else:
        print("No checkpoint provided, starting from scratch.")
        return None, None

# Load the checkpoint
epoch, best_val_loss = load_checkpoint(model, optimizer, checkpoint_path)

# Step 2: Generate predictions using the test data
model.eval()  # Switch the model to evaluation mode

predictions = []
with torch.no_grad():
    for batch_X, batch_y in test_loader_3:  # test_loader_3 is your DataLoader for the test set
        batch_X = batch_X.to(device)  # Move data to the same device as the model (GPU/CPU)
        x_enc = batch_X  # Encoder input
        x_dec = batch_X[:, -15:, :]  # Decoder input (using the last 15 time steps)
        
        # Get model predictions
        pred = model(x_enc, x_dec)
        
        # Take only the last time step prediction for each batch
        predictions.append(pred[:, -1, :].cpu().numpy())

# Step 3: Concatenate predictions into a single array
predictions = np.concatenate(predictions, axis=0)

# Step 4: Ensure predictions match the size of the test set (if necessary)
pred_len = len(predictions)
test_data_sliced = test_data_3_months.iloc[:pred_len]  # Slicing test data to match predictions length

# Step 5: Convert predictions to DataFrame for plotting
predictions_df = pd.DataFrame(predictions, index=test_data_sliced.index, columns=['predictions'])

# Step 6: Filter the data to start from December
start_date = '2023-12-01'  # Specify the start date for the plot

train_data_filtered = train_data_3_months[train_data_3_months.index >= start_date]
val_data_filtered = val_data_3_months[val_data_3_months.index >= start_date]
test_data_filtered = test_data_sliced[test_data_sliced.index >= start_date]
predictions_filtered = predictions_df[predictions_df.index >= start_date]

# Step 7: Calculate RMSE, MAPE, and residuals
actual_values = test_data_filtered['Hs'].values  # Actual test data values
predicted_values = predictions_filtered['predictions'].values  # Predicted values

# Calculate RMSE and MAPE
rmse = mean_squared_error(actual_values, predicted_values, squared=False)
mape = mean_absolute_percentage_error(actual_values, predicted_values)

# Calculate residuals and standard deviation for confidence intervals
residuals = actual_values - predicted_values
std_dev = np.std(residuals)
conf_interval = 1.96 * std_dev  # 95% confidence interval

# Print RMSE and MAPE
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape * 100:.2f}%")

# Step 8: Plot predictions along with train, validation, test data, and confidence intervals
fig, ax = plt.subplots(figsize=(10, 5))

# Plot train data with a blue solid line
train_data_filtered.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=2)

# Plot validation data with a purple dashed line
val_data_filtered.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=2)

# Plot actual test data with an orange dotted line
test_data_filtered.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=2)

# Plot model predictions with a green solid line
predictions_filtered.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Plot confidence intervals
upper_bound = predictions_filtered['predictions'] + conf_interval
lower_bound = predictions_filtered['predictions'] - conf_interval
ax.fill_between(predictions_filtered.index, lower_bound, upper_bound, color='green', alpha=0.3, label="95% Confidence Interval")

# Add labels and legend
ax.set_xlabel('Datetime')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Train, Validation, Test, and Predictions from December Onwards\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()

# Show the plot
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Assuming predictions, actual test data, and confidence interval calculations are already in place

# Step 1: Define the zoom-in start date to focus on the test data and predictions
zoom_start_date = '2023-12-20'  # Adjust this date to focus on the test and prediction period

# Step 2: Filter data to zoom-in on the specified range
train_data_zoomed = train_data_3_months[train_data_3_months.index >= zoom_start_date]
val_data_zoomed = val_data_3_months[val_data_3_months.index >= zoom_start_date]
test_data_zoomed = test_data_sliced[test_data_sliced.index >= zoom_start_date]
predictions_zoomed = predictions_df[predictions_df.index >= zoom_start_date]

# Step 3: Plot zoomed-in view
fig, ax = plt.subplots(figsize=(12, 6))

# Plot zoomed train data
train_data_zoomed.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=1.5)

# Plot zoomed validation data
val_data_zoomed.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=1.5)

# Plot zoomed test data
test_data_zoomed.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=1.5)

# Plot zoomed model predictions
predictions_zoomed.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Plot confidence intervals for the predictions
upper_bound_zoomed = predictions_zoomed['predictions'] + conf_interval
lower_bound_zoomed = predictions_zoomed['predictions'] - conf_interval
ax.fill_between(predictions_zoomed.index, lower_bound_zoomed, upper_bound_zoomed, color='green', alpha=0.2, label="95% Confidence Interval")

# Step 4: Add labels, title, and legend
ax.set_xlabel('Date')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Zoomed-in View of Test Data and Predictions with Confidence Interval\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()

# Show grid and plot
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import torch

def forecast_14_days(model, last_data, scaler=None):
    """
    Fungsi untuk memprediksi 14 hari ke depan menggunakan model yang telah dilatih.
    
    :param model: Model yang telah dilatih.
    :param last_data: Data terakhir yang akan digunakan sebagai input untuk prediksi (sequential input).
    :param scaler: Scaler yang digunakan pada data, untuk mengembalikan hasil prediksi ke skala asli jika perlu.
    """
    model.eval()  # Set model ke mode evaluasi
    
    # Prediksi 14 hari ke depan
    with torch.no_grad():
        # Ubah data terakhir menjadi tensor dan pindahkan ke device
        last_data_tensor = torch.Tensor(last_data).unsqueeze(0).to(device)  # Tambahkan batch dimension
        
        # Gunakan model untuk melakukan prediksi 14 langkah ke depan
        x_enc = last_data_tensor  # Input terakhir untuk encoder
        x_dec = last_data_tensor[:, -15:, :]  # Ambil 15 langkah terakhir sebagai input untuk decoder
        
        forecast = model(x_enc, x_dec)
    
    # Hasil prediksi
    forecast = forecast.cpu().numpy().reshape(-1, 1)  # Ubah bentuk menjadi array (14, 1)
    
    # Slice the first 14 predictions if forecast length is longer
    forecast = forecast[:14]  # Ambil hanya 14 hari prediksi pertama
    
    # Inverse scaling jika diperlukan
    if scaler is not None:
        forecast = scaler.inverse_transform(forecast)
    
    return forecast


def plot_forecast_14_days(forecast, title='14-Day Forecast'):
    """
    Fungsi untuk memplot prediksi 14 hari ke depan.
    
    :param forecast: Array hasil prediksi (14 hari ke depan).
    :param title: Judul plot.
    """
    days = list(range(1, 15))  # Hari 1 hingga 14
    
    plt.figure(figsize=(10, 6))
    plt.plot(days, forecast, label='Forecast', color='green', marker='o')
    plt.title(title, fontsize=14)
    plt.xlabel('Days')
    plt.ylabel('Forecasted Value')
    plt.grid(True)
    plt.legend()
    plt.show()


# Contoh penggunaan forecast 14 hari setelah pelatihan model
# Pastikan scaler telah di-fit ke data training sebelumnya
scaler.fit(train_data_6_months)

# Ambil data terbaru dari dataset validasi atau test set sebagai input
# last_data bisa berasal dari data validasi atau test set
last_data = test_data_6_months.values[-16:]  # Ambil 16 titik data terakhir sebagai input untuk prediksi

# Lakukan forecast 14 hari ke depan
forecast_14 = forecast_14_days(model, last_data, scaler=scaler)

# Visualisasikan prediksi 14 hari ke depan
plot_forecast_14_days(forecast_14, title='14-Day Forecast Using Trained Model')


#### 96 Hours

In [ ]:
# Step 5: Create a window-based dataset using scaled_data (24 jam)
def create_windows_6_96(data, window_size=64, forecast_horizon=224):
    X, y = [], []
    for i in range(len(data) - window_size - forecast_horizon + 1):
        X.append(data.iloc[i: i + window_size].values)
        y.append(data.iloc[i + window_size: i + window_size + forecast_horizon].values)
    return np.array(X), np.array(y)

In [ ]:
# Buat dataset berbasis window untuk setiap skenario (24 jam)
X_train_3, y_train_3 = create_windows_6_96(train_data_3_months)
X_train_6, y_train_6 = create_windows_6_96(train_data_6_months)
X_train_8, y_train_8 = create_windows_6_96(train_data_8_months)

X_val_3, y_val_3 = create_windows_6_96(val_data_3_months)
X_val_6, y_val_6 = create_windows_6_96(val_data_6_months)
X_val_8, y_val_8 = create_windows_6_96(val_data_8_months)

X_test_3, y_test_3 = create_windows_6_96(test_data_3_months)
X_test_6, y_test_6 = create_windows_6_96(test_data_6_months)
X_test_8, y_test_8 = create_windows_6_96(test_data_8_months)

# Cek hasilnya untuk melihat ukuran setiap dataset
print(f"Shape of X_train_3: {X_train_3.shape}, y_train_3: {y_train_3.shape}")
print(f"Shape of X_train_6: {X_train_6.shape}, y_train_6: {y_train_6.shape}")
print(f"Shape of X_train_8: {X_train_8.shape}, y_train_8: {y_train_8.shape}")
print(f"Shape of X_val_3: {X_val_3.shape}, y_val_3: {y_val_3.shape}")
print(f"Shape of X_val_6: {X_val_6.shape}, y_val_6: {y_val_6.shape}")
print(f"Shape of X_val_8: {X_val_8.shape}, y_val_8: {y_val_8.shape}")
print(f"Shape of X_test_3: {X_test_3.shape}, y_test_3: {y_test_3.shape}")
print(f"Shape of X_test_6: {X_test_6.shape}, y_test_6: {y_test_6.shape}")
print(f"Shape of X_test_8: {X_test_8.shape}, y_test_8: {y_test_8.shape}")


In [ ]:
# Step 6: Create DataLoaders
batch_size = 64
train_loader_3 = DataLoader(TensorDataset(torch.Tensor(X_train_3), torch.Tensor(y_train_3)), batch_size=batch_size, shuffle=True)
val_loader_3 = DataLoader(TensorDataset(torch.Tensor(X_val_3), torch.Tensor(y_val_3)), batch_size=batch_size, shuffle=False)
test_loader_3 = DataLoader(TensorDataset(torch.Tensor(X_test_3), torch.Tensor(y_test_3)), batch_size=batch_size, shuffle=False)

train_loader_6 = DataLoader(TensorDataset(torch.Tensor(X_train_6), torch.Tensor(y_train_6)), batch_size=batch_size, shuffle=True)
val_loader_6 = DataLoader(TensorDataset(torch.Tensor(X_val_6), torch.Tensor(y_val_6)), batch_size=batch_size, shuffle=False)
test_loader_6 = DataLoader(TensorDataset(torch.Tensor(X_test_6), torch.Tensor(y_test_6)), batch_size=batch_size, shuffle=False)

train_loader_8 = DataLoader(TensorDataset(torch.Tensor(X_train_8), torch.Tensor(y_train_8)), batch_size=batch_size, shuffle=True)
val_loader_8 = DataLoader(TensorDataset(torch.Tensor(X_val_8), torch.Tensor(y_val_8)), batch_size=batch_size, shuffle=False)
test_loader_8 = DataLoader(TensorDataset(torch.Tensor(X_test_8), torch.Tensor(y_test_8)), batch_size=batch_size, shuffle=False)

In [ ]:
import matplotlib.pyplot as plt

def train_and_evaluate_auto6_96(params, save_best_only=True, checkpoint_filename="autoformer6m96.pth.tar", patience=5, scaler=None):
    checkpoint_dir = "./checkpoints"
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)

    checkpoint_filepath = os.path.join(checkpoint_dir, checkpoint_filename)

    # Definisikan model
    model = Autoformer(
        enc_in=1,
        dec_in=1,
        c_out=1,
        seq_len=64,
        label_len=15,
        out_len=224,  # Sesuaikan dengan panjang output yang diharapkan
        d_model=params['d_model'],
        n_heads=params['n_heads'],
        e_layers=params['e_layers'],
        d_layers=params['d_layers'],
        d_ff=params['d_ff'],
        moving_avg=params['moving_avg'],
        dropout=params['dropout'],
        factor=params['factor'],
        activation=params['activation']
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])
    criterion = nn.MSELoss()  # Loss function
    
    best_val_loss = float('inf')  # Untuk menyimpan loss terbaik
    best_rmse = float('inf')  # Untuk menyimpan RMSE terbaik
    best_mape = float('inf')  # Untuk menyimpan MAPE terbaik
    num_epochs = 50  # Jumlah maksimum epoch
    epochs_no_improve = 0  # Untuk melacak jumlah epoch tanpa perbaikan
    early_stop = False  # Status untuk early stopping

    # Inisialisasi list untuk menyimpan train dan validation loss
    train_losses = []
    val_losses = []
    
    for epoch in range(num_epochs):
        if early_stop:
            print("Early stopping")
            break
            
        model.train()
        train_loss = 0.0
        for batch_X, batch_y in train_loader_6:
            optimizer.zero_grad()
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            x_enc = batch_X
            x_dec = batch_X[:, -15:, :]
            output = model(x_enc, x_dec)

            # Sesuaikan batch_y agar memiliki ukuran yang sama dengan output model
            batch_y = batch_y[:, :output.shape[1]]  # Potong atau sesuaikan batch_y

            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader_6)
        train_losses.append(train_loss)  # Simpan train loss

        # Evaluasi pada set validasi
        model.eval()
        val_loss = 0.0
        val_true, val_pred = [], []
        with torch.no_grad():
            for batch_X, batch_y in val_loader_6:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                x_enc = batch_X
                x_dec = batch_X[:, -15:, :]
                output = model(x_enc, x_dec)

                # Sesuaikan batch_y di set validasi
                batch_y = batch_y[:, :output.shape[1]]  # Sesuaikan ukuran batch_y

                val_true.append(batch_y.cpu().numpy())
                val_pred.append(output.cpu().numpy())
                loss = criterion(output, batch_y)
                val_loss += loss.item()

        val_loss /= len(val_loader_6)
        val_losses.append(val_loss)  # Simpan val loss

        # Inverse scaling untuk metrik jika scaler diberikan
        val_true = np.concatenate(val_true, axis=0).reshape(-1, 1)  # Sesuaikan bentuk data
        val_pred = np.concatenate(val_pred, axis=0).reshape(-1, 1)
        
        if scaler is not None:
            val_true = scaler.inverse_transform(val_true)  # Mengembalikan ke skala asli
            val_pred = scaler.inverse_transform(val_pred)

        # Hitung RMSE, MAPE
        rmse, mape = calculate_metrics(val_true, val_pred)
        
        print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.4f}")

        # Cek apakah ada perbaikan dalam validation loss tanpa min_delta
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse
            best_mape = mape
            epochs_no_improve = 0
            if save_best_only:
                save_checkpoint(model, optimizer, epoch, val_loss, filename=checkpoint_filepath)
                print(f"Saving checkpoint at epoch {epoch} with validation loss {val_loss:.4f}")
        else:
            epochs_no_improve += 1  # Tidak ada perbaikan, tambahkan hitungan
        
        # Cek apakah harus early stop
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            early_stop = True
    
    # Visualisasi train dan val loss setelah selesai pelatihan
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Train Loss and Validation Loss Over Epochs')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Kembalikan best_val_loss, rmse, dan mape
    return best_val_loss, best_rmse, best_mape


In [ ]:
import random

device = 'mps' if torch.cuda.is_available() else 'cpu'
def random_search_with_checkpoint6auto96(param_grid, num_iter=10):
    best_val_loss = float('inf')  # Untuk melacak loss validasi terbaik
    best_rmse = float('inf')  # Untuk melacak RMSE terbaik
    best_mape = float('inf')  # Untuk melacak MAPE terbaik
    best_params = None

    # Daftar hyperparameter dan pilih secara acak
    keys = list(param_grid.keys())
    evaluated_params = set()  # Untuk melacak kombinasi parameter yang dievaluasi

    for i in range(num_iter):  # Menjalankan pencarian acak sebanyak num_iter kali
        # Pilih hyperparameter secara acak
        params = {key: random.choice(param_grid[key]) for key in keys}

        # Buat representasi parameter yang dapat di-hash
        params_tuple = tuple(sorted(params.items()))
        if params_tuple in evaluated_params:
            continue  # Lewati parameter yang sudah dievaluasi

        evaluated_params.add(params_tuple)  # Tambahkan ke set yang dievaluasi
        print(f"Evaluating with params: {params}")

        # Jalankan pelatihan untuk kombinasi parameter dan dapatkan val_loss, rmse, mape
        val_loss, rmse, mape = train_and_evaluate_auto6_96(params, checkpoint_filename="autoformer6m96.pth.tar")

        # Simpan parameter terbaik jika val_loss lebih baik
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse  # Simpan RMSE terbaik
            best_mape = mape  # Simpan MAPE terbaik
            best_params = params

    # Cetak hasil terbaik setelah pencarian hyperparameter selesai
    print(f"\nBest Validation Loss: {best_val_loss:.4f}")
    print(f"Best RMSE: {best_rmse:.4f}")
    print(f"Best MAPE: {best_mape:.4f}")
    print(f"Best Hyperparameters: {best_params}")

    return best_params


In [ ]:
best_params = {
    'd_model': [256],          # Dimensi model
    'n_heads': [4],            # Jumlah attention heads
    'e_layers': [3],           # Jumlah layer di encoder
    'd_layers': [2],           # Jumlah layer di decoder
    'd_ff': [512],             # Dimensi feed-forward (disarankan 2x atau 4x dari d_model)
    'moving_avg': [25],        # Ukuran rata-rata bergerak (bisa disesuaikan)
    'dropout': [0.3],          # Dropout rate
    'factor': [1],             # Faktor autocorrelation (1 adalah default yang umum)
    'activation': ['relu'],      # Aktivasi, bisa 'relu' atau 'gelu'
    'learning_rate': [0.001]   # Laju pembelajaran
}

# Jalankan Grid Search dengan checkpoint untuk model 3 bulan
best_params = random_search_with_checkpoint6auto96(best_params)


In [ ]:
best_params = {
    'd_model': [256],          # Dimensi model
    'n_heads': [4],            # Jumlah attention heads
    'e_layers': [3],           # Jumlah layer di encoder
    'd_layers': [2],           # Jumlah layer di decoder
    'd_ff': [512],             # Dimensi feed-forward (disarankan 2x atau 4x dari d_model)
    'moving_avg': [25],        # Ukuran rata-rata bergerak (bisa disesuaikan)
    'dropout': [0.3],          # Dropout rate
    'factor': [1],             # Faktor autocorrelation (1 adalah default yang umum)
    'activation': ['relu'],      # Aktivasi, bisa 'relu' atau 'gelu'
    'learning_rate': [0.001]   # Laju pembelajaran
}

# Jalankan Grid Search dengan checkpoint untuk model 3 bulan
best_params = random_search_with_checkpoint6auto96(best_params)


In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Define your model (Autoformer) and optimizer
# Adjust the parameters according to your model architecture
model = Autoformer(
    enc_in=1,
    dec_in=1,
    c_out=1,
    seq_len=64,
    label_len=15,
    out_len=224,  # Expected output length
    d_model=256,  # Example configuration
    n_heads=4,
    e_layers=3,
    d_layers=2,
    d_ff=512,
    moving_avg=25,
    dropout=0.3,
    factor=1,
    activation='relu'
).to(device)

# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Step 1: Load model checkpoint
checkpoint_path = './checkpoints/autoformer6m96.pth.tar'  # Path to your checkpoint file

def load_checkpoint(model, optimizer, checkpoint_path):
    if checkpoint_path is not None:
        print(f"Loading checkpoint from '{checkpoint_path}'...")
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        epoch = checkpoint.get('epoch', None)
        best_val_loss = checkpoint.get('best_val_loss', None)  # Handle missing best_val_loss
        print(f"Checkpoint loaded. Resuming from epoch {epoch}")
        return epoch, best_val_loss
    else:
        print("No checkpoint provided, starting from scratch.")
        return None, None

# Load the checkpoint
epoch, best_val_loss = load_checkpoint(model, optimizer, checkpoint_path)

# Step 2: Generate predictions using the test data
model.eval()  # Switch the model to evaluation mode

predictions = []
with torch.no_grad():
    for batch_X, batch_y in test_loader_6:  # test_loader_3 is your DataLoader for the test set
        batch_X = batch_X.to(device)  # Move data to the same device as the model (GPU/CPU)
        x_enc = batch_X  # Encoder input
        x_dec = batch_X[:, -15:, :]  # Decoder input (using the last 15 time steps)
        
        # Get model predictions
        pred = model(x_enc, x_dec)
        
        # Take only the last time step prediction for each batch
        predictions.append(pred[:, -1, :].cpu().numpy())

# Step 3: Concatenate predictions into a single array
predictions = np.concatenate(predictions, axis=0)

# Step 4: Ensure predictions match the size of the test set (if necessary)
pred_len = len(predictions)
test_data_sliced = test_data_3_months.iloc[:pred_len]  # Slicing test data to match predictions length

# Step 5: Convert predictions to DataFrame for plotting
predictions_df = pd.DataFrame(predictions, index=test_data_sliced.index, columns=['predictions'])

# Step 6: Filter the data to start from December
start_date = '2023-12-01'  # Specify the start date for the plot

train_data_filtered = train_data_6_months[train_data_6_months.index >= start_date]
val_data_filtered = val_data_6_months[val_data_6_months.index >= start_date]
test_data_filtered = test_data_sliced[test_data_sliced.index >= start_date]
predictions_filtered = predictions_df[predictions_df.index >= start_date]

# Step 7: Calculate RMSE, MAPE, and residuals
actual_values = test_data_filtered['Hs'].values  # Actual test data values
predicted_values = predictions_filtered['predictions'].values  # Predicted values

# Calculate RMSE and MAPE
rmse = mean_squared_error(actual_values, predicted_values, squared=False)
mape = mean_absolute_percentage_error(actual_values, predicted_values)

# Calculate residuals and standard deviation for confidence intervals
residuals = actual_values - predicted_values
std_dev = np.std(residuals)
conf_interval = 1.96 * std_dev  # 95% confidence interval

# Print RMSE and MAPE
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape * 100:.2f}%")

# Step 8: Plot predictions along with train, validation, test data, and confidence intervals
fig, ax = plt.subplots(figsize=(10, 5))

# Plot train data with a blue solid line
train_data_filtered.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=2)

# Plot validation data with a purple dashed line
val_data_filtered.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=2)

# Plot actual test data with an orange dotted line
test_data_filtered.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=2)

# Plot model predictions with a green solid line
predictions_filtered.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Plot confidence intervals
upper_bound = predictions_filtered['predictions'] + conf_interval
lower_bound = predictions_filtered['predictions'] - conf_interval
ax.fill_between(predictions_filtered.index, lower_bound, upper_bound, color='green', alpha=0.3, label="95% Confidence Interval")

# Add labels and legend
ax.set_xlabel('Datetime')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Train, Validation, Test, and Predictions from December Onwards\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()

# Show the plot
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import torch

def forecast_14_days(model, last_data, scaler=None):
    """
    Fungsi untuk memprediksi 14 hari ke depan menggunakan model yang telah dilatih.
    
    :param model: Model yang telah dilatih.
    :param last_data: Data terakhir yang akan digunakan sebagai input untuk prediksi (sequential input).
    :param scaler: Scaler yang digunakan pada data, untuk mengembalikan hasil prediksi ke skala asli jika perlu.
    """
    model.eval()  # Set model ke mode evaluasi
    
    # Prediksi 14 hari ke depan
    with torch.no_grad():
        # Ubah data terakhir menjadi tensor dan pindahkan ke device
        last_data_tensor = torch.Tensor(last_data).unsqueeze(0).to(device)  # Tambahkan batch dimension
        
        # Gunakan model untuk melakukan prediksi 14 langkah ke depan
        x_enc = last_data_tensor  # Input terakhir untuk encoder
        x_dec = last_data_tensor[:, -15:, :]  # Ambil 15 langkah terakhir sebagai input untuk decoder
        
        forecast = model(x_enc, x_dec)
    
    # Hasil prediksi
    forecast = forecast.cpu().numpy().reshape(-1, 1)  # Ubah bentuk menjadi array (14, 1)
    
    # Slice the first 14 predictions if forecast length is longer
    forecast = forecast[:14]  # Ambil hanya 14 hari prediksi pertama
    
    # Inverse scaling jika diperlukan
    if scaler is not None:
        forecast = scaler.inverse_transform(forecast)
    
    return forecast


def plot_forecast_14_days(forecast, title='14-Day Forecast'):
    """
    Fungsi untuk memplot prediksi 14 hari ke depan.
    
    :param forecast: Array hasil prediksi (14 hari ke depan).
    :param title: Judul plot.
    """
    days = list(range(1, 15))  # Hari 1 hingga 14
    
    plt.figure(figsize=(10, 6))
    plt.plot(days, forecast, label='Forecast', color='green', marker='o')
    plt.title(title, fontsize=14)
    plt.xlabel('Days')
    plt.ylabel('Forecasted Value')
    plt.grid(True)
    plt.legend()
    plt.show()


# Contoh penggunaan forecast 14 hari setelah pelatihan model
# Pastikan scaler telah di-fit ke data training sebelumnya
scaler.fit(train_data_6_months)

# Ambil data terbaru dari dataset validasi atau test set sebagai input
# last_data bisa berasal dari data validasi atau test set
last_data = test_data_6_months.values[-16:]  # Ambil 16 titik data terakhir sebagai input untuk prediksi

# Lakukan forecast 14 hari ke depan
forecast_14 = forecast_14_days(model, last_data, scaler=scaler)

# Visualisasikan prediksi 14 hari ke depan
plot_forecast_14_days(forecast_14, title='14-Day Forecast Using Trained Model')


#### Train 8 months

#### 24 Hours

In [ ]:
# Step 5: Create a window-based dataset using scaled_data (24 jam)
def create_windows_8_24(data, window_size=16, forecast_horizon=224):
    X, y = [], []
    for i in range(len(data) - window_size - forecast_horizon + 1):
        X.append(data.iloc[i: i + window_size].values)
        y.append(data.iloc[i + window_size: i + window_size + forecast_horizon].values)
    return np.array(X), np.array(y)

In [ ]:
# Buat dataset berbasis window untuk setiap skenario (24 jam)
X_train_3, y_train_3 = create_windows_8_24(train_data_3_months)
X_train_6, y_train_6 = create_windows_8_24(train_data_6_months)
X_train_8, y_train_8 = create_windows_8_24(train_data_8_months)

X_val_3, y_val_3 = create_windows_8_24(val_data_3_months)
X_val_6, y_val_6 = create_windows_8_24(val_data_6_months)
X_val_8, y_val_8 = create_windows_8_24(val_data_8_months)

X_test_3, y_test_3 = create_windows_8_24(test_data_3_months)
X_test_6, y_test_6 = create_windows_8_24(test_data_6_months)
X_test_8, y_test_8 = create_windows_8_24(test_data_8_months)

# Cek hasilnya untuk melihat ukuran setiap dataset
print(f"Shape of X_train_3: {X_train_3.shape}, y_train_3: {y_train_3.shape}")
print(f"Shape of X_train_6: {X_train_6.shape}, y_train_6: {y_train_6.shape}")
print(f"Shape of X_train_8: {X_train_8.shape}, y_train_8: {y_train_8.shape}")
print(f"Shape of X_val_3: {X_val_3.shape}, y_val_3: {y_val_3.shape}")
print(f"Shape of X_val_6: {X_val_6.shape}, y_val_6: {y_val_6.shape}")
print(f"Shape of X_val_8: {X_val_8.shape}, y_val_8: {y_val_8.shape}")
print(f"Shape of X_test_3: {X_test_3.shape}, y_test_3: {y_test_3.shape}")
print(f"Shape of X_test_6: {X_test_6.shape}, y_test_6: {y_test_6.shape}")
print(f"Shape of X_test_8: {X_test_8.shape}, y_test_8: {y_test_8.shape}")


In [ ]:
# Step 6: Create DataLoaders
batch_size = 64
train_loader_3 = DataLoader(TensorDataset(torch.Tensor(X_train_3), torch.Tensor(y_train_3)), batch_size=batch_size, shuffle=True)
val_loader_3 = DataLoader(TensorDataset(torch.Tensor(X_val_3), torch.Tensor(y_val_3)), batch_size=batch_size, shuffle=False)
test_loader_3 = DataLoader(TensorDataset(torch.Tensor(X_test_3), torch.Tensor(y_test_3)), batch_size=batch_size, shuffle=False)

train_loader_6 = DataLoader(TensorDataset(torch.Tensor(X_train_6), torch.Tensor(y_train_6)), batch_size=batch_size, shuffle=True)
val_loader_6 = DataLoader(TensorDataset(torch.Tensor(X_val_6), torch.Tensor(y_val_6)), batch_size=batch_size, shuffle=False)
test_loader_6 = DataLoader(TensorDataset(torch.Tensor(X_test_6), torch.Tensor(y_test_6)), batch_size=batch_size, shuffle=False)

train_loader_8 = DataLoader(TensorDataset(torch.Tensor(X_train_8), torch.Tensor(y_train_8)), batch_size=batch_size, shuffle=True)
val_loader_8 = DataLoader(TensorDataset(torch.Tensor(X_val_8), torch.Tensor(y_val_8)), batch_size=batch_size, shuffle=False)
test_loader_8 = DataLoader(TensorDataset(torch.Tensor(X_test_8), torch.Tensor(y_test_8)), batch_size=batch_size, shuffle=False)

In [ ]:
import matplotlib.pyplot as plt

def train_and_evaluate_auto8_24(params, save_best_only=True, checkpoint_filename="autoformer8m24.pth.tar", patience=5, scaler=None):
    checkpoint_dir = "./checkpoints"
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)

    checkpoint_filepath = os.path.join(checkpoint_dir, checkpoint_filename)

    # Definisikan model
    model = Autoformer(
        enc_in=1,
        dec_in=1,
        c_out=1,
        seq_len=16,
        label_len=15,
        out_len=224,  # Sesuaikan dengan panjang output yang diharapkan
        d_model=params['d_model'],
        n_heads=params['n_heads'],
        e_layers=params['e_layers'],
        d_layers=params['d_layers'],
        d_ff=params['d_ff'],
        moving_avg=params['moving_avg'],
        dropout=params['dropout'],
        factor=params['factor'],
        activation=params['activation']
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])
    criterion = nn.MSELoss()  # Loss function
    
    best_val_loss = float('inf')  # Untuk menyimpan loss terbaik
    best_rmse = float('inf')  # Untuk menyimpan RMSE terbaik
    best_mape = float('inf')  # Untuk menyimpan MAPE terbaik
    num_epochs = 50  # Jumlah maksimum epoch
    epochs_no_improve = 0  # Untuk melacak jumlah epoch tanpa perbaikan
    early_stop = False  # Status untuk early stopping

    # Inisialisasi list untuk menyimpan train dan validation loss
    train_losses = []
    val_losses = []
    
    for epoch in range(num_epochs):
        if early_stop:
            print("Early stopping")
            break
            
        model.train()
        train_loss = 0.0
        for batch_X, batch_y in train_loader_8:
            optimizer.zero_grad()
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            x_enc = batch_X
            x_dec = batch_X[:, -15:, :]
            output = model(x_enc, x_dec)

            # Sesuaikan batch_y agar memiliki ukuran yang sama dengan output model
            batch_y = batch_y[:, :output.shape[1]]  # Potong atau sesuaikan batch_y

            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader_8)
        train_losses.append(train_loss)  # Simpan train loss

        # Evaluasi pada set validasi
        model.eval()
        val_loss = 0.0
        val_true, val_pred = [], []
        with torch.no_grad():
            for batch_X, batch_y in val_loader_8:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                x_enc = batch_X
                x_dec = batch_X[:, -15:, :]
                output = model(x_enc, x_dec)

                # Sesuaikan batch_y di set validasi
                batch_y = batch_y[:, :output.shape[1]]  # Sesuaikan ukuran batch_y

                val_true.append(batch_y.cpu().numpy())
                val_pred.append(output.cpu().numpy())
                loss = criterion(output, batch_y)
                val_loss += loss.item()

        val_loss /= len(val_loader_8)
        val_losses.append(val_loss)  # Simpan val loss

        # Inverse scaling untuk metrik jika scaler diberikan
        val_true = np.concatenate(val_true, axis=0).reshape(-1, 1)  # Sesuaikan bentuk data
        val_pred = np.concatenate(val_pred, axis=0).reshape(-1, 1)
        
        if scaler is not None:
            val_true = scaler.inverse_transform(val_true)  # Mengembalikan ke skala asli
            val_pred = scaler.inverse_transform(val_pred)

        # Hitung RMSE, MAPE
        rmse, mape = calculate_metrics(val_true, val_pred)
        
        print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.4f}")

        # Cek apakah ada perbaikan dalam validation loss tanpa min_delta
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse
            best_mape = mape
            epochs_no_improve = 0
            if save_best_only:
                save_checkpoint(model, optimizer, epoch, val_loss, filename=checkpoint_filepath)
                print(f"Saving checkpoint at epoch {epoch} with validation loss {val_loss:.4f}")
        else:
            epochs_no_improve += 1  # Tidak ada perbaikan, tambahkan hitungan
        
        # Cek apakah harus early stop
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            early_stop = True
    
    # Visualisasi train dan val loss setelah selesai pelatihan
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Train Loss and Validation Loss Over Epochs')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Kembalikan best_val_loss, rmse, dan mape
    return best_val_loss, best_rmse, best_mape


In [ ]:
import random

device = 'mps' if torch.cuda.is_available() else 'cpu'
def random_search_with_checkpoint8auto24(param_grid, num_iter=10):
    best_val_loss = float('inf')  # Untuk melacak loss validasi terbaik
    best_rmse = float('inf')  # Untuk melacak RMSE terbaik
    best_mape = float('inf')  # Untuk melacak MAPE terbaik
    best_params = None

    # Daftar hyperparameter dan pilih secara acak
    keys = list(param_grid.keys())
    evaluated_params = set()  # Untuk melacak kombinasi parameter yang dievaluasi

    for i in range(num_iter):  # Menjalankan pencarian acak sebanyak num_iter kali
        # Pilih hyperparameter secara acak
        params = {key: random.choice(param_grid[key]) for key in keys}

        # Buat representasi parameter yang dapat di-hash
        params_tuple = tuple(sorted(params.items()))
        if params_tuple in evaluated_params:
            continue  # Lewati parameter yang sudah dievaluasi

        evaluated_params.add(params_tuple)  # Tambahkan ke set yang dievaluasi
        print(f"Evaluating with params: {params}")

        # Jalankan pelatihan untuk kombinasi parameter dan dapatkan val_loss, rmse, mape
        val_loss, rmse, mape = train_and_evaluate_auto8_24(params, checkpoint_filename="autoformer8m24.pth.tar")

        # Simpan parameter terbaik jika val_loss lebih baik
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse  # Simpan RMSE terbaik
            best_mape = mape  # Simpan MAPE terbaik
            best_params = params

    # Cetak hasil terbaik setelah pencarian hyperparameter selesai
    print(f"\nBest Validation Loss: {best_val_loss:.4f}")
    print(f"Best RMSE: {best_rmse:.4f}")
    print(f"Best MAPE: {best_mape:.4f}")
    print(f"Best Hyperparameters: {best_params}")

    return best_params


In [ ]:
best_params = {
    'd_model': [256],          # Dimensi model
    'n_heads': [4],            # Jumlah attention heads
    'e_layers': [3],           # Jumlah layer di encoder
    'd_layers': [2],           # Jumlah layer di decoder
    'd_ff': [512],             # Dimensi feed-forward (disarankan 2x atau 4x dari d_model)
    'moving_avg': [25],        # Ukuran rata-rata bergerak (bisa disesuaikan)
    'dropout': [0.3],          # Dropout rate
    'factor': [1],             # Faktor autocorrelation (1 adalah default yang umum)
    'activation': ['relu'],      # Aktivasi, bisa 'relu' atau 'gelu'
    'learning_rate': [0.001]   # Laju pembelajaran
}

# Jalankan Grid Search dengan checkpoint untuk model 3 bulan
best_params = random_search_with_checkpoint8auto24(best_params)


In [ ]:
best_params = {
    'd_model': [256],          # Dimensi model
    'n_heads': [4],            # Jumlah attention heads
    'e_layers': [3],           # Jumlah layer di encoder
    'd_layers': [2],           # Jumlah layer di decoder
    'd_ff': [512],             # Dimensi feed-forward (disarankan 2x atau 4x dari d_model)
    'moving_avg': [25],        # Ukuran rata-rata bergerak (bisa disesuaikan)
    'dropout': [0.3],          # Dropout rate
    'factor': [1],             # Faktor autocorrelation (1 adalah default yang umum)
    'activation': ['relu'],      # Aktivasi, bisa 'relu' atau 'gelu'
    'learning_rate': [0.001]   # Laju pembelajaran
}

# Jalankan Grid Search dengan checkpoint untuk model 3 bulan
best_params = random_search_with_checkpoint8auto24(best_params)


In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Define your model (Autoformer) and optimizer
# Adjust the parameters according to your model architecture
model = Autoformer(
    enc_in=1,
    dec_in=1,
    c_out=1,
    seq_len=16,
    label_len=15,
    out_len=224,  # Expected output length
    d_model=256,  # Example configuration
    n_heads=4,
    e_layers=3,
    d_layers=2,
    d_ff=512,
    moving_avg=25,
    dropout=0.3,
    factor=1,
    activation='relu'
).to(device)

# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Step 1: Load model checkpoint
checkpoint_path = './checkpoints/autoformer8m24.pth.tar'  # Path to your checkpoint file

def load_checkpoint(model, optimizer, checkpoint_path):
    if checkpoint_path is not None:
        print(f"Loading checkpoint from '{checkpoint_path}'...")
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        epoch = checkpoint.get('epoch', None)
        best_val_loss = checkpoint.get('best_val_loss', None)  # Handle missing best_val_loss
        print(f"Checkpoint loaded. Resuming from epoch {epoch}")
        return epoch, best_val_loss
    else:
        print("No checkpoint provided, starting from scratch.")
        return None, None

# Load the checkpoint
epoch, best_val_loss = load_checkpoint(model, optimizer, checkpoint_path)

# Step 2: Generate predictions using the test data
model.eval()  # Switch the model to evaluation mode

predictions = []
with torch.no_grad():
    for batch_X, batch_y in test_loader_8:  # test_loader_3 is your DataLoader for the test set
        batch_X = batch_X.to(device)  # Move data to the same device as the model (GPU/CPU)
        x_enc = batch_X  # Encoder input
        x_dec = batch_X[:, -15:, :]  # Decoder input (using the last 15 time steps)
        
        # Get model predictions
        pred = model(x_enc, x_dec)
        
        # Take only the last time step prediction for each batch
        predictions.append(pred[:, -1, :].cpu().numpy())

# Step 3: Concatenate predictions into a single array
predictions = np.concatenate(predictions, axis=0)

# Step 4: Ensure predictions match the size of the test set (if necessary)
pred_len = len(predictions)
test_data_sliced = test_data_8_months.iloc[:pred_len]  # Slicing test data to match predictions length

# Step 5: Convert predictions to DataFrame for plotting
predictions_df = pd.DataFrame(predictions, index=test_data_sliced.index, columns=['predictions'])

# Step 6: Filter the data to start from December
start_date = '2023-12-01'  # Specify the start date for the plot

train_data_filtered = train_data_8_months[train_data_8_months.index >= start_date]
val_data_filtered = val_data_8_months[val_data_8_months.index >= start_date]
test_data_filtered = test_data_sliced[test_data_sliced.index >= start_date]
predictions_filtered = predictions_df[predictions_df.index >= start_date]

# Step 7: Calculate RMSE, MAPE, and residuals
actual_values = test_data_filtered['Hs'].values  # Actual test data values
predicted_values = predictions_filtered['predictions'].values  # Predicted values

# Calculate RMSE and MAPE
rmse = mean_squared_error(actual_values, predicted_values, squared=False)
mape = mean_absolute_percentage_error(actual_values, predicted_values)

# Calculate residuals and standard deviation for confidence intervals
residuals = actual_values - predicted_values
std_dev = np.std(residuals)
conf_interval = 1.96 * std_dev  # 95% confidence interval

# Print RMSE and MAPE
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape * 100:.2f}%")

# Step 8: Plot predictions along with train, validation, test data, and confidence intervals
fig, ax = plt.subplots(figsize=(10, 5))

# Plot train data with a blue solid line
train_data_filtered.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=2)

# Plot validation data with a purple dashed line
val_data_filtered.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=2)

# Plot actual test data with an orange dotted line
test_data_filtered.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=2)

# Plot model predictions with a green solid line
predictions_filtered.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Plot confidence intervals
upper_bound = predictions_filtered['predictions'] + conf_interval
lower_bound = predictions_filtered['predictions'] - conf_interval
ax.fill_between(predictions_filtered.index, lower_bound, upper_bound, color='green', alpha=0.3, label="95% Confidence Interval")

# Add labels and legend
ax.set_xlabel('Datetime')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Train, Validation, Test, and Predictions from December Onwards\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()

# Show the plot
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Assuming predictions, actual test data, and confidence interval calculations are already in place

# Step 1: Define the zoom-in start date to focus on the test data and predictions
zoom_start_date = '2023-12-05'  # Adjust this date to focus on the test and prediction period

# Step 2: Filter data to zoom-in on the specified range
train_data_zoomed = train_data_8_months[train_data_8_months.index >= zoom_start_date]
val_data_zoomed = val_data_8_months[val_data_8_months.index >= zoom_start_date]
test_data_zoomed = test_data_sliced[test_data_sliced.index >= zoom_start_date]
predictions_zoomed = predictions_df[predictions_df.index >= zoom_start_date]

# Step 3: Plot zoomed-in view
fig, ax = plt.subplots(figsize=(12, 6))

# Plot zoomed train data
train_data_zoomed.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=1.5)

# Plot zoomed validation data
val_data_zoomed.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=1.5)

# Plot zoomed test data
test_data_zoomed.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=1.5)

# Plot zoomed model predictions
predictions_zoomed.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Plot confidence intervals for the predictions
upper_bound_zoomed = predictions_zoomed['predictions'] + conf_interval
lower_bound_zoomed = predictions_zoomed['predictions'] - conf_interval
ax.fill_between(predictions_zoomed.index, lower_bound_zoomed, upper_bound_zoomed, color='green', alpha=0.2, label="95% Confidence Interval")

# Step 4: Add labels, title, and legend
ax.set_xlabel('Date')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Zoomed-in View of Test Data and Predictions with Confidence Interval\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()

# Show grid and plot
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import torch

def forecast_14_days(model, last_data, scaler=None):
    """
    Fungsi untuk memprediksi 14 hari ke depan menggunakan model yang telah dilatih.
    
    :param model: Model yang telah dilatih.
    :param last_data: Data terakhir yang akan digunakan sebagai input untuk prediksi (sequential input).
    :param scaler: Scaler yang digunakan pada data, untuk mengembalikan hasil prediksi ke skala asli jika perlu.
    """
    model.eval()  # Set model ke mode evaluasi
    
    # Prediksi 14 hari ke depan
    with torch.no_grad():
        # Ubah data terakhir menjadi tensor dan pindahkan ke device
        last_data_tensor = torch.Tensor(last_data).unsqueeze(0).to(device)  # Tambahkan batch dimension
        
        # Gunakan model untuk melakukan prediksi 14 langkah ke depan
        x_enc = last_data_tensor  # Input terakhir untuk encoder
        x_dec = last_data_tensor[:, -15:, :]  # Ambil 15 langkah terakhir sebagai input untuk decoder
        
        forecast = model(x_enc, x_dec)
    
    # Hasil prediksi
    forecast = forecast.cpu().numpy().reshape(-1, 1)  # Ubah bentuk menjadi array (14, 1)
    
    # Slice the first 14 predictions if forecast length is longer
    forecast = forecast[:14]  # Ambil hanya 14 hari prediksi pertama
    
    # Inverse scaling jika diperlukan
    if scaler is not None:
        forecast = scaler.inverse_transform(forecast)
    
    return forecast


def plot_forecast_14_days(forecast, title='14-Day Forecast'):
    """
    Fungsi untuk memplot prediksi 14 hari ke depan.
    
    :param forecast: Array hasil prediksi (14 hari ke depan).
    :param title: Judul plot.
    """
    days = list(range(1, 15))  # Hari 1 hingga 14
    
    plt.figure(figsize=(10, 6))
    plt.plot(days, forecast, label='Forecast', color='green', marker='o')
    plt.title(title, fontsize=14)
    plt.xlabel('Days')
    plt.ylabel('Forecasted Value')
    plt.grid(True)
    plt.legend()
    plt.show()


# Contoh penggunaan forecast 14 hari setelah pelatihan model
# Pastikan scaler telah di-fit ke data training sebelumnya
scaler.fit(train_data_8_months)

# Ambil data terbaru dari dataset validasi atau test set sebagai input
# last_data bisa berasal dari data validasi atau test set
last_data = test_data_8_months.values[-16:]  # Ambil 16 titik data terakhir sebagai input untuk prediksi

# Lakukan forecast 14 hari ke depan
forecast_14 = forecast_14_days(model, last_data, scaler=scaler)

# Visualisasikan prediksi 14 hari ke depan
plot_forecast_14_days(forecast_14, title='14-Day Forecast Using Trained Model')


#### 48 Hours

In [ ]:
# Step 5: Create a window-based dataset using scaled_data (24 jam)
def create_windows_8_48(data, window_size=32, forecast_horizon=224):
    X, y = [], []
    for i in range(len(data) - window_size - forecast_horizon + 1):
        X.append(data.iloc[i: i + window_size].values)
        y.append(data.iloc[i + window_size: i + window_size + forecast_horizon].values)
    return np.array(X), np.array(y)

In [ ]:
# Buat dataset berbasis window untuk setiap skenario (24 jam)
X_train_3, y_train_3 = create_windows_8_48(train_data_3_months)
X_train_6, y_train_6 = create_windows_8_48(train_data_6_months)
X_train_8, y_train_8 = create_windows_8_48(train_data_8_months)

X_val_3, y_val_3 = create_windows_8_48(val_data_3_months)
X_val_6, y_val_6 = create_windows_8_48(val_data_6_months)
X_val_8, y_val_8 = create_windows_8_48(val_data_8_months)

X_test_3, y_test_3 = create_windows_8_48(test_data_3_months)
X_test_6, y_test_6 = create_windows_8_48(test_data_6_months)
X_test_8, y_test_8 = create_windows_8_48(test_data_8_months)

# Cek hasilnya untuk melihat ukuran setiap dataset
print(f"Shape of X_train_3: {X_train_3.shape}, y_train_3: {y_train_3.shape}")
print(f"Shape of X_train_6: {X_train_6.shape}, y_train_6: {y_train_6.shape}")
print(f"Shape of X_train_8: {X_train_8.shape}, y_train_8: {y_train_8.shape}")
print(f"Shape of X_val_3: {X_val_3.shape}, y_val_3: {y_val_3.shape}")
print(f"Shape of X_val_6: {X_val_6.shape}, y_val_6: {y_val_6.shape}")
print(f"Shape of X_val_8: {X_val_8.shape}, y_val_8: {y_val_8.shape}")
print(f"Shape of X_test_3: {X_test_3.shape}, y_test_3: {y_test_3.shape}")
print(f"Shape of X_test_6: {X_test_6.shape}, y_test_6: {y_test_6.shape}")
print(f"Shape of X_test_8: {X_test_8.shape}, y_test_8: {y_test_8.shape}")


In [ ]:
# Step 6: Create DataLoaders
batch_size = 64
train_loader_3 = DataLoader(TensorDataset(torch.Tensor(X_train_3), torch.Tensor(y_train_3)), batch_size=batch_size, shuffle=True)
val_loader_3 = DataLoader(TensorDataset(torch.Tensor(X_val_3), torch.Tensor(y_val_3)), batch_size=batch_size, shuffle=False)
test_loader_3 = DataLoader(TensorDataset(torch.Tensor(X_test_3), torch.Tensor(y_test_3)), batch_size=batch_size, shuffle=False)

train_loader_6 = DataLoader(TensorDataset(torch.Tensor(X_train_6), torch.Tensor(y_train_6)), batch_size=batch_size, shuffle=True)
val_loader_6 = DataLoader(TensorDataset(torch.Tensor(X_val_6), torch.Tensor(y_val_6)), batch_size=batch_size, shuffle=False)
test_loader_6 = DataLoader(TensorDataset(torch.Tensor(X_test_6), torch.Tensor(y_test_6)), batch_size=batch_size, shuffle=False)

train_loader_8 = DataLoader(TensorDataset(torch.Tensor(X_train_8), torch.Tensor(y_train_8)), batch_size=batch_size, shuffle=True)
val_loader_8 = DataLoader(TensorDataset(torch.Tensor(X_val_8), torch.Tensor(y_val_8)), batch_size=batch_size, shuffle=False)
test_loader_8 = DataLoader(TensorDataset(torch.Tensor(X_test_8), torch.Tensor(y_test_8)), batch_size=batch_size, shuffle=False)

In [ ]:

import matplotlib.pyplot as plt

def train_and_evaluate_auto8_48(params, save_best_only=True, checkpoint_filename="autoformer8m48.pth.tar", patience=5, scaler=None):
    checkpoint_dir = "./checkpoints"
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)

    checkpoint_filepath = os.path.join(checkpoint_dir, checkpoint_filename)

    # Definisikan model
    model = Autoformer(
        enc_in=1,
        dec_in=1,
        c_out=1,
        seq_len=32,
        label_len=15,
        out_len=224,  # Sesuaikan dengan panjang output yang diharapkan
        d_model=params['d_model'],
        n_heads=params['n_heads'],
        e_layers=params['e_layers'],
        d_layers=params['d_layers'],
        d_ff=params['d_ff'],
        moving_avg=params['moving_avg'],
        dropout=params['dropout'],
        factor=params['factor'],
        activation=params['activation']
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])
    criterion = nn.MSELoss()  # Loss function
    
    best_val_loss = float('inf')  # Untuk menyimpan loss terbaik
    best_rmse = float('inf')  # Untuk menyimpan RMSE terbaik
    best_mape = float('inf')  # Untuk menyimpan MAPE terbaik
    num_epochs = 50  # Jumlah maksimum epoch
    epochs_no_improve = 0  # Untuk melacak jumlah epoch tanpa perbaikan
    early_stop = False  # Status untuk early stopping

    # Inisialisasi list untuk menyimpan train dan validation loss
    train_losses = []
    val_losses = []
    
    for epoch in range(num_epochs):
        if early_stop:
            print("Early stopping")
            break
            
        model.train()
        train_loss = 0.0
        for batch_X, batch_y in train_loader_8:
            optimizer.zero_grad()
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            x_enc = batch_X
            x_dec = batch_X[:, -15:, :]
            output = model(x_enc, x_dec)

            # Sesuaikan batch_y agar memiliki ukuran yang sama dengan output model
            batch_y = batch_y[:, :output.shape[1]]  # Potong atau sesuaikan batch_y

            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader_8)
        train_losses.append(train_loss)  # Simpan train loss

        # Evaluasi pada set validasi
        model.eval()
        val_loss = 0.0
        val_true, val_pred = [], []
        with torch.no_grad():
            for batch_X, batch_y in val_loader_8:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                x_enc = batch_X
                x_dec = batch_X[:, -15:, :]
                output = model(x_enc, x_dec)

                # Sesuaikan batch_y di set validasi
                batch_y = batch_y[:, :output.shape[1]]  # Sesuaikan ukuran batch_y

                val_true.append(batch_y.cpu().numpy())
                val_pred.append(output.cpu().numpy())
                loss = criterion(output, batch_y)
                val_loss += loss.item()

        val_loss /= len(val_loader_8)
        val_losses.append(val_loss)  # Simpan val loss

        # Inverse scaling untuk metrik jika scaler diberikan
        val_true = np.concatenate(val_true, axis=0).reshape(-1, 1)  # Sesuaikan bentuk data
        val_pred = np.concatenate(val_pred, axis=0).reshape(-1, 1)
        
        if scaler is not None:
            val_true = scaler.inverse_transform(val_true)  # Mengembalikan ke skala asli
            val_pred = scaler.inverse_transform(val_pred)

        # Hitung RMSE, MAPE
        rmse, mape = calculate_metrics(val_true, val_pred)
        
        print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.4f}")

        # Cek apakah ada perbaikan dalam validation loss tanpa min_delta
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse
            best_mape = mape
            epochs_no_improve = 0
            if save_best_only:
                save_checkpoint(model, optimizer, epoch, val_loss, filename=checkpoint_filepath)
                print(f"Saving checkpoint at epoch {epoch} with validation loss {val_loss:.4f}")
        else:
            epochs_no_improve += 1  # Tidak ada perbaikan, tambahkan hitungan
        
        # Cek apakah harus early stop
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            early_stop = True
    
    # Visualisasi train dan val loss setelah selesai pelatihan
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Train Loss and Validation Loss Over Epochs')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Kembalikan best_val_loss, rmse, dan mape
    return best_val_loss, best_rmse, best_mape


In [ ]:
import random

device = 'mps' if torch.cuda.is_available() else 'cpu'
def random_search_with_checkpoint8auto48(param_grid, num_iter=10):
    best_val_loss = float('inf')  # Untuk melacak loss validasi terbaik
    best_rmse = float('inf')  # Untuk melacak RMSE terbaik
    best_mape = float('inf')  # Untuk melacak MAPE terbaik
    best_params = None

    # Daftar hyperparameter dan pilih secara acak
    keys = list(param_grid.keys())
    evaluated_params = set()  # Untuk melacak kombinasi parameter yang dievaluasi

    for i in range(num_iter):  # Menjalankan pencarian acak sebanyak num_iter kali
        # Pilih hyperparameter secara acak
        params = {key: random.choice(param_grid[key]) for key in keys}

        # Buat representasi parameter yang dapat di-hash
        params_tuple = tuple(sorted(params.items()))
        if params_tuple in evaluated_params:
            continue  # Lewati parameter yang sudah dievaluasi

        evaluated_params.add(params_tuple)  # Tambahkan ke set yang dievaluasi
        print(f"Evaluating with params: {params}")

        # Jalankan pelatihan untuk kombinasi parameter dan dapatkan val_loss, rmse, mape
        val_loss, rmse, mape = train_and_evaluate_auto8_48(params, checkpoint_filename="autoformer8m48.pth.tar")

        # Simpan parameter terbaik jika val_loss lebih baik
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse  # Simpan RMSE terbaik
            best_mape = mape  # Simpan MAPE terbaik
            best_params = params

    # Cetak hasil terbaik setelah pencarian hyperparameter selesai
    print(f"\nBest Validation Loss: {best_val_loss:.4f}")
    print(f"Best RMSE: {best_rmse:.4f}")
    print(f"Best MAPE: {best_mape:.4f}")
    print(f"Best Hyperparameters: {best_params}")

    return best_params


In [ ]:
best_params = {
    'd_model': [256],          # Dimensi model
    'n_heads': [4],            # Jumlah attention heads
    'e_layers': [3],           # Jumlah layer di encoder
    'd_layers': [2],           # Jumlah layer di decoder
    'd_ff': [512],             # Dimensi feed-forward (disarankan 2x atau 4x dari d_model)
    'moving_avg': [25],        # Ukuran rata-rata bergerak (bisa disesuaikan)
    'dropout': [0.3],          # Dropout rate
    'factor': [1],             # Faktor autocorrelation (1 adalah default yang umum)
    'activation': ['relu'],      # Aktivasi, bisa 'relu' atau 'gelu'
    'learning_rate': [0.001]   # Laju pembelajaran
}

# Jalankan Grid Search dengan checkpoint untuk model 3 bulan
best_params = random_search_with_checkpoint8auto48(best_params)


In [ ]:
best_params = {
    'd_model': [256],          # Dimensi model
    'n_heads': [4],            # Jumlah attention heads
    'e_layers': [3],           # Jumlah layer di encoder
    'd_layers': [2],           # Jumlah layer di decoder
    'd_ff': [512],             # Dimensi feed-forward (disarankan 2x atau 4x dari d_model)
    'moving_avg': [25],        # Ukuran rata-rata bergerak (bisa disesuaikan)
    'dropout': [0.3],          # Dropout rate
    'factor': [1],             # Faktor autocorrelation (1 adalah default yang umum)
    'activation': ['relu'],      # Aktivasi, bisa 'relu' atau 'gelu'
    'learning_rate': [0.001]   # Laju pembelajaran
}

# Jalankan Grid Search dengan checkpoint untuk model 3 bulan
best_params = random_search_with_checkpoint8auto48(best_params)


In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Define your model (Autoformer) and optimizer
# Adjust the parameters according to your model architecture
model = Autoformer(
    enc_in=1,
    dec_in=1,
    c_out=1,
    seq_len=32,
    label_len=15,
    out_len=224,  # Expected output length
    d_model=256,  # Example configuration
    n_heads=4,
    e_layers=3,
    d_layers=2,
    d_ff=512,
    moving_avg=25,
    dropout=0.3,
    factor=1,
    activation='relu'
).to(device)

# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Step 1: Load model checkpoint
checkpoint_path = './checkpoints/autoformer8m48.pth.tar'  # Path to your checkpoint file

def load_checkpoint(model, optimizer, checkpoint_path):
    if checkpoint_path is not None:
        print(f"Loading checkpoint from '{checkpoint_path}'...")
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        epoch = checkpoint.get('epoch', None)
        best_val_loss = checkpoint.get('best_val_loss', None)  # Handle missing best_val_loss
        print(f"Checkpoint loaded. Resuming from epoch {epoch}")
        return epoch, best_val_loss
    else:
        print("No checkpoint provided, starting from scratch.")
        return None, None

# Load the checkpoint
epoch, best_val_loss = load_checkpoint(model, optimizer, checkpoint_path)

# Step 2: Generate predictions using the test data
model.eval()  # Switch the model to evaluation mode

predictions = []
with torch.no_grad():
    for batch_X, batch_y in test_loader_8:  # test_loader_3 is your DataLoader for the test set
        batch_X = batch_X.to(device)  # Move data to the same device as the model (GPU/CPU)
        x_enc = batch_X  # Encoder input
        x_dec = batch_X[:, -15:, :]  # Decoder input (using the last 15 time steps)
        
        # Get model predictions
        pred = model(x_enc, x_dec)
        
        # Take only the last time step prediction for each batch
        predictions.append(pred[:, -1, :].cpu().numpy())

# Step 3: Concatenate predictions into a single array
predictions = np.concatenate(predictions, axis=0)

# Step 4: Ensure predictions match the size of the test set (if necessary)
pred_len = len(predictions)
test_data_sliced = test_data_8_months.iloc[:pred_len]  # Slicing test data to match predictions length

# Step 5: Convert predictions to DataFrame for plotting
predictions_df = pd.DataFrame(predictions, index=test_data_sliced.index, columns=['predictions'])

# Step 6: Filter the data to start from December
start_date = '2023-12-01'  # Specify the start date for the plot

train_data_filtered = train_data_8_months[train_data_8_months.index >= start_date]
val_data_filtered = val_data_8_months[val_data_8_months.index >= start_date]
test_data_filtered = test_data_sliced[test_data_sliced.index >= start_date]
predictions_filtered = predictions_df[predictions_df.index >= start_date]

# Step 7: Calculate RMSE, MAPE, and residuals
actual_values = test_data_filtered['Hs'].values  # Actual test data values
predicted_values = predictions_filtered['predictions'].values  # Predicted values

# Calculate RMSE and MAPE
rmse = mean_squared_error(actual_values, predicted_values, squared=False)
mape = mean_absolute_percentage_error(actual_values, predicted_values)

# Calculate residuals and standard deviation for confidence intervals
residuals = actual_values - predicted_values
std_dev = np.std(residuals)
conf_interval = 1.96 * std_dev  # 95% confidence interval

# Print RMSE and MAPE
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape * 100:.2f}%")

# Step 8: Plot predictions along with train, validation, test data, and confidence intervals
fig, ax = plt.subplots(figsize=(10, 5))

# Plot train data with a blue solid line
train_data_filtered.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=2)

# Plot validation data with a purple dashed line
val_data_filtered.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=2)

# Plot actual test data with an orange dotted line
test_data_filtered.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=2)

# Plot model predictions with a green solid line
predictions_filtered.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Plot confidence intervals
upper_bound = predictions_filtered['predictions'] + conf_interval
lower_bound = predictions_filtered['predictions'] - conf_interval
ax.fill_between(predictions_filtered.index, lower_bound, upper_bound, color='green', alpha=0.3, label="95% Confidence Interval")

# Add labels and legend
ax.set_xlabel('Datetime')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Train, Validation, Test, and Predictions from December Onwards\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()

# Show the plot
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Assuming predictions, actual test data, and confidence interval calculations are already in place

# Step 1: Define the zoom-in start date to focus on the test data and predictions
zoom_start_date = '2023-12-05'  # Adjust this date to focus on the test and prediction period

# Step 2: Filter data to zoom-in on the specified range
train_data_zoomed = train_data_8_months[train_data_8_months.index >= zoom_start_date]
val_data_zoomed = val_data_8_months[val_data_8_months.index >= zoom_start_date]
test_data_zoomed = test_data_sliced[test_data_sliced.index >= zoom_start_date]
predictions_zoomed = predictions_df[predictions_df.index >= zoom_start_date]

# Step 3: Plot zoomed-in view
fig, ax = plt.subplots(figsize=(12, 6))

# Plot zoomed train data
train_data_zoomed.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=1.5)

# Plot zoomed validation data
val_data_zoomed.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=1.5)

# Plot zoomed test data
test_data_zoomed.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=1.5)

# Plot zoomed model predictions
predictions_zoomed.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Plot confidence intervals for the predictions
upper_bound_zoomed = predictions_zoomed['predictions'] + conf_interval
lower_bound_zoomed = predictions_zoomed['predictions'] - conf_interval
ax.fill_between(predictions_zoomed.index, lower_bound_zoomed, upper_bound_zoomed, color='green', alpha=0.2, label="95% Confidence Interval")

# Step 4: Add labels, title, and legend
ax.set_xlabel('Date')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Zoomed-in View of Test Data and Predictions with Confidence Interval\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()

# Show grid and plot
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import torch

def forecast_14_days(model, last_data, scaler=None):
    """
    Fungsi untuk memprediksi 14 hari ke depan menggunakan model yang telah dilatih.
    
    :param model: Model yang telah dilatih.
    :param last_data: Data terakhir yang akan digunakan sebagai input untuk prediksi (sequential input).
    :param scaler: Scaler yang digunakan pada data, untuk mengembalikan hasil prediksi ke skala asli jika perlu.
    """
    model.eval()  # Set model ke mode evaluasi
    
    # Prediksi 14 hari ke depan
    with torch.no_grad():
        # Ubah data terakhir menjadi tensor dan pindahkan ke device
        last_data_tensor = torch.Tensor(last_data).unsqueeze(0).to(device)  # Tambahkan batch dimension
        
        # Gunakan model untuk melakukan prediksi 14 langkah ke depan
        x_enc = last_data_tensor  # Input terakhir untuk encoder
        x_dec = last_data_tensor[:, -15:, :]  # Ambil 15 langkah terakhir sebagai input untuk decoder
        
        forecast = model(x_enc, x_dec)
    
    # Hasil prediksi
    forecast = forecast.cpu().numpy().reshape(-1, 1)  # Ubah bentuk menjadi array (14, 1)
    
    # Slice the first 14 predictions if forecast length is longer
    forecast = forecast[:14]  # Ambil hanya 14 hari prediksi pertama
    
    # Inverse scaling jika diperlukan
    if scaler is not None:
        forecast = scaler.inverse_transform(forecast)
    
    return forecast


def plot_forecast_14_days(forecast, title='14-Day Forecast'):
    """
    Fungsi untuk memplot prediksi 14 hari ke depan.
    
    :param forecast: Array hasil prediksi (14 hari ke depan).
    :param title: Judul plot.
    """
    days = list(range(1, 15))  # Hari 1 hingga 14
    
    plt.figure(figsize=(10, 6))
    plt.plot(days, forecast, label='Forecast', color='green', marker='o')
    plt.title(title, fontsize=14)
    plt.xlabel('Days')
    plt.ylabel('Forecasted Value')
    plt.grid(True)
    plt.legend()
    plt.show()


# Contoh penggunaan forecast 14 hari setelah pelatihan model
# Pastikan scaler telah di-fit ke data training sebelumnya
scaler.fit(train_data_8_months)

# Ambil data terbaru dari dataset validasi atau test set sebagai input
# last_data bisa berasal dari data validasi atau test set
last_data = test_data_8_months.values[-16:]  # Ambil 16 titik data terakhir sebagai input untuk prediksi

# Lakukan forecast 14 hari ke depan
forecast_14 = forecast_14_days(model, last_data, scaler=scaler)

# Visualisasikan prediksi 14 hari ke depan
plot_forecast_14_days(forecast_14, title='14-Day Forecast Using Trained Model')


#### Train data 8 Months - 96 Hours

In [ ]:
# Step 5: Create a window-based dataset using scaled_data (24 jam)
def create_windows_8_96(data, window_size=64, forecast_horizon=224):
    X, y = [], []
    for i in range(len(data) - window_size - forecast_horizon + 1):
        X.append(data.iloc[i: i + window_size].values)
        y.append(data.iloc[i + window_size: i + window_size + forecast_horizon].values)
    return np.array(X), np.array(y)

In [ ]:
# Buat dataset berbasis window untuk setiap skenario (24 jam)
X_train_3, y_train_3 = create_windows_8_96(train_data_3_months)
X_train_6, y_train_6 = create_windows_8_96(train_data_6_months)
X_train_8, y_train_8 = create_windows_8_96(train_data_8_months)

X_val_3, y_val_3 = create_windows_8_96(val_data_3_months)
X_val_6, y_val_6 = create_windows_8_96(val_data_6_months)
X_val_8, y_val_8 = create_windows_8_96(val_data_8_months)

X_test_3, y_test_3 = create_windows_8_96(test_data_3_months)
X_test_6, y_test_6 = create_windows_8_96(test_data_6_months)
X_test_8, y_test_8 = create_windows_8_96(test_data_8_months)

# Cek hasilnya untuk melihat ukuran setiap dataset
print(f"Shape of X_train_3: {X_train_3.shape}, y_train_3: {y_train_3.shape}")
print(f"Shape of X_train_6: {X_train_6.shape}, y_train_6: {y_train_6.shape}")
print(f"Shape of X_train_8: {X_train_8.shape}, y_train_8: {y_train_8.shape}")
print(f"Shape of X_val_3: {X_val_3.shape}, y_val_3: {y_val_3.shape}")
print(f"Shape of X_val_6: {X_val_6.shape}, y_val_6: {y_val_6.shape}")
print(f"Shape of X_val_8: {X_val_8.shape}, y_val_8: {y_val_8.shape}")
print(f"Shape of X_test_3: {X_test_3.shape}, y_test_3: {y_test_3.shape}")
print(f"Shape of X_test_6: {X_test_6.shape}, y_test_6: {y_test_6.shape}")
print(f"Shape of X_test_8: {X_test_8.shape}, y_test_8: {y_test_8.shape}")


In [ ]:
import matplotlib.pyplot as plt

def train_and_evaluate_auto8_96(params, save_best_only=True, checkpoint_filename="autoformer8m96.pth.tar", patience=5, scaler=None):
    checkpoint_dir = "./checkpoints"
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)

    checkpoint_filepath = os.path.join(checkpoint_dir, checkpoint_filename)

    # Definisikan model
    model = Autoformer(
        enc_in=1,
        dec_in=1,
        c_out=1,
        seq_len=64,
        label_len=15,
        out_len=224,  # Sesuaikan dengan panjang output yang diharapkan
        d_model=params['d_model'],
        n_heads=params['n_heads'],
        e_layers=params['e_layers'],
        d_layers=params['d_layers'],
        d_ff=params['d_ff'],
        moving_avg=params['moving_avg'],
        dropout=params['dropout'],
        factor=params['factor'],
        activation=params['activation']
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])
    criterion = nn.MSELoss()  # Loss function
    
    best_val_loss = float('inf')  # Untuk menyimpan loss terbaik
    best_rmse = float('inf')  # Untuk menyimpan RMSE terbaik
    best_mape = float('inf')  # Untuk menyimpan MAPE terbaik
    num_epochs = 50  # Jumlah maksimum epoch
    epochs_no_improve = 0  # Untuk melacak jumlah epoch tanpa perbaikan
    early_stop = False  # Status untuk early stopping

    # Inisialisasi list untuk menyimpan train dan validation loss
    train_losses = []
    val_losses = []
    
    for epoch in range(num_epochs):
        if early_stop:
            print("Early stopping")
            break
            
        model.train()
        train_loss = 0.0
        for batch_X, batch_y in train_loader_8:
            optimizer.zero_grad()
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            x_enc = batch_X
            x_dec = batch_X[:, -15:, :]
            output = model(x_enc, x_dec)

            # Sesuaikan batch_y agar memiliki ukuran yang sama dengan output model
            batch_y = batch_y[:, :output.shape[1]]  # Potong atau sesuaikan batch_y

            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader_8)
        train_losses.append(train_loss)  # Simpan train loss

        # Evaluasi pada set validasi
        model.eval()
        val_loss = 0.0
        val_true, val_pred = [], []
        with torch.no_grad():
            for batch_X, batch_y in val_loader_8:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                x_enc = batch_X
                x_dec = batch_X[:, -15:, :]
                output = model(x_enc, x_dec)

                # Sesuaikan batch_y di set validasi
                batch_y = batch_y[:, :output.shape[1]]  # Sesuaikan ukuran batch_y

                val_true.append(batch_y.cpu().numpy())
                val_pred.append(output.cpu().numpy())
                loss = criterion(output, batch_y)
                val_loss += loss.item()

        val_loss /= len(val_loader_8)
        val_losses.append(val_loss)  # Simpan val loss

        # Inverse scaling untuk metrik jika scaler diberikan
        val_true = np.concatenate(val_true, axis=0).reshape(-1, 1)  # Sesuaikan bentuk data
        val_pred = np.concatenate(val_pred, axis=0).reshape(-1, 1)
        
        if scaler is not None:
            val_true = scaler.inverse_transform(val_true)  # Mengembalikan ke skala asli
            val_pred = scaler.inverse_transform(val_pred)

        # Hitung RMSE, MAPE
        rmse, mape = calculate_metrics(val_true, val_pred)
        
        print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.4f}")

        # Cek apakah ada perbaikan dalam validation loss tanpa min_delta
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse
            best_mape = mape
            epochs_no_improve = 0
            if save_best_only:
                save_checkpoint(model, optimizer, epoch, val_loss, filename=checkpoint_filepath)
                print(f"Saving checkpoint at epoch {epoch} with validation loss {val_loss:.4f}")
        else:
            epochs_no_improve += 1  # Tidak ada perbaikan, tambahkan hitungan
        
        # Cek apakah harus early stop
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            early_stop = True
    
    # Visualisasi train dan val loss setelah selesai pelatihan
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Train Loss and Validation Loss Over Epochs')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Kembalikan best_val_loss, rmse, dan mape
    return best_val_loss, best_rmse, best_mape


In [ ]:
import random

device = 'mps' if torch.cuda.is_available() else 'cpu'
def random_search_with_checkpoint8auto96(param_grid, num_iter=10):
    best_val_loss = float('inf')  # Untuk melacak loss validasi terbaik
    best_rmse = float('inf')  # Untuk melacak RMSE terbaik
    best_mape = float('inf')  # Untuk melacak MAPE terbaik
    best_params = None

    # Daftar hyperparameter dan pilih secara acak
    keys = list(param_grid.keys())
    evaluated_params = set()  # Untuk melacak kombinasi parameter yang dievaluasi

    for i in range(num_iter):  # Menjalankan pencarian acak sebanyak num_iter kali
        # Pilih hyperparameter secara acak
        params = {key: random.choice(param_grid[key]) for key in keys}

        # Buat representasi parameter yang dapat di-hash
        params_tuple = tuple(sorted(params.items()))
        if params_tuple in evaluated_params:
            continue  # Lewati parameter yang sudah dievaluasi

        evaluated_params.add(params_tuple)  # Tambahkan ke set yang dievaluasi
        print(f"Evaluating with params: {params}")

        # Jalankan pelatihan untuk kombinasi parameter dan dapatkan val_loss, rmse, mape
        val_loss, rmse, mape = train_and_evaluate_auto8_96(params, checkpoint_filename="autoformer8m96.pth.tar")

        # Simpan parameter terbaik jika val_loss lebih baik
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse  # Simpan RMSE terbaik
            best_mape = mape  # Simpan MAPE terbaik
            best_params = params

    # Cetak hasil terbaik setelah pencarian hyperparameter selesai
    print(f"\nBest Validation Loss: {best_val_loss:.4f}")
    print(f"Best RMSE: {best_rmse:.4f}")
    print(f"Best MAPE: {best_mape:.4f}")
    print(f"Best Hyperparameters: {best_params}")

    return best_params


In [ ]:
best_params = {
    'd_model': [256],          # Dimensi model
    'n_heads': [4],            # Jumlah attention heads
    'e_layers': [3],           # Jumlah layer di encoder
    'd_layers': [2],           # Jumlah layer di decoder
    'd_ff': [512],             # Dimensi feed-forward (disarankan 2x atau 4x dari d_model)
    'moving_avg': [25],        # Ukuran rata-rata bergerak (bisa disesuaikan)
    'dropout': [0.3],          # Dropout rate
    'factor': [1],             # Faktor autocorrelation (1 adalah default yang umum)
    'activation': ['relu'],      # Aktivasi, bisa 'relu' atau 'gelu'
    'learning_rate': [0.001]   # Laju pembelajaran
}

# Jalankan Grid Search dengan checkpoint untuk model 3 bulan
best_params = random_search_with_checkpoint8auto96(best_params)

In [ ]:
best_params = {
    'd_model': [256],          # Dimensi model
    'n_heads': [4],            # Jumlah attention heads
    'e_layers': [3],           # Jumlah layer di encoder
    'd_layers': [2],           # Jumlah layer di decoder
    'd_ff': [512],             # Dimensi feed-forward (disarankan 2x atau 4x dari d_model)
    'moving_avg': [25],        # Ukuran rata-rata bergerak (bisa disesuaikan)
    'dropout': [0.3],          # Dropout rate
    'factor': [1],             # Faktor autocorrelation (1 adalah default yang umum)
    'activation': ['relu'],      # Aktivasi, bisa 'relu' atau 'gelu'
    'learning_rate': [0.001]   # Laju pembelajaran
}

# Jalankan Grid Search dengan checkpoint untuk model 3 bulan
best_params = random_search_with_checkpoint8auto96(best_params)

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Define your model (Autoformer) and optimizer
# Adjust the parameters according to your model architecture
model = Autoformer(
    enc_in=1,
    dec_in=1,
    c_out=1,
    seq_len=64,
    label_len=15,
    out_len=224,  # Expected output length
    d_model=256,  # Example configuration
    n_heads=4,
    e_layers=3,
    d_layers=2,
    d_ff=512,
    moving_avg=25,
    dropout=0.3,
    factor=1,
    activation='relu'
).to(device)

# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Step 1: Load model checkpoint
checkpoint_path = './checkpoints/autoformer8m96.pth.tar'  # Path to your checkpoint file

def load_checkpoint(model, optimizer, checkpoint_path):
    if checkpoint_path is not None:
        print(f"Loading checkpoint from '{checkpoint_path}'...")
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        epoch = checkpoint.get('epoch', None)
        best_val_loss = checkpoint.get('best_val_loss', None)  # Handle missing best_val_loss
        print(f"Checkpoint loaded. Resuming from epoch {epoch}")
        return epoch, best_val_loss
    else:
        print("No checkpoint provided, starting from scratch.")
        return None, None

# Load the checkpoint
epoch, best_val_loss = load_checkpoint(model, optimizer, checkpoint_path)

# Step 2: Generate predictions using the test data
model.eval()  # Switch the model to evaluation mode

predictions = []
with torch.no_grad():
    for batch_X, batch_y in test_loader_8:  # test_loader_3 is your DataLoader for the test set
        batch_X = batch_X.to(device)  # Move data to the same device as the model (GPU/CPU)
        x_enc = batch_X  # Encoder input
        x_dec = batch_X[:, -15:, :]  # Decoder input (using the last 15 time steps)
        
        # Get model predictions
        pred = model(x_enc, x_dec)
        
        # Take only the last time step prediction for each batch
        predictions.append(pred[:, -1, :].cpu().numpy())

# Step 3: Concatenate predictions into a single array
predictions = np.concatenate(predictions, axis=0)

# Step 4: Ensure predictions match the size of the test set (if necessary)
pred_len = len(predictions)
test_data_sliced = test_data_8_months.iloc[:pred_len]  # Slicing test data to match predictions length

# Step 5: Convert predictions to DataFrame for plotting
predictions_df = pd.DataFrame(predictions, index=test_data_sliced.index, columns=['predictions'])

# Step 6: Filter the data to start from December
start_date = '2023-12-01'  # Specify the start date for the plot

train_data_filtered = train_data_8_months[train_data_8_months.index >= start_date]
val_data_filtered = val_data_8_months[val_data_8_months.index >= start_date]
test_data_filtered = test_data_sliced[test_data_sliced.index >= start_date]
predictions_filtered = predictions_df[predictions_df.index >= start_date]

# Step 7: Calculate RMSE, MAPE, and residuals
actual_values = test_data_filtered['Hs'].values  # Actual test data values
predicted_values = predictions_filtered['predictions'].values  # Predicted values

# Calculate RMSE and MAPE
rmse = mean_squared_error(actual_values, predicted_values, squared=False)
mape = mean_absolute_percentage_error(actual_values, predicted_values)

# Calculate residuals and standard deviation for confidence intervals
residuals = actual_values - predicted_values
std_dev = np.std(residuals)
conf_interval = 1.96 * std_dev  # 95% confidence interval

# Print RMSE and MAPE
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape * 100:.2f}%")

# Step 8: Plot predictions along with train, validation, test data, and confidence intervals
fig, ax = plt.subplots(figsize=(10, 5))

# Plot train data with a blue solid line
train_data_filtered.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=2)

# Plot validation data with a purple dashed line
val_data_filtered.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=2)

# Plot actual test data with an orange dotted line
test_data_filtered.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=2)

# Plot model predictions with a green solid line
predictions_filtered.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Plot confidence intervals
upper_bound = predictions_filtered['predictions'] + conf_interval
lower_bound = predictions_filtered['predictions'] - conf_interval
ax.fill_between(predictions_filtered.index, lower_bound, upper_bound, color='green', alpha=0.3, label="95% Confidence Interval")

# Add labels and legend
ax.set_xlabel('Datetime')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Train, Validation, Test, and Predictions from December Onwards\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()

# Show the plot
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Assuming predictions, actual test data, and confidence interval calculations are already in place

# Step 1: Define the zoom-in start date to focus on the test data and predictions
zoom_start_date = '2023-12-05'  # Adjust this date to focus on the test and prediction period

# Step 2: Filter data to zoom-in on the specified range
train_data_zoomed = train_data_8_months[train_data_8_months.index >= zoom_start_date]
val_data_zoomed = val_data_8_months[val_data_8_months.index >= zoom_start_date]
test_data_zoomed = test_data_sliced[test_data_sliced.index >= zoom_start_date]
predictions_zoomed = predictions_df[predictions_df.index >= zoom_start_date]

# Step 3: Plot zoomed-in view
fig, ax = plt.subplots(figsize=(12, 6))

# Plot zoomed train data
train_data_zoomed.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=1.5)

# Plot zoomed validation data
val_data_zoomed.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=1.5)

# Plot zoomed test data
test_data_zoomed.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=1.5)

# Plot zoomed model predictions
predictions_zoomed.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Plot confidence intervals for the predictions
upper_bound_zoomed = predictions_zoomed['predictions'] + conf_interval
lower_bound_zoomed = predictions_zoomed['predictions'] - conf_interval
ax.fill_between(predictions_zoomed.index, lower_bound_zoomed, upper_bound_zoomed, color='green', alpha=0.2, label="95% Confidence Interval")

# Step 4: Add labels, title, and legend
ax.set_xlabel('Date')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Zoomed-in View of Test Data and Predictions with Confidence Interval\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()

# Show grid and plot
plt.grid(True)
plt.show()

### TRANSFORMER

In [ ]:
import math
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
import math

#### Arsitektur Transformer

In [ ]:
# Positionwise Feed-Forward Networks
class PositionWiseFFN(nn.Module):
    """The positionwise feed-forward network."""
    def __init__(self, ffn_num_hiddens, ffn_num_outputs):
        super().__init__()
        self.dense1 = nn.Linear(ffn_num_outputs, ffn_num_hiddens)
        self.relu = nn.ReLU()
        self.dense2 = nn.Linear(ffn_num_hiddens, ffn_num_outputs)

    def forward(self, X):
        return self.dense2(self.relu(self.dense1(X)))

# Residual Connection and Layer normalization
class AddNorm(nn.Module):
    """The residual connection followed by layer normalization."""
    def __init__(self, norm_shape, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.ln = nn.LayerNorm(norm_shape)

    def forward(self, X, Y):
        return self.ln(self.dropout(Y) + X)

# Scaled Dot-Product Attention with Masking
class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=None):
        super().__init__()
        self.dropout = nn.Dropout(dropout) if dropout else None
    
    def forward(self, queries, keys, values, valid_lens=None):
        d_k = queries.size(-1)
        scores = torch.matmul(queries, keys.transpose(-2, -1)) / math.sqrt(d_k)
        
        if valid_lens is not None:
            # Apply attention mask
            mask = torch.arange(scores.size(-1), device=valid_lens.device)[None, :] >= valid_lens[:, None]
            scores = scores.masked_fill(mask.unsqueeze(1), float('-inf'))
        
        attention_weights = F.softmax(scores, dim=-1)
        if self.dropout:
            attention_weights = self.dropout(attention_weights)
        return torch.matmul(attention_weights, values)

# Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, num_hiddens, num_heads, dropout, bias=False):
        super().__init__()
        self.num_heads = num_heads
        self.attention = ScaledDotProductAttention(dropout)
        self.W_q = nn.Linear(num_hiddens, num_hiddens, bias=bias)
        self.W_k = nn.Linear(num_hiddens, num_hiddens, bias=bias)
        self.W_v = nn.Linear(num_hiddens, num_hiddens, bias=bias)
        self.W_o = nn.Linear(num_hiddens, num_hiddens, bias=bias)

    def forward(self, queries, keys, values, valid_lens):
        queries = self.transpose_qkv(self.W_q(queries))
        keys = self.transpose_qkv(self.W_k(keys))
        values = self.transpose_qkv(self.W_v(values))

        if valid_lens is not None:
            valid_lens = torch.repeat_interleave(valid_lens, repeats=self.num_heads, dim=0)

        output = self.attention(queries, keys, values, valid_lens)
        output_concat = self.transpose_output(output)
        return self.W_o(output_concat)

    def transpose_qkv(self, X):
        # Shape input: (batch_size, num_queries, num_hiddens)
        X = X.reshape(X.shape[0], X.shape[1], self.num_heads, -1)
        return X.permute(0, 2, 1, 3)  # Output shape: (batch_size, num_heads, num_queries, num_hiddens // num_heads)

    def transpose_output(self, X):
        X = X.permute(0, 2, 1, 3).reshape(X.shape[0], X.shape[2], -1)
        return X

# Transformer Encoder Block
class TransformerEncoderBlock(nn.Module):
    """The Transformer encoder block."""
    def __init__(self, num_hiddens, ffn_num_hiddens, num_heads, dropout):
        super().__init__()
        self.attention = MultiHeadAttention(num_hiddens, num_heads, dropout)
        self.addnorm1 = AddNorm(num_hiddens, dropout)
        self.ffn = PositionWiseFFN(ffn_num_hiddens, num_hiddens)
        self.addnorm2 = AddNorm(num_hiddens, dropout)

    def forward(self, X, valid_lens):
        Y = self.addnorm1(X, self.attention(X, X, X, valid_lens))
        return self.addnorm2(Y, self.ffn(Y))

# Positional Encoding with debugging for dimensions
# Positional Encoding without debug print statements
class PositionalEncoding(nn.Module):
    """Positional encoding as described in the Transformer paper."""
    def __init__(self, num_hiddens, dropout, max_len=1000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.P = torch.zeros((1, max_len, num_hiddens))
        X = torch.arange(max_len, dtype=torch.float32).reshape(-1, 1) / torch.pow(
            10000, torch.arange(0, num_hiddens, 2, dtype=torch.float32) / num_hiddens)
        self.P[:, :, 0::2] = torch.sin(X)
        self.P[:, :, 1::2] = torch.cos(X)

    def forward(self, X):
        # Add positional encoding to the input without printing shapes
        X = X + self.P[:, :X.size(1), :].to(X.device)  # Ensure this matches input length
        return self.dropout(X)


# Transformer Encoder
class TransformerEncoder(nn.Module):
    """The Transformer encoder."""
    def __init__(self, vocab_size, num_hiddens, ffn_num_hiddens, num_heads, num_blks, dropout):
        super().__init__()
        self.num_hiddens = num_hiddens
        self.embedding = nn.Embedding(vocab_size, num_hiddens)
        self.pos_encoding = PositionalEncoding(num_hiddens, dropout)
        self.blks = nn.Sequential()
        for i in range(num_blks):
            self.blks.add_module(f"block{i}", TransformerEncoderBlock(num_hiddens, ffn_num_hiddens, num_heads, dropout))

    def forward(self, X, valid_lens):
        X = self.pos_encoding(self.embedding(X) * math.sqrt(self.num_hiddens))
        for blk in self.blks:
            X = blk(X, valid_lens)
        return X

# Transformer Decoder Block
class TransformerDecoderBlock(nn.Module):
    def __init__(self, num_hiddens, ffn_num_hiddens, num_heads, dropout):
        super().__init__()
        self.attention1 = MultiHeadAttention(num_hiddens, num_heads, dropout)
        self.addnorm1 = AddNorm(num_hiddens, dropout)
        self.attention2 = MultiHeadAttention(num_hiddens, num_heads, dropout)
        self.addnorm2 = AddNorm(num_hiddens, dropout)
        self.ffn = PositionWiseFFN(ffn_num_hiddens, num_hiddens)
        self.addnorm3 = AddNorm(num_hiddens, dropout)

    def forward(self, X, state, enc_outputs, valid_lens):
        X = self.addnorm1(X, self.attention1(X, X, X, valid_lens))
        X = self.addnorm2(X, self.attention2(X, enc_outputs, enc_outputs, valid_lens))
        return self.addnorm3(X, self.ffn(X)), state

# Transformer Decoder
class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, num_hiddens, ffn_num_hiddens, num_heads, num_blks, dropout):
        super().__init__()
        self.num_hiddens = num_hiddens
        self.embedding = nn.Embedding(vocab_size, num_hiddens)
        self.pos_encoding = PositionalEncoding(num_hiddens, dropout)
        self.blks = nn.Sequential()
        for i in range(num_blks):
            self.blks.add_module(f"block{i}", TransformerDecoderBlock(num_hiddens, ffn_num_hiddens, num_heads, dropout))
        self.dense = nn.Linear(num_hiddens, vocab_size)

    def forward(self, X, state, enc_outputs, valid_lens):
        X = self.pos_encoding(self.embedding(X) * math.sqrt(self.num_hiddens))
        for blk in self.blks:
            X, state = blk(X, state, enc_outputs, valid_lens)
        return self.dense(X), state

In [ ]:
class TimeSeriesTransformer(nn.Module):
    def __init__(self, input_size, d_model, n_heads, n_layers, d_ff, dropout, output_size, seq_len, label_len, out_len):
        super(TimeSeriesTransformer, self).__init__()

        # Embedding layer untuk memproyeksikan input_size (jumlah fitur) ke dimensi d_model
        self.input_embedding = nn.Linear(input_size, d_model)

        # Tentukan max_len untuk positional encoding berdasarkan panjang urutan maksimum
        max_len = max(seq_len, label_len, out_len)
        self.pos_encoding = PositionalEncoding(d_model, dropout, max_len=max_len)

        # Encoder Transformer
        encoder_layers = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_ff, dropout=dropout)
        self.encoder = nn.TransformerEncoder(encoder_layers, num_layers=n_layers)

        # Decoder Transformer
        decoder_layers = nn.TransformerDecoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_ff, dropout=dropout)
        self.decoder = nn.TransformerDecoder(decoder_layers, num_layers=n_layers)

        # Fully connected layer untuk memproyeksikan d_model ke output_size (nilai yang di-forecast)
        self.fc_out = nn.Linear(d_model, output_size)

        # Simpan panjang urutan, panjang label, dan panjang keluaran
        self.seq_len = seq_len
        self.label_len = label_len
        self.out_len = out_len

    def forward(self, x_enc, x_dec):
        # Proyeksikan input ke dimensi model (d_model)
        x_enc = self.input_embedding(x_enc)  # (batch_size, seq_len, d_model)
        x_dec = self.input_embedding(x_dec)  # (batch_size, out_len, d_model)

        # Tambahkan positional encoding
        x_enc = self.pos_encoding(x_enc)
        x_dec = self.pos_encoding(x_dec)

        # Encoder
        enc_output = self.encoder(x_enc.permute(1, 0, 2))  # (seq_len, batch_size, d_model)

        # Decoder
        dec_output = self.decoder(x_dec.permute(1, 0, 2), enc_output)  # (out_len, batch_size, d_model)

        # Output layer, ambil hanya output untuk langkah-langkah waktu yang di-forecast
        return self.fc_out(dec_output.permute(1, 0, 2))  # (batch_size, out_len, output_size)


#### Train 3 Months - 24 Hours

In [ ]:
# Step 5: Create a window-based dataset using scaled_data (24 jam)
def create_windows_3_24(data, window_size=16, forecast_horizon=224):
    X, y = [], []
    for i in range(len(data) - window_size - forecast_horizon + 1):
        X.append(data.iloc[i: i + window_size].values)
        y.append(data.iloc[i + window_size: i + window_size + forecast_horizon].values)
    return np.array(X), np.array(y)

In [ ]:
# Buat dataset berbasis window untuk setiap skenario (24 jam)
X_train_3, y_train_3 = create_windows_3_24(train_data_3_months)
X_train_6, y_train_6 = create_windows_3_24(train_data_6_months)
X_train_8, y_train_8 = create_windows_3_24(train_data_8_months)

X_val_3, y_val_3 = create_windows_3_24(val_data_3_months)
X_val_6, y_val_6 = create_windows_3_24(val_data_6_months)
X_val_8, y_val_8 = create_windows_3_24(val_data_8_months)

X_test_3, y_test_3 = create_windows_3_24(test_data_3_months)
X_test_6, y_test_6 = create_windows_3_24(test_data_6_months)
X_test_8, y_test_8 = create_windows_3_24(test_data_8_months)

# Cek hasilnya untuk melihat ukuran setiap dataset
print(f"Shape of X_train_3: {X_train_3.shape}, y_train_3: {y_train_3.shape}")
print(f"Shape of X_train_6: {X_train_6.shape}, y_train_6: {y_train_6.shape}")
print(f"Shape of X_train_8: {X_train_8.shape}, y_train_8: {y_train_8.shape}")
print(f"Shape of X_val_3: {X_val_3.shape}, y_val_3: {y_val_3.shape}")
print(f"Shape of X_val_6: {X_val_6.shape}, y_val_6: {y_val_6.shape}")
print(f"Shape of X_val_8: {X_val_8.shape}, y_val_8: {y_val_8.shape}")
print(f"Shape of X_test_3: {X_test_3.shape}, y_test_3: {y_test_3.shape}")
print(f"Shape of X_test_6: {X_test_6.shape}, y_test_6: {y_test_6.shape}")
print(f"Shape of X_test_8: {X_test_8.shape}, y_test_8: {y_test_8.shape}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
from torch import nn

def train_and_evaluate_transformer_3_24(params, train_loader_3, val_loader_3, device, save_best_only=True, checkpoint_filename="transformer_model_3_24.pth.tar", patience=10, scaler=None):
    checkpoint_dir = "./checkpoints"
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)

    checkpoint_filepath = os.path.join(checkpoint_dir, checkpoint_filename)

    # Increase the model complexity by tuning d_model, d_ff, and num_heads if needed
    model = TimeSeriesTransformer(
        input_size=1,
        d_model=params['d_model'],
        n_heads=params['n_heads'],
        n_layers=params['n_layers'],
        d_ff=params['d_ff'],
        dropout=params['dropout'],
        output_size=1,  # Adjust according to your data
        seq_len=params['seq_len'],
        label_len=params['label_len'],
        out_len=params['out_len']
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])
    criterion = nn.MSELoss()

    best_val_loss = float('inf')
    best_rmse = float('inf')
    best_mape = float('inf')
    num_epochs = 100  # Increased to allow better learning
    epochs_no_improve = 0
    early_stop = False

    train_losses = []
    val_losses = []

    # Training loop
    for epoch in range(num_epochs):
        if early_stop:
            print("Early stopping")
            break

        model.train()
        train_loss = 0.0
        for batch_X, batch_y in train_loader_3:
            optimizer.zero_grad()
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            x_enc = batch_X
            x_dec = batch_X[:, -params['label_len']:, :]
            output = model(x_enc, x_dec)

            batch_y = batch_y[:, :output.shape[1]]

            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader_3)
        train_losses.append(train_loss)

        # Validation loop
        model.eval()
        val_loss = 0.0
        val_true, val_pred = [], []
        with torch.no_grad():
            for batch_X, batch_y in val_loader_3:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                x_enc = batch_X
                x_dec = batch_X[:, -params['label_len']:, :]
                output = model(x_enc, x_dec)

                batch_y = batch_y[:, :output.shape[1]]

                val_true.append(batch_y.cpu().numpy())
                val_pred.append(output.cpu().numpy())
                loss = criterion(output, batch_y)
                val_loss += loss.item()

        val_loss /= len(val_loader_3)
        val_losses.append(val_loss)

        # Flatten predictions for RMSE and MAPE
        val_true_flat = np.concatenate(val_true, axis=0).reshape(-1, 1)
        val_pred_flat = np.concatenate(val_pred, axis=0).reshape(-1, 1)
        
        if scaler is not None:
            val_true_flat = scaler.inverse_transform(val_true_flat)
            val_pred_flat = scaler.inverse_transform(val_pred_flat)

        # Compute RMSE and MAPE
        rmse, mape = calculate_metrics(val_true_flat, val_pred_flat)
        
        print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.4f}")

        # Save checkpoint if validation loss improves
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse
            best_mape = mape
            epochs_no_improve = 0
            if save_best_only:
                save_checkpoint(model, optimizer, epoch, val_loss, filename=checkpoint_filepath)
                print(f"Saving checkpoint at epoch {epoch} with validation loss {val_loss:.4f}")
        else:
            epochs_no_improve += 1
        
        # Early stopping check
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            early_stop = True

    # Plot train and validation losses
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Train Loss and Validation Loss Over Epochs')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Return metrics
    return best_val_loss, best_rmse, best_mape


In [ ]:
def random_search_transformer_3_24(param_grid, train_loader_3, val_loader_3, device, num_iter=10):
    best_val_loss = float('inf')
    best_rmse = float('inf')
    best_mape = float('inf')
    best_params = None

    keys = list(param_grid.keys())
    evaluated_params = set()

    for i in range(num_iter):
        params = {key: random.choice(param_grid[key]) for key in keys}
        params_tuple = tuple(sorted(params.items()))
        if params_tuple in evaluated_params:
            continue

        evaluated_params.add(params_tuple)
        print(f"Evaluating with params: {params}")

        val_loss, rmse, mape = train_and_evaluate_transformer_3_24(
            params=params, 
            train_loader_3=train_loader_3, 
            val_loader_3=val_loader_3, 
            device=device
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse
            best_mape = mape
            best_params = params

    print(f"\nBest Validation Loss: {best_val_loss:.4f}")
    print(f"Best RMSE: {best_rmse:.4f}")
    print(f"Best MAPE: {best_mape:.4f}")
    print(f"Best Hyperparameters: {best_params}")

    return best_params


In [ ]:
from torch.utils.data import DataLoader, TensorDataset
import torch

# Step 1: Correct DataLoader Setup
# Ensure that each dataset has exactly two tensors: inputs (X) and targets (y).
train_loader_3 = DataLoader(
    TensorDataset(torch.tensor(X_train_3, dtype=torch.float32), torch.tensor(y_train_3, dtype=torch.float32)),
    batch_size=64,
    shuffle=True
)
val_loader_3 = DataLoader(
    TensorDataset(torch.tensor(X_val_3, dtype=torch.float32), torch.tensor(y_val_3, dtype=torch.float32)),
    batch_size=64,
    shuffle=False
)

# Step 2: Test DataLoader to Verify Output Structure
for batch_X, batch_y in train_loader_3:
    print("Input batch shape:", batch_X.shape)  # Should match (batch_size, seq_len, input_size)
    print("Target batch shape:", batch_y.shape)  # Should match (batch_size, forecast_horizon, output_size)
    break  # Only check the first batch to confirm structure


In [ ]:
# 2. Define Hyperparameter Grid
param_grid = {
    'd_model': [256],
    'n_heads': [4],
    'n_layers': [3],
    'd_ff': [512],
    'dropout': [0.3],
    'seq_len': [16],  # Based on the create_windows_3_24 function
    'label_len': [15],  # Set to the size of labels used in your function
    'out_len': [224],  # Forecast horizon size
    'learning_rate': [0.001]
}
# 3. Specify Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 4. Run Random Search
best_params = random_search_transformer_3_24(param_grid, train_loader_3, val_loader_3, device)

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Define your model (Autoformer) and optimizer
model = TimeSeriesTransformer(
    input_size=1,           # Number of input features, set to 1 if univariate
    d_model=256,            # Model dimensions
    n_heads=4,              # Number of attention heads
    n_layers=3,             # Number of transformer layers
    d_ff=512,               # Dimension of feed-forward network
    dropout=0.3,            # Dropout rate
    output_size=1,          # Forecasted output dimensions
    seq_len=16,             # Length of the input sequence
    label_len=15,           # Length of label sequence (for the decoder input)
    out_len=224             # Length of the output sequence
).to(device)

# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Step 1: Load model checkpoint
checkpoint_path = './checkpoints/transformer_model_3_24.pth.tar'

def load_checkpoint(model, optimizer, checkpoint_path):
    if checkpoint_path is not None:
        print(f"Loading checkpoint from '{checkpoint_path}'...")
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        epoch = checkpoint.get('epoch', None)
        best_val_loss = checkpoint.get('best_val_loss', None)
        print(f"Checkpoint loaded. Resuming from epoch {epoch}")
        return epoch, best_val_loss
    else:
        print("No checkpoint provided, starting from scratch.")
        return None, None

# Load the checkpoint
epoch, best_val_loss = load_checkpoint(model, optimizer, checkpoint_path)

# Step 2: Generate predictions using the test data
model.eval()  # Switch the model to evaluation mode

predictions = []
with torch.no_grad():
    for batch_X, batch_y in test_loader_3:  # test_loader_8 is your DataLoader for the test set
        batch_X = batch_X.to(device)
        x_enc = batch_X
        x_dec = batch_X[:, -15:, :]  # Last 15 time steps for decoder input
        
        pred = model(x_enc, x_dec)
        predictions.append(pred[:, -1, :].cpu().numpy())

# Step 3: Concatenate predictions into a single array
predictions = np.concatenate(predictions, axis=0)

# Step 4: Ensure predictions match the size of the test set
pred_len = len(predictions)
test_data_sliced = test_data_3_months.iloc[:pred_len]

# Step 5: Convert predictions to DataFrame for plotting
predictions_df = pd.DataFrame(predictions, index=test_data_sliced.index, columns=['predictions'])

# Step 6: Filter data to start from December
start_date = '2023-12-20'
train_data_filtered = train_data_3_months[train_data_3_months.index >= start_date]
val_data_filtered = val_data_3_months[val_data_3_months.index >= start_date]
test_data_filtered = test_data_sliced[test_data_sliced.index >= start_date]
predictions_filtered = predictions_df[predictions_df.index >= start_date]

# Step 7: Calculate RMSE, MAPE, and residuals
actual_values = test_data_filtered['Hs'].values
predicted_values = predictions_filtered['predictions'].values

# Calculate RMSE and MAPE
rmse = mean_squared_error(actual_values, predicted_values, squared=False)
mape = mean_absolute_percentage_error(actual_values, predicted_values)

# Calculate residuals and standard deviation for confidence intervals
residuals = actual_values - predicted_values
std_dev = np.std(residuals)
conf_interval = 1.96 * std_dev  # 95% confidence interval

# Print RMSE and MAPE
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape * 100:.2f}%")

# Step 8: Plot predictions along with train, validation, test data, and confidence intervals
fig, ax = plt.subplots(figsize=(10, 5))

train_data_filtered.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=2)
val_data_filtered.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=2)
test_data_filtered.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=2)
predictions_filtered.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Plot confidence intervals
upper_bound = predictions_filtered['predictions'] + conf_interval
lower_bound = predictions_filtered['predictions'] - conf_interval
ax.fill_between(predictions_filtered.index, lower_bound, upper_bound, color='green', alpha=0.3, label="95% Confidence Interval")

# Labels, legend, and grid
ax.set_xlabel('Datetime')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Train, Validation, Test, and Predictions from December Onwards\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Assuming predictions, actual test data, and confidence interval calculations are already in place

# Step 1: Define the zoom-in start date to focus on the test data and predictions
zoom_start_date = '2023-12-08'  # Adjust this date to focus on the test and prediction period

# Step 2: Filter data to zoom-in on the specified range
train_data_zoomed = train_data_3_months[train_data_3_months.index >= zoom_start_date]
val_data_zoomed = val_data_3_months[val_data_3_months.index >= zoom_start_date]
test_data_zoomed = test_data_sliced[test_data_sliced.index >= zoom_start_date]
predictions_zoomed = predictions_df[predictions_df.index >= zoom_start_date]

# Step 3: Plot zoomed-in view
fig, ax = plt.subplots(figsize=(12, 6))

# Plot zoomed train data
train_data_zoomed.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=1.5)

# Plot zoomed validation data
val_data_zoomed.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=1.5)

# Plot zoomed test data
test_data_zoomed.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=1.5)

# Plot zoomed model predictions
predictions_zoomed.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Plot confidence intervals for the predictions
upper_bound_zoomed = predictions_zoomed['predictions'] + conf_interval
lower_bound_zoomed = predictions_zoomed['predictions'] - conf_interval
ax.fill_between(predictions_zoomed.index, lower_bound_zoomed, upper_bound_zoomed, color='green', alpha=0.2, label="95% Confidence Interval")

# Step 4: Add labels, title, and legend
ax.set_xlabel('Date')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Zoomed-in View of Test Data and Predictions with Confidence Interval\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()

# Show grid and plot
plt.grid(True)
plt.show()

#### Train 3 Months - 48 Hours

In [ ]:
# Step 5: Create a window-based dataset using scaled_data (24 jam)
def create_windows_3_48(data, window_size=32, forecast_horizon=224):
    X, y = [], []
    for i in range(len(data) - window_size - forecast_horizon + 1):
        X.append(data.iloc[i: i + window_size].values)
        y.append(data.iloc[i + window_size: i + window_size + forecast_horizon].values)
    return np.array(X), np.array(y)

In [ ]:
# Buat dataset berbasis window untuk setiap skenario (24 jam)
X_train_3, y_train_3 = create_windows_3_48(train_data_3_months)
X_train_6, y_train_6 = create_windows_3_48(train_data_6_months)
X_train_8, y_train_8 = create_windows_3_48(train_data_8_months)

X_val_3, y_val_3 = create_windows_3_48(val_data_3_months)
X_val_6, y_val_6 = create_windows_3_48(val_data_6_months)
X_val_8, y_val_8 = create_windows_3_48(val_data_8_months)

X_test_3, y_test_3 = create_windows_3_48(test_data_3_months)
X_test_6, y_test_6 = create_windows_3_48(test_data_6_months)
X_test_8, y_test_8 = create_windows_3_48(test_data_8_months)

# Cek hasilnya untuk melihat ukuran setiap dataset
print(f"Shape of X_train_3: {X_train_3.shape}, y_train_3: {y_train_3.shape}")
print(f"Shape of X_train_6: {X_train_6.shape}, y_train_6: {y_train_6.shape}")
print(f"Shape of X_train_8: {X_train_8.shape}, y_train_8: {y_train_8.shape}")
print(f"Shape of X_val_3: {X_val_3.shape}, y_val_3: {y_val_3.shape}")
print(f"Shape of X_val_6: {X_val_6.shape}, y_val_6: {y_val_6.shape}")
print(f"Shape of X_val_8: {X_val_8.shape}, y_val_8: {y_val_8.shape}")
print(f"Shape of X_test_3: {X_test_3.shape}, y_test_3: {y_test_3.shape}")
print(f"Shape of X_test_6: {X_test_6.shape}, y_test_6: {y_test_6.shape}")
print(f"Shape of X_test_8: {X_test_8.shape}, y_test_8: {y_test_8.shape}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
from torch import nn

def train_and_evaluate_transformer_3_48(params, train_loader_3, val_loader_3, device, save_best_only=True, checkpoint_filename="transformer_model_3_48.pth.tar", patience=10, scaler=None):
    checkpoint_dir = "./checkpoints"
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)

    checkpoint_filepath = os.path.join(checkpoint_dir, checkpoint_filename)

    # Increase the model complexity by tuning d_model, d_ff, and num_heads if needed
    model = TimeSeriesTransformer(
        input_size=1,
        d_model=params['d_model'],
        n_heads=params['n_heads'],
        n_layers=params['n_layers'],
        d_ff=params['d_ff'],
        dropout=params['dropout'],
        output_size=1,  # Adjust according to your data
        seq_len=params['seq_len'],
        label_len=params['label_len'],
        out_len=params['out_len']
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])
    criterion = nn.MSELoss()

    best_val_loss = float('inf')
    best_rmse = float('inf')
    best_mape = float('inf')
    num_epochs = 100  # Increased to allow better learning
    epochs_no_improve = 0
    early_stop = False

    train_losses = []
    val_losses = []

    # Training loop
    for epoch in range(num_epochs):
        if early_stop:
            print("Early stopping")
            break

        model.train()
        train_loss = 0.0
        for batch_X, batch_y in train_loader_3:
            optimizer.zero_grad()
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            x_enc = batch_X
            x_dec = batch_X[:, -params['label_len']:, :]
            output = model(x_enc, x_dec)

            batch_y = batch_y[:, :output.shape[1]]

            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader_3)
        train_losses.append(train_loss)

        # Validation loop
        model.eval()
        val_loss = 0.0
        val_true, val_pred = [], []
        with torch.no_grad():
            for batch_X, batch_y in val_loader_3:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                x_enc = batch_X
                x_dec = batch_X[:, -params['label_len']:, :]
                output = model(x_enc, x_dec)

                batch_y = batch_y[:, :output.shape[1]]

                val_true.append(batch_y.cpu().numpy())
                val_pred.append(output.cpu().numpy())
                loss = criterion(output, batch_y)
                val_loss += loss.item()

        val_loss /= len(val_loader_3)
        val_losses.append(val_loss)

        # Flatten predictions for RMSE and MAPE
        val_true_flat = np.concatenate(val_true, axis=0).reshape(-1, 1)
        val_pred_flat = np.concatenate(val_pred, axis=0).reshape(-1, 1)
        
        if scaler is not None:
            val_true_flat = scaler.inverse_transform(val_true_flat)
            val_pred_flat = scaler.inverse_transform(val_pred_flat)

        # Compute RMSE and MAPE
        rmse, mape = calculate_metrics(val_true_flat, val_pred_flat)
        
        print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.4f}")

        # Save checkpoint if validation loss improves
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse
            best_mape = mape
            epochs_no_improve = 0
            if save_best_only:
                save_checkpoint(model, optimizer, epoch, val_loss, filename=checkpoint_filepath)
                print(f"Saving checkpoint at epoch {epoch} with validation loss {val_loss:.4f}")
        else:
            epochs_no_improve += 1
        
        # Early stopping check
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            early_stop = True

    # Plot train and validation losses
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Train Loss and Validation Loss Over Epochs')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Return metrics
    return best_val_loss, best_rmse, best_mape


In [ ]:
def random_search_transformer_3_48(param_grid, train_loader_3, val_loader_3, device, num_iter=10):
    best_val_loss = float('inf')
    best_rmse = float('inf')
    best_mape = float('inf')
    best_params = None

    keys = list(param_grid.keys())
    evaluated_params = set()

    for i in range(num_iter):
        params = {key: random.choice(param_grid[key]) for key in keys}
        params_tuple = tuple(sorted(params.items()))
        if params_tuple in evaluated_params:
            continue

        evaluated_params.add(params_tuple)
        print(f"Evaluating with params: {params}")

        val_loss, rmse, mape = train_and_evaluate_transformer_3_48(
            params=params, 
            train_loader_3=train_loader_3, 
            val_loader_3=val_loader_3, 
            device=device
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse
            best_mape = mape
            best_params = params

    print(f"\nBest Validation Loss: {best_val_loss:.4f}")
    print(f"Best RMSE: {best_rmse:.4f}")
    print(f"Best MAPE: {best_mape:.4f}")
    print(f"Best Hyperparameters: {best_params}")

    return best_params


In [ ]:
# 2. Define Hyperparameter Grid
param_grid = {
    'd_model': [256],
    'n_heads': [4],
    'n_layers': [3],
    'd_ff': [512],
    'dropout': [0.3],
    'seq_len': [32],  # Based on the create_windows_3_24 function
    'label_len': [15],  # Set to the size of labels used in your function
    'out_len': [224],  # Forecast horizon size
    'learning_rate': [0.001]
}
# 3. Specify Device
device = torch.device('mps' if torch.cuda.is_available() else 'cpu')

# 4. Run Random Search
best_params = random_search_transformer_3_48(param_grid, train_loader_3, val_loader_3, device)

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Define your model (Autoformer) and optimizer
model = TimeSeriesTransformer(
    input_size=1,           # Number of input features, set to 1 if univariate
    d_model=256,            # Model dimensions
    n_heads=4,              # Number of attention heads
    n_layers=3,             # Number of transformer layers
    d_ff=512,               # Dimension of feed-forward network
    dropout=0.3,            # Dropout rate
    output_size=1,          # Forecasted output dimensions
    seq_len=32,             # Length of the input sequence
    label_len=15,           # Length of label sequence (for the decoder input)
    out_len=224             # Length of the output sequence
).to(device)

# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Step 1: Load model checkpoint
checkpoint_path = './checkpoints/transformer_model_3_48.pth.tar'

def load_checkpoint(model, optimizer, checkpoint_path):
    if checkpoint_path is not None:
        print(f"Loading checkpoint from '{checkpoint_path}'...")
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        epoch = checkpoint.get('epoch', None)
        best_val_loss = checkpoint.get('best_val_loss', None)
        print(f"Checkpoint loaded. Resuming from epoch {epoch}")
        return epoch, best_val_loss
    else:
        print("No checkpoint provided, starting from scratch.")
        return None, None

# Load the checkpoint
epoch, best_val_loss = load_checkpoint(model, optimizer, checkpoint_path)

# Step 2: Generate predictions using the test data
model.eval()  # Switch the model to evaluation mode

predictions = []
with torch.no_grad():
    for batch_X, batch_y in test_loader_3:  # test_loader_8 is your DataLoader for the test set
        batch_X = batch_X.to(device)
        x_enc = batch_X
        x_dec = batch_X[:, -15:, :]  # Last 15 time steps for decoder input
        
        pred = model(x_enc, x_dec)
        predictions.append(pred[:, -1, :].cpu().numpy())

# Step 3: Concatenate predictions into a single array
predictions = np.concatenate(predictions, axis=0)

# Step 4: Ensure predictions match the size of the test set
pred_len = len(predictions)
test_data_sliced = test_data_3_months.iloc[:pred_len]

# Step 5: Convert predictions to DataFrame for plotting
predictions_df = pd.DataFrame(predictions, index=test_data_sliced.index, columns=['predictions'])

# Step 6: Filter data to start from December
start_date = '2023-12-20'
train_data_filtered = train_data_3_months[train_data_3_months.index >= start_date]
val_data_filtered = val_data_3_months[val_data_3_months.index >= start_date]
test_data_filtered = test_data_sliced[test_data_sliced.index >= start_date]
predictions_filtered = predictions_df[predictions_df.index >= start_date]

# Step 7: Calculate RMSE, MAPE, and residuals
actual_values = test_data_filtered['Hs'].values
predicted_values = predictions_filtered['predictions'].values

# Calculate RMSE and MAPE
rmse = mean_squared_error(actual_values, predicted_values, squared=False)
mape = mean_absolute_percentage_error(actual_values, predicted_values)

# Calculate residuals and standard deviation for confidence intervals
residuals = actual_values - predicted_values
std_dev = np.std(residuals)
conf_interval = 1.96 * std_dev  # 95% confidence interval

# Print RMSE and MAPE
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape * 100:.2f}%")

# Step 8: Plot predictions along with train, validation, test data, and confidence intervals
fig, ax = plt.subplots(figsize=(10, 5))

train_data_filtered.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=2)
val_data_filtered.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=2)
test_data_filtered.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=2)
predictions_filtered.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Plot confidence intervals
upper_bound = predictions_filtered['predictions'] + conf_interval
lower_bound = predictions_filtered['predictions'] - conf_interval
ax.fill_between(predictions_filtered.index, lower_bound, upper_bound, color='green', alpha=0.3, label="95% Confidence Interval")

# Labels, legend, and grid
ax.set_xlabel('Datetime')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Train, Validation, Test, and Predictions from December Onwards\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Assuming predictions, actual test data, and confidence interval calculations are already in place

# Step 1: Define the zoom-in start date to focus on the test data and predictions
zoom_start_date = '2023-12-20'  # Adjust this date to focus on the test and prediction period

# Step 2: Filter data to zoom-in on the specified range
train_data_zoomed = train_data_3_months[train_data_3_months.index >= zoom_start_date]
val_data_zoomed = val_data_3_months[val_data_3_months.index >= zoom_start_date]
test_data_zoomed = test_data_sliced[test_data_sliced.index >= zoom_start_date]
predictions_zoomed = predictions_df[predictions_df.index >= zoom_start_date]

# Step 3: Plot zoomed-in view
fig, ax = plt.subplots(figsize=(12, 6))

# Plot zoomed train data
train_data_zoomed.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=1.5)

# Plot zoomed validation data
val_data_zoomed.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=1.5)

# Plot zoomed test data
test_data_zoomed.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=1.5)

# Plot zoomed model predictions
predictions_zoomed.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Plot confidence intervals for the predictions
upper_bound_zoomed = predictions_zoomed['predictions'] + conf_interval
lower_bound_zoomed = predictions_zoomed['predictions'] - conf_interval
ax.fill_between(predictions_zoomed.index, lower_bound_zoomed, upper_bound_zoomed, color='green', alpha=0.2, label="95% Confidence Interval")

# Step 4: Add labels, title, and legend
ax.set_xlabel('Date')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Zoomed-in View of Test Data and Predictions with Confidence Interval\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()

# Show grid and plot
plt.grid(True)
plt.show()

#### Train data 3 Months - 96 Hours

In [ ]:
# Step 5: Create a window-based dataset using scaled_data (24 jam)
def create_windows_3_96(data, window_size=64, forecast_horizon=224):
    X, y = [], []
    for i in range(len(data) - window_size - forecast_horizon + 1):
        X.append(data.iloc[i: i + window_size].values)
        y.append(data.iloc[i + window_size: i + window_size + forecast_horizon].values)
    return np.array(X), np.array(y)

In [ ]:
# Buat dataset berbasis window untuk setiap skenario (24 jam)
X_train_3, y_train_3 = create_windows_3_96(train_data_3_months)
X_train_6, y_train_6 = create_windows_3_96(train_data_6_months)
X_train_8, y_train_8 = create_windows_3_96(train_data_8_months)

X_val_3, y_val_3 = create_windows_3_96(val_data_3_months)
X_val_6, y_val_6 = create_windows_3_96(val_data_6_months)
X_val_8, y_val_8 = create_windows_3_96(val_data_8_months)

X_test_3, y_test_3 = create_windows_3_96(test_data_3_months)
X_test_6, y_test_6 = create_windows_3_96(test_data_6_months)
X_test_8, y_test_8 = create_windows_3_96(test_data_8_months)

# Cek hasilnya untuk melihat ukuran setiap dataset
print(f"Shape of X_train_3: {X_train_3.shape}, y_train_3: {y_train_3.shape}")
print(f"Shape of X_train_6: {X_train_6.shape}, y_train_6: {y_train_6.shape}")
print(f"Shape of X_train_8: {X_train_8.shape}, y_train_8: {y_train_8.shape}")
print(f"Shape of X_val_3: {X_val_3.shape}, y_val_3: {y_val_3.shape}")
print(f"Shape of X_val_6: {X_val_6.shape}, y_val_6: {y_val_6.shape}")
print(f"Shape of X_val_8: {X_val_8.shape}, y_val_8: {y_val_8.shape}")
print(f"Shape of X_test_3: {X_test_3.shape}, y_test_3: {y_test_3.shape}")
print(f"Shape of X_test_6: {X_test_6.shape}, y_test_6: {y_test_6.shape}")
print(f"Shape of X_test_8: {X_test_8.shape}, y_test_8: {y_test_8.shape}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
from torch import nn

def train_and_evaluate_transformer_3_96(params, train_loader_3, val_loader_3, device, save_best_only=True, checkpoint_filename="transformer_model_3_96.pth.tar", patience=10, scaler=None):
    checkpoint_dir = "./checkpoints"
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)

    checkpoint_filepath = os.path.join(checkpoint_dir, checkpoint_filename)

    # Increase the model complexity by tuning d_model, d_ff, and num_heads if needed
    model = TimeSeriesTransformer(
        input_size=1,
        d_model=params['d_model'],
        n_heads=params['n_heads'],
        n_layers=params['n_layers'],
        d_ff=params['d_ff'],
        dropout=params['dropout'],
        output_size=1,  # Adjust according to your data
        seq_len=params['seq_len'],
        label_len=params['label_len'],
        out_len=params['out_len']
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])
    criterion = nn.MSELoss()

    best_val_loss = float('inf')
    best_rmse = float('inf')
    best_mape = float('inf')
    num_epochs = 100  # Increased to allow better learning
    epochs_no_improve = 0
    early_stop = False

    train_losses = []
    val_losses = []

    # Training loop
    for epoch in range(num_epochs):
        if early_stop:
            print("Early stopping")
            break

        model.train()
        train_loss = 0.0
        for batch_X, batch_y in train_loader_3:
            optimizer.zero_grad()
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            x_enc = batch_X
            x_dec = batch_X[:, -params['label_len']:, :]
            output = model(x_enc, x_dec)

            batch_y = batch_y[:, :output.shape[1]]

            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader_3)
        train_losses.append(train_loss)

        # Validation loop
        model.eval()
        val_loss = 0.0
        val_true, val_pred = [], []
        with torch.no_grad():
            for batch_X, batch_y in val_loader_3:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                x_enc = batch_X
                x_dec = batch_X[:, -params['label_len']:, :]
                output = model(x_enc, x_dec)

                batch_y = batch_y[:, :output.shape[1]]

                val_true.append(batch_y.cpu().numpy())
                val_pred.append(output.cpu().numpy())
                loss = criterion(output, batch_y)
                val_loss += loss.item()

        val_loss /= len(val_loader_3)
        val_losses.append(val_loss)

        # Flatten predictions for RMSE and MAPE
        val_true_flat = np.concatenate(val_true, axis=0).reshape(-1, 1)
        val_pred_flat = np.concatenate(val_pred, axis=0).reshape(-1, 1)
        
        if scaler is not None:
            val_true_flat = scaler.inverse_transform(val_true_flat)
            val_pred_flat = scaler.inverse_transform(val_pred_flat)

        # Compute RMSE and MAPE
        rmse, mape = calculate_metrics(val_true_flat, val_pred_flat)
        
        print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.4f}")

        # Save checkpoint if validation loss improves
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse
            best_mape = mape
            epochs_no_improve = 0
            if save_best_only:
                save_checkpoint(model, optimizer, epoch, val_loss, filename=checkpoint_filepath)
                print(f"Saving checkpoint at epoch {epoch} with validation loss {val_loss:.4f}")
        else:
            epochs_no_improve += 1
        
        # Early stopping check
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            early_stop = True

    # Plot train and validation losses
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Train Loss and Validation Loss Over Epochs')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Return metrics
    return best_val_loss, best_rmse, best_mape


In [ ]:
def random_search_transformer_3_96(param_grid, train_loader_3, val_loader_3, device, num_iter=10):
    best_val_loss = float('inf')
    best_rmse = float('inf')
    best_mape = float('inf')
    best_params = None

    keys = list(param_grid.keys())
    evaluated_params = set()

    for i in range(num_iter):
        params = {key: random.choice(param_grid[key]) for key in keys}
        params_tuple = tuple(sorted(params.items()))
        if params_tuple in evaluated_params:
            continue

        evaluated_params.add(params_tuple)
        print(f"Evaluating with params: {params}")

        val_loss, rmse, mape = train_and_evaluate_transformer_3_96(
            params=params, 
            train_loader_3=train_loader_3, 
            val_loader_3=val_loader_3, 
            device=device
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse
            best_mape = mape
            best_params = params

    print(f"\nBest Validation Loss: {best_val_loss:.4f}")
    print(f"Best RMSE: {best_rmse:.4f}")
    print(f"Best MAPE: {best_mape:.4f}")
    print(f"Best Hyperparameters: {best_params}")

    return best_params


In [ ]:
# 2. Define Hyperparameter Grid
param_grid = {
    'd_model': [256],
    'n_heads': [4],
    'n_layers': [3],
    'd_ff': [512],
    'dropout': [0.3],
    'seq_len': [64],  # Based on the create_windows_3_24 function
    'label_len': [15],  # Set to the size of labels used in your function
    'out_len': [224],  # Forecast horizon size
    'learning_rate': [0.001]
}
# 3. Specify Device
device = torch.device('mps' if torch.cuda.is_available() else 'cpu')

# 4. Run Random Search
best_params = random_search_transformer_3_96(param_grid, train_loader_3, val_loader_3, device)

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Define your model (Autoformer) and optimizer
model = TimeSeriesTransformer(
    input_size=1,           # Number of input features, set to 1 if univariate
    d_model=256,            # Model dimensions
    n_heads=4,              # Number of attention heads
    n_layers=3,             # Number of transformer layers
    d_ff=512,               # Dimension of feed-forward network
    dropout=0.3,            # Dropout rate
    output_size=1,          # Forecasted output dimensions
    seq_len=64,             # Length of the input sequence
    label_len=15,           # Length of label sequence (for the decoder input)
    out_len=224             # Length of the output sequence
).to(device)

# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Step 1: Load model checkpoint
checkpoint_path = './checkpoints/transformer_model_3_96.pth.tar'

def load_checkpoint(model, optimizer, checkpoint_path):
    if checkpoint_path is not None:
        print(f"Loading checkpoint from '{checkpoint_path}'...")
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        epoch = checkpoint.get('epoch', None)
        best_val_loss = checkpoint.get('best_val_loss', None)
        print(f"Checkpoint loaded. Resuming from epoch {epoch}")
        return epoch, best_val_loss
    else:
        print("No checkpoint provided, starting from scratch.")
        return None, None

# Load the checkpoint
epoch, best_val_loss = load_checkpoint(model, optimizer, checkpoint_path)

# Step 2: Generate predictions using the test data
model.eval()  # Switch the model to evaluation mode

predictions = []
with torch.no_grad():
    for batch_X, batch_y in test_loader_3:  # test_loader_8 is your DataLoader for the test set
        batch_X = batch_X.to(device)
        x_enc = batch_X
        x_dec = batch_X[:, -15:, :]  # Last 15 time steps for decoder input
        
        pred = model(x_enc, x_dec)
        predictions.append(pred[:, -1, :].cpu().numpy())

# Step 3: Concatenate predictions into a single array
predictions = np.concatenate(predictions, axis=0)

# Step 4: Ensure predictions match the size of the test set
pred_len = len(predictions)
test_data_sliced = test_data_3_months.iloc[:pred_len]

# Step 5: Convert predictions to DataFrame for plotting
predictions_df = pd.DataFrame(predictions, index=test_data_sliced.index, columns=['predictions'])

# Step 6: Filter data to start from December
start_date = '2023-12-20'
train_data_filtered = train_data_3_months[train_data_3_months.index >= start_date]
val_data_filtered = val_data_3_months[val_data_3_months.index >= start_date]
test_data_filtered = test_data_sliced[test_data_sliced.index >= start_date]
predictions_filtered = predictions_df[predictions_df.index >= start_date]

# Step 7: Calculate RMSE, MAPE, and residuals
actual_values = test_data_filtered['Hs'].values
predicted_values = predictions_filtered['predictions'].values

# Calculate RMSE and MAPE
rmse = mean_squared_error(actual_values, predicted_values, squared=False)
mape = mean_absolute_percentage_error(actual_values, predicted_values)

# Calculate residuals and standard deviation for confidence intervals
residuals = actual_values - predicted_values
std_dev = np.std(residuals)
conf_interval = 1.96 * std_dev  # 95% confidence interval

# Print RMSE and MAPE
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape * 100:.2f}%")

# Step 8: Plot predictions along with train, validation, test data, and confidence intervals
fig, ax = plt.subplots(figsize=(10, 5))

train_data_filtered.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=2)
val_data_filtered.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=2)
test_data_filtered.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=2)
predictions_filtered.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Plot confidence intervals
upper_bound = predictions_filtered['predictions'] + conf_interval
lower_bound = predictions_filtered['predictions'] - conf_interval
ax.fill_between(predictions_filtered.index, lower_bound, upper_bound, color='green', alpha=0.3, label="95% Confidence Interval")

# Labels, legend, and grid
ax.set_xlabel('Datetime')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Train, Validation, Test, and Predictions from December Onwards\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Assuming predictions, actual test data, and confidence interval calculations are already in place

# Step 1: Define the zoom-in start date to focus on the test data and predictions
zoom_start_date = '2023-12-25'  # Adjust this date to focus on the test and prediction period

# Step 2: Filter data to zoom-in on the specified range
train_data_zoomed = train_data_3_months[train_data_3_months.index >= zoom_start_date]
val_data_zoomed = val_data_3_months[val_data_3_months.index >= zoom_start_date]
test_data_zoomed = test_data_sliced[test_data_sliced.index >= zoom_start_date]
predictions_zoomed = predictions_df[predictions_df.index >= zoom_start_date]

# Step 3: Plot zoomed-in view
fig, ax = plt.subplots(figsize=(12, 6))

# Plot zoomed train data
train_data_zoomed.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=1.5)

# Plot zoomed validation data
val_data_zoomed.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=1.5)

# Plot zoomed test data
test_data_zoomed.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=1.5)

# Plot zoomed model predictions
predictions_zoomed.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Plot confidence intervals for the predictions
upper_bound_zoomed = predictions_zoomed['predictions'] + conf_interval
lower_bound_zoomed = predictions_zoomed['predictions'] - conf_interval
ax.fill_between(predictions_zoomed.index, lower_bound_zoomed, upper_bound_zoomed, color='green', alpha=0.2, label="95% Confidence Interval")

# Step 4: Add labels, title, and legend
ax.set_xlabel('Date')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Zoomed-in View of Test Data and Predictions with Confidence Interval\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()

# Show grid and plot
plt.grid(True)
plt.show()

#### Train data 6 Months - 24 Hours

In [ ]:
# Step 5: Create a window-based dataset using scaled_data (24 jam)
def create_windows_6_24(data, window_size=16, forecast_horizon=224):
    X, y = [], []
    for i in range(len(data) - window_size - forecast_horizon + 1):
        X.append(data.iloc[i: i + window_size].values)
        y.append(data.iloc[i + window_size: i + window_size + forecast_horizon].values)
    return np.array(X), np.array(y)

In [ ]:
# Buat dataset berbasis window untuk setiap skenario (24 jam)
X_train_3, y_train_3 = create_windows_6_24(train_data_3_months)
X_train_6, y_train_6 = create_windows_6_24(train_data_6_months)
X_train_8, y_train_8 = create_windows_6_24(train_data_8_months)

X_val_3, y_val_3 = create_windows_6_24(val_data_3_months)
X_val_6, y_val_6 = create_windows_6_24(val_data_6_months)
X_val_8, y_val_8 = create_windows_6_24(val_data_8_months)

X_test_3, y_test_3 = create_windows_6_24(test_data_3_months)
X_test_6, y_test_6 = create_windows_6_24(test_data_6_months)
X_test_8, y_test_8 = create_windows_6_24(test_data_8_months)

# Cek hasilnya untuk melihat ukuran setiap dataset
print(f"Shape of X_train_3: {X_train_3.shape}, y_train_3: {y_train_3.shape}")
print(f"Shape of X_train_6: {X_train_6.shape}, y_train_6: {y_train_6.shape}")
print(f"Shape of X_train_8: {X_train_8.shape}, y_train_8: {y_train_8.shape}")
print(f"Shape of X_val_3: {X_val_3.shape}, y_val_3: {y_val_3.shape}")
print(f"Shape of X_val_6: {X_val_6.shape}, y_val_6: {y_val_6.shape}")
print(f"Shape of X_val_8: {X_val_8.shape}, y_val_8: {y_val_8.shape}")
print(f"Shape of X_test_3: {X_test_3.shape}, y_test_3: {y_test_3.shape}")
print(f"Shape of X_test_6: {X_test_6.shape}, y_test_6: {y_test_6.shape}")
print(f"Shape of X_test_8: {X_test_8.shape}, y_test_8: {y_test_8.shape}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
from torch import nn

def train_and_evaluate_transformer_6_24(params, train_loader_6, val_loader_6, device, save_best_only=True, checkpoint_filename="transformer_model_6_24.pth.tar", patience=10, scaler=None):
    checkpoint_dir = "./checkpoints"
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)

    checkpoint_filepath = os.path.join(checkpoint_dir, checkpoint_filename)

    # Increase the model complexity by tuning d_model, d_ff, and num_heads if needed
    model = TimeSeriesTransformer(
        input_size=1,
        d_model=params['d_model'],
        n_heads=params['n_heads'],
        n_layers=params['n_layers'],
        d_ff=params['d_ff'],
        dropout=params['dropout'],
        output_size=1,  # Adjust according to your data
        seq_len=params['seq_len'],
        label_len=params['label_len'],
        out_len=params['out_len']
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])
    criterion = nn.MSELoss()

    best_val_loss = float('inf')
    best_rmse = float('inf')
    best_mape = float('inf')
    num_epochs = 100  # Increased to allow better learning
    epochs_no_improve = 0
    early_stop = False

    train_losses = []
    val_losses = []

    # Training loop
    for epoch in range(num_epochs):
        if early_stop:
            print("Early stopping")
            break

        model.train()
        train_loss = 0.0
        for batch_X, batch_y in train_loader_6:
            optimizer.zero_grad()
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            x_enc = batch_X
            x_dec = batch_X[:, -params['label_len']:, :]
            output = model(x_enc, x_dec)

            batch_y = batch_y[:, :output.shape[1]]

            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader_6)
        train_losses.append(train_loss)

        # Validation loop
        model.eval()
        val_loss = 0.0
        val_true, val_pred = [], []
        with torch.no_grad():
            for batch_X, batch_y in val_loader_6:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                x_enc = batch_X
                x_dec = batch_X[:, -params['label_len']:, :]
                output = model(x_enc, x_dec)

                batch_y = batch_y[:, :output.shape[1]]

                val_true.append(batch_y.cpu().numpy())
                val_pred.append(output.cpu().numpy())
                loss = criterion(output, batch_y)
                val_loss += loss.item()

        val_loss /= len(val_loader_6)
        val_losses.append(val_loss)

        # Flatten predictions for RMSE and MAPE
        val_true_flat = np.concatenate(val_true, axis=0).reshape(-1, 1)
        val_pred_flat = np.concatenate(val_pred, axis=0).reshape(-1, 1)
        
        if scaler is not None:
            val_true_flat = scaler.inverse_transform(val_true_flat)
            val_pred_flat = scaler.inverse_transform(val_pred_flat)

        # Compute RMSE and MAPE
        rmse, mape = calculate_metrics(val_true_flat, val_pred_flat)
        
        print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.4f}")

        # Save checkpoint if validation loss improves
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse
            best_mape = mape
            epochs_no_improve = 0
            if save_best_only:
                save_checkpoint(model, optimizer, epoch, val_loss, filename=checkpoint_filepath)
                print(f"Saving checkpoint at epoch {epoch} with validation loss {val_loss:.4f}")
        else:
            epochs_no_improve += 1
        
        # Early stopping check
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            early_stop = True

    # Plot train and validation losses
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Train Loss and Validation Loss Over Epochs')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Return metrics
    return best_val_loss, best_rmse, best_mape


In [ ]:
def random_search_transformer_6_24(param_grid, train_loader_6, val_loader_6, device, num_iter=10):
    best_val_loss = float('inf')
    best_rmse = float('inf')
    best_mape = float('inf')
    best_params = None

    keys = list(param_grid.keys())
    evaluated_params = set()

    for i in range(num_iter):
        params = {key: random.choice(param_grid[key]) for key in keys}
        params_tuple = tuple(sorted(params.items()))
        if params_tuple in evaluated_params:
            continue

        evaluated_params.add(params_tuple)
        print(f"Evaluating with params: {params}")

        val_loss, rmse, mape = train_and_evaluate_transformer_6_24(
            params=params, 
            train_loader_6=train_loader_6, 
            val_loader_6=val_loader_6, 
            device=device
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse
            best_mape = mape
            best_params = params

    print(f"\nBest Validation Loss: {best_val_loss:.4f}")
    print(f"Best RMSE: {best_rmse:.4f}")
    print(f"Best MAPE: {best_mape:.4f}")
    print(f"Best Hyperparameters: {best_params}")

    return best_params


In [ ]:
# 2. Define Hyperparameter Grid
param_grid = {
    'd_model': [256],
    'n_heads': [4],
    'n_layers': [3],
    'd_ff': [512],
    'dropout': [0.3],
    'seq_len': [16],  # Based on the create_windows_3_24 function
    'label_len': [15],  # Set to the size of labels used in your function
    'out_len': [224],  # Forecast horizon size
    'learning_rate': [0.001]
}
# 3. Specify Device
device = torch.device('mps' if torch.cuda.is_available() else 'cpu')

# 4. Run Random Search
best_params = random_search_transformer_6_24(param_grid, train_loader_6, val_loader_6, device)

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Define your model (Autoformer) and optimizer
model = TimeSeriesTransformer(
    input_size=1,           # Number of input features, set to 1 if univariate
    d_model=256,            # Model dimensions
    n_heads=4,              # Number of attention heads
    n_layers=3,             # Number of transformer layers
    d_ff=512,               # Dimension of feed-forward network
    dropout=0.3,            # Dropout rate
    output_size=1,          # Forecasted output dimensions
    seq_len=16,             # Length of the input sequence
    label_len=15,           # Length of label sequence (for the decoder input)
    out_len=224             # Length of the output sequence
).to(device)

# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Step 1: Load model checkpoint
checkpoint_path = './checkpoints/transformer_model_6_24.pth.tar'

def load_checkpoint(model, optimizer, checkpoint_path):
    if checkpoint_path is not None:
        print(f"Loading checkpoint from '{checkpoint_path}'...")
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        epoch = checkpoint.get('epoch', None)
        best_val_loss = checkpoint.get('best_val_loss', None)
        print(f"Checkpoint loaded. Resuming from epoch {epoch}")
        return epoch, best_val_loss
    else:
        print("No checkpoint provided, starting from scratch.")
        return None, None

# Load the checkpoint
epoch, best_val_loss = load_checkpoint(model, optimizer, checkpoint_path)

# Step 2: Generate predictions using the test data
model.eval()  # Switch the model to evaluation mode

predictions = []
with torch.no_grad():
    for batch_X, batch_y in test_loader_6:  # test_loader_8 is your DataLoader for the test set
        batch_X = batch_X.to(device)
        x_enc = batch_X
        x_dec = batch_X[:, -15:, :]  # Last 15 time steps for decoder input
        
        pred = model(x_enc, x_dec)
        predictions.append(pred[:, -1, :].cpu().numpy())

# Step 3: Concatenate predictions into a single array
predictions = np.concatenate(predictions, axis=0)

# Step 4: Ensure predictions match the size of the test set
pred_len = len(predictions)
test_data_sliced = test_data_6_months.iloc[:pred_len]

# Step 5: Convert predictions to DataFrame for plotting
predictions_df = pd.DataFrame(predictions, index=test_data_sliced.index, columns=['predictions'])

# Step 6: Filter data to start from December
start_date = '2023-12-20'
train_data_filtered = train_data_6_months[train_data_6_months.index >= start_date]
val_data_filtered = val_data_6_months[val_data_6_months.index >= start_date]
test_data_filtered = test_data_sliced[test_data_sliced.index >= start_date]
predictions_filtered = predictions_df[predictions_df.index >= start_date]

# Step 7: Calculate RMSE, MAPE, and residuals
actual_values = test_data_filtered['Hs'].values
predicted_values = predictions_filtered['predictions'].values

# Calculate RMSE and MAPE
rmse = mean_squared_error(actual_values, predicted_values, squared=False)
mape = mean_absolute_percentage_error(actual_values, predicted_values)

# Calculate residuals and standard deviation for confidence intervals
residuals = actual_values - predicted_values
std_dev = np.std(residuals)
conf_interval = 1.96 * std_dev  # 95% confidence interval

# Print RMSE and MAPE
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape * 100:.2f}%")

# Step 8: Plot predictions along with train, validation, test data, and confidence intervals
fig, ax = plt.subplots(figsize=(10, 5))

train_data_filtered.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=2)
val_data_filtered.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=2)
test_data_filtered.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=2)
predictions_filtered.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Plot confidence intervals
upper_bound = predictions_filtered['predictions'] + conf_interval
lower_bound = predictions_filtered['predictions'] - conf_interval
ax.fill_between(predictions_filtered.index, lower_bound, upper_bound, color='green', alpha=0.3, label="95% Confidence Interval")

# Labels, legend, and grid
ax.set_xlabel('Datetime')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Train, Validation, Test, and Predictions from December Onwards\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()
plt.grid(True)
plt.show()


#### Train Data 6 Months - 48 Hours

In [ ]:
# Step 5: Create a window-based dataset using scaled_data (3 Months - 24 jam)
def create_windows_6_48(data, window_size=32, forecast_horizon=224):
    X, y = [], []
    for i in range(len(data) - window_size - forecast_horizon + 1):
        X.append(data.iloc[i: i + window_size].values)
        y.append(data.iloc[i + window_size: i + window_size + forecast_horizon].values)
    return np.array(X), np.array(y)

In [ ]:
# Buat dataset berbasis window untuk setiap skenario (24 jam)
X_train_3, y_train_3 = create_windows_6_48(train_data_3_months)
X_train_6, y_train_6 = create_windows_6_48(train_data_6_months)
X_train_8, y_train_8 = create_windows_6_48(train_data_8_months)

X_val_3, y_val_3 = create_windows_6_48(val_data_3_months)
X_val_6, y_val_6 = create_windows_6_48(val_data_6_months)
X_val_8, y_val_8 = create_windows_6_48(val_data_8_months)

X_test_3, y_test_3 = create_windows_6_48(test_data_3_months)
X_test_6, y_test_6 = create_windows_6_48(test_data_6_months)
X_test_8, y_test_8 = create_windows_6_48(test_data_8_months)

# Cek hasilnya untuk melihat ukuran setiap dataset
print(f"Shape of X_train_3: {X_train_3.shape}, y_train_3: {y_train_3.shape}")
print(f"Shape of X_train_6: {X_train_6.shape}, y_train_6: {y_train_6.shape}")
print(f"Shape of X_train_8: {X_train_8.shape}, y_train_8: {y_train_8.shape}")
print(f"Shape of X_val_3: {X_val_3.shape}, y_val_3: {y_val_3.shape}")
print(f"Shape of X_val_6: {X_val_6.shape}, y_val_6: {y_val_6.shape}")
print(f"Shape of X_val_8: {X_val_8.shape}, y_val_8: {y_val_8.shape}")
print(f"Shape of X_test_3: {X_test_3.shape}, y_test_3: {y_test_3.shape}")
print(f"Shape of X_test_6: {X_test_6.shape}, y_test_6: {y_test_6.shape}")
print(f"Shape of X_test_8: {X_test_8.shape}, y_test_8: {y_test_8.shape}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
from torch import nn

def train_and_evaluate_transformer_6_48(params, train_loader_6, val_loader_6, device, save_best_only=True, checkpoint_filename="transformer_model_6_48.pth.tar", patience=10, scaler=None):
    checkpoint_dir = "./checkpoints"
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)

    checkpoint_filepath = os.path.join(checkpoint_dir, checkpoint_filename)

    # Increase the model complexity by tuning d_model, d_ff, and num_heads if needed
    model = TimeSeriesTransformer(
        input_size=1,
        d_model=params['d_model'],
        n_heads=params['n_heads'],
        n_layers=params['n_layers'],
        d_ff=params['d_ff'],
        dropout=params['dropout'],
        output_size=1,  # Adjust according to your data
        seq_len=params['seq_len'],
        label_len=params['label_len'],
        out_len=params['out_len']
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])
    criterion = nn.MSELoss()

    best_val_loss = float('inf')
    best_rmse = float('inf')
    best_mape = float('inf')
    num_epochs = 100  # Increased to allow better learning
    epochs_no_improve = 0
    early_stop = False

    train_losses = []
    val_losses = []

    # Training loop
    for epoch in range(num_epochs):
        if early_stop:
            print("Early stopping")
            break

        model.train()
        train_loss = 0.0
        for batch_X, batch_y in train_loader_6:
            optimizer.zero_grad()
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            x_enc = batch_X
            x_dec = batch_X[:, -params['label_len']:, :]
            output = model(x_enc, x_dec)

            batch_y = batch_y[:, :output.shape[1]]

            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader_6)
        train_losses.append(train_loss)

        # Validation loop
        model.eval()
        val_loss = 0.0
        val_true, val_pred = [], []
        with torch.no_grad():
            for batch_X, batch_y in val_loader_6:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                x_enc = batch_X
                x_dec = batch_X[:, -params['label_len']:, :]
                output = model(x_enc, x_dec)

                batch_y = batch_y[:, :output.shape[1]]

                val_true.append(batch_y.cpu().numpy())
                val_pred.append(output.cpu().numpy())
                loss = criterion(output, batch_y)
                val_loss += loss.item()

        val_loss /= len(val_loader_6)
        val_losses.append(val_loss)

        # Flatten predictions for RMSE and MAPE
        val_true_flat = np.concatenate(val_true, axis=0).reshape(-1, 1)
        val_pred_flat = np.concatenate(val_pred, axis=0).reshape(-1, 1)
        
        if scaler is not None:
            val_true_flat = scaler.inverse_transform(val_true_flat)
            val_pred_flat = scaler.inverse_transform(val_pred_flat)

        # Compute RMSE and MAPE
        rmse, mape = calculate_metrics(val_true_flat, val_pred_flat)
        
        print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.4f}")

        # Save checkpoint if validation loss improves
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse
            best_mape = mape
            epochs_no_improve = 0
            if save_best_only:
                save_checkpoint(model, optimizer, epoch, val_loss, filename=checkpoint_filepath)
                print(f"Saving checkpoint at epoch {epoch} with validation loss {val_loss:.4f}")
        else:
            epochs_no_improve += 1
        
        # Early stopping check
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            early_stop = True

    # Plot train and validation losses
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Train Loss and Validation Loss Over Epochs')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Return metrics
    return best_val_loss, best_rmse, best_mape


In [ ]:
def random_search_transformer_6_48(param_grid, train_loader_6, val_loader_6, device, num_iter=10):
    best_val_loss = float('inf')
    best_rmse = float('inf')
    best_mape = float('inf')
    best_params = None

    keys = list(param_grid.keys())
    evaluated_params = set()

    for i in range(num_iter):
        params = {key: random.choice(param_grid[key]) for key in keys}
        params_tuple = tuple(sorted(params.items()))
        if params_tuple in evaluated_params:
            continue

        evaluated_params.add(params_tuple)
        print(f"Evaluating with params: {params}")

        val_loss, rmse, mape = train_and_evaluate_transformer_6_48(
            params=params, 
            train_loader_6=train_loader_6, 
            val_loader_6=val_loader_6, 
            device=device
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_rmse = rmse
            best_mape = mape
            best_params = params

    print(f"\nBest Validation Loss: {best_val_loss:.4f}")
    print(f"Best RMSE: {best_rmse:.4f}")
    print(f"Best MAPE: {best_mape:.4f}")
    print(f"Best Hyperparameters: {best_params}")

    return best_params


In [ ]:
# 2. Define Hyperparameter Grid
param_grid = {
    'd_model': [256],
    'n_heads': [4],
    'n_layers': [3],
    'd_ff': [512],
    'dropout': [0.3],
    'seq_len': [32],  # Based on the create_windows_3_24 function
    'label_len': [15],  # Set to the size of labels used in your function
    'out_len': [224],  # Forecast horizon size
    'learning_rate': [0.001]
}
# 3. Specify Device
device = torch.device('mps' if torch.cuda.is_available() else 'cpu')

# 4. Run Random Search
best_params = random_search_transformer_6_48(param_grid, train_loader_6, val_loader_6, device)

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Define your model (Autoformer) and optimizer
model = TimeSeriesTransformer(
    input_size=1,           # Number of input features, set to 1 if univariate
    d_model=256,            # Model dimensions
    n_heads=4,              # Number of attention heads
    n_layers=3,             # Number of transformer layers
    d_ff=512,               # Dimension of feed-forward network
    dropout=0.3,            # Dropout rate
    output_size=1,          # Forecasted output dimensions
    seq_len=32,             # Length of the input sequence
    label_len=15,           # Length of label sequence (for the decoder input)
    out_len=224             # Length of the output sequence
).to(device)

# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Step 1: Load model checkpoint
checkpoint_path = './checkpoints/transformer_model_6_48.pth.tar'

def load_checkpoint(model, optimizer, checkpoint_path):
    if checkpoint_path is not None:
        print(f"Loading checkpoint from '{checkpoint_path}'...")
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        epoch = checkpoint.get('epoch', None)
        best_val_loss = checkpoint.get('best_val_loss', None)
        print(f"Checkpoint loaded. Resuming from epoch {epoch}")
        return epoch, best_val_loss
    else:
        print("No checkpoint provided, starting from scratch.")
        return None, None

# Load the checkpoint
epoch, best_val_loss = load_checkpoint(model, optimizer, checkpoint_path)

# Step 2: Generate predictions using the test data
model.eval()  # Switch the model to evaluation mode

predictions = []
with torch.no_grad():
    for batch_X, batch_y in test_loader_6:  # test_loader_8 is your DataLoader for the test set
        batch_X = batch_X.to(device)
        x_enc = batch_X
        x_dec = batch_X[:, -15:, :]  # Last 15 time steps for decoder input
        
        pred = model(x_enc, x_dec)
        predictions.append(pred[:, -1, :].cpu().numpy())

# Step 3: Concatenate predictions into a single array
predictions = np.concatenate(predictions, axis=0)

# Step 4: Ensure predictions match the size of the test set
pred_len = len(predictions)
test_data_sliced = test_data_6_months.iloc[:pred_len]

# Step 5: Convert predictions to DataFrame for plotting
predictions_df = pd.DataFrame(predictions, index=test_data_sliced.index, columns=['predictions'])

# Step 6: Filter data to start from December
start_date = '2023-12-20'
train_data_filtered = train_data_6_months[train_data_6_months.index >= start_date]
val_data_filtered = val_data_6_months[val_data_6_months.index >= start_date]
test_data_filtered = test_data_sliced[test_data_sliced.index >= start_date]
predictions_filtered = predictions_df[predictions_df.index >= start_date]

# Step 7: Calculate RMSE, MAPE, and residuals
actual_values = test_data_filtered['Hs'].values
predicted_values = predictions_filtered['predictions'].values

# Calculate RMSE and MAPE
rmse = mean_squared_error(actual_values, predicted_values, squared=False)
mape = mean_absolute_percentage_error(actual_values, predicted_values)

# Calculate residuals and standard deviation for confidence intervals
residuals = actual_values - predicted_values
std_dev = np.std(residuals)
conf_interval = 1.96 * std_dev  # 95% confidence interval

# Print RMSE and MAPE
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape * 100:.2f}%")

# Step 8: Plot predictions along with train, validation, test data, and confidence intervals
fig, ax = plt.subplots(figsize=(10, 5))

train_data_filtered.plot(ax=ax, label='Train Data', color='blue', linestyle='-', linewidth=2)
val_data_filtered.plot(ax=ax, label='Validation Data', color='purple', linestyle='--', linewidth=2)
test_data_filtered.plot(ax=ax, label='Test Data', color='orange', linestyle=':', linewidth=2)
predictions_filtered.plot(ax=ax, label='Predictions', color='green', linestyle='-', linewidth=2)

# Plot confidence intervals
upper_bound = predictions_filtered['predictions'] + conf_interval
lower_bound = predictions_filtered['predictions'] - conf_interval
ax.fill_between(predictions_filtered.index, lower_bound, upper_bound, color='green', alpha=0.3, label="95% Confidence Interval")

# Labels, legend, and grid
ax.set_xlabel('Datetime')
ax.set_ylabel('Scaled Wave Height (Hs)')
ax.set_title(f'Train, Validation, Test, and Predictions from December Onwards\nRMSE: {rmse:.4f}, MAPE: {mape * 100:.2f}%')
ax.legend()
plt.grid(True)
plt.show()


#### Train Data 6 Months - 96 Hours

In [ ]:
# Step 5: Create a window-based dataset using scaled_data (3 Months - 24 jam)
def create_windows_6_96(data, window_size=32, forecast_horizon=224):
    X, y = [], []
    for i in range(len(data) - window_size - forecast_horizon + 1):
        X.append(data.iloc[i: i + window_size].values)
        y.append(data.iloc[i + window_size: i + window_size + forecast_horizon].values)
    return np.array(X), np.array(y)

In [ ]:
# Buat dataset berbasis window untuk setiap skenario (24 jam)
X_train_3, y_train_3 = create_windows_6_96(train_data_3_months)
X_train_6, y_train_6 = create_windows_6_96(train_data_6_months)
X_train_8, y_train_8 = create_windows_6_96(train_data_8_months)

X_val_3, y_val_3 = create_windows_6_96(val_data_3_months)
X_val_6, y_val_6 = create_windows_6_96(val_data_6_months)
X_val_8, y_val_8 = create_windows_6_96(val_data_8_months)

X_test_3, y_test_3 = create_windows_6_96(test_data_3_months)
X_test_6, y_test_6 = create_windows_6_96(test_data_6_months)
X_test_8, y_test_8 = create_windows_6_96(test_data_8_months)

# Cek hasilnya untuk melihat ukuran setiap dataset
print(f"Shape of X_train_3: {X_train_3.shape}, y_train_3: {y_train_3.shape}")
print(f"Shape of X_train_6: {X_train_6.shape}, y_train_6: {y_train_6.shape}")
print(f"Shape of X_train_8: {X_train_8.shape}, y_train_8: {y_train_8.shape}")
print(f"Shape of X_val_3: {X_val_3.shape}, y_val_3: {y_val_3.shape}")
print(f"Shape of X_val_6: {X_val_6.shape}, y_val_6: {y_val_6.shape}")
print(f"Shape of X_val_8: {X_val_8.shape}, y_val_8: {y_val_8.shape}")
print(f"Shape of X_test_3: {X_test_3.shape}, y_test_3: {y_test_3.shape}")
print(f"Shape of X_test_6: {X_test_6.shape}, y_test_6: {y_test_6.shape}")
print(f"Shape of X_test_8: {X_test_8.shape}, y_test_8: {y_test_8.shape}")


In [ ]:
# Step 5: Create a window-based dataset using scaled_data (3 Months - 24 jam)
def create_windows_xgboost3_24(data, window_size=16, forecast_horizon=224):
    X, y = [], []
    for i in range(len(data) - window_size - forecast_horizon + 1):
        X.append(data.iloc[i: i + window_size].values)
        y.append(data.iloc[i + window_size: i + window_size + forecast_horizon].values)
    return np.array(X), np.array(y)

In [ ]:
# Buat dataset berbasis window untuk setiap skenario (24 jam)
X_train_3, y_train_3 = create_windows_xgboost3_24(train_data_3_months)
X_train_6, y_train_6 = create_windows_xgboost3_24(train_data_6_months)
X_train_8, y_train_8 = create_windows_xgboost3_24(train_data_8_months)

X_val_3, y_val_3 = create_windows_xgboost3_24(val_data_3_months)
X_val_6, y_val_6 = create_windows_xgboost3_24(val_data_6_months)
X_val_8, y_val_8 = create_windows_xgboost3_24(val_data_8_months)

X_test_3, y_test_3 = create_windows_xgboost3_24(test_data_3_months)
X_test_6, y_test_6 = create_windows_xgboost3_24(test_data_6_months)
X_test_8, y_test_8 = create_windows_xgboost3_24(test_data_8_months)

# Cek hasilnya untuk melihat ukuran setiap dataset
print(f"Shape of X_train_3: {X_train_3.shape}, y_train_3: {y_train_3.shape}")
print(f"Shape of X_train_6: {X_train_6.shape}, y_train_6: {y_train_6.shape}")
print(f"Shape of X_train_8: {X_train_8.shape}, y_train_8: {y_train_8.shape}")
print(f"Shape of X_val_3: {X_val_3.shape}, y_val_3: {y_val_3.shape}")
print(f"Shape of X_val_6: {X_val_6.shape}, y_val_6: {y_val_6.shape}")
print(f"Shape of X_val_8: {X_val_8.shape}, y_val_8: {y_val_8.shape}")
print(f"Shape of X_test_3: {X_test_3.shape}, y_test_3: {y_test_3.shape}")
print(f"Shape of X_test_6: {X_test_6.shape}, y_test_6: {y_test_6.shape}")
print(f"Shape of X_test_8: {X_test_8.shape}, y_test_8: {y_test_8.shape}")


In [ ]:
# Reshape data to 2D [num_samples, sequence_length * num_features]
X_train_reshaped = X_train_3.reshape(X_train_3.shape[0], -1)
y_train_reshaped = y_train_3.reshape(-1)  # Flatten targets to 1D

X_val_reshaped = X_val_3.reshape(X_val_3.shape[0], -1)
y_val_reshaped = y_val_3.reshape(-1)

X_test_reshaped = X_test_3.reshape(X_test_3.shape[0], -1)
y_test_reshaped = y_test_3.reshape(-1)

In [ ]:
from sklearn.model_selection import KFold
import itertools

# Define a parameter grid for tuning
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200, 300],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0],
    'gamma': [0, 0.1, 0.2]
}

# Flatten parameter grid for random search or exhaustive grid search
param_combinations = list(itertools.product(
    param_grid['max_depth'],
    param_grid['learning_rate'],
    param_grid['n_estimators'],
    param_grid['subsample'],
    param_grid['colsample_bytree'],
    param_grid['gamma']
))

# Function for cross-validation with XGBoost on multiple parameter configurations
def cross_val_xgboost(X_train, y_train, param_combinations, n_splits=5, scaler=None):
    best_rmse = float('inf')
    best_params = None

    # Perform cross-validation on each parameter combination
    for params in param_combinations:
        current_params = {
            'max_depth': params[0],
            'learning_rate': params[1],
            'n_estimators': params[2],
            'subsample': params[3],
            'colsample_bytree': params[4],
            'gamma': params[5]
        }
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
        rmse_scores = []
        mape_scores = []
        
        print(f"Evaluating parameters: {current_params}")

        for train_index, val_index in kf.split(X_train):
            X_train_cv, X_val_cv = X_train[train_index], X_train[val_index]
            y_train_cv, y_val_cv = y_train[train_index], y_train[val_index]
            
            model = xgb.XGBRegressor(**current_params, random_state=42)
            model.fit(X_train_cv, y_train_cv)
            
            y_val_pred = model.predict(X_val_cv)
            
            # If scaling was applied, inverse transform predictions and true values
            if scaler is not None:
                y_val_cv = scaler.inverse_transform(y_val_cv.reshape(-1, 1)).flatten()
                y_val_pred = scaler.inverse_transform(y_val_pred.reshape(-1, 1)).flatten()

            rmse = np.sqrt(mean_squared_error(y_val_cv, y_val_pred))
            mape = mean_absolute_percentage_error(y_val_cv, y_val_pred)

            rmse_scores.append(rmse)
            mape_scores.append(mape)
        
        avg_rmse = np.mean(rmse_scores)
        avg_mape = np.mean(mape_scores) * 100  # Convert to percentage

        print(f"Average RMSE: {avg_rmse:.4f}, Average MAPE: {avg_mape:.2f}% for parameters {current_params}")
        
        # Save the best parameters based on RMSE
        if avg_rmse < best_rmse:
            best_rmse = avg_rmse
            best_params = current_params

    print(f"\nBest RMSE: {best_rmse:.4f} with parameters: {best_params}")
    return best_params

# Run cross-validation to find the best parameters
best_params = cross_val_xgboost(X_train_reshaped, y_train_3, param_combinations)

# Train the best model with these parameters
best_model, val_rmse, val_mape = train_and_evaluate_single_xgboost(
    best_params, X_train_reshaped, y_train_3, X_val_reshaped, y_val_3
)


In [ ]:
# Evaluate the model on the test set
def evaluate_on_test_set(model, X_test, y_test, scaler=None):
    # Make predictions
    y_test_pred = model.predict(X_test)
    
    # Inverse scaling if a scaler is provided
    if scaler is not None:
        y_test = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
        y_test_pred = scaler.inverse_transform(y_test_pred.reshape(-1, 1)).flatten()
    
    # Calculate RMSE and MAPE
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    test_mape = mean_absolute_percentage_error(y_test, y_test_pred)
    
    print(f"Test RMSE: {test_rmse:.4f}, Test MAPE: {test_mape * 100:.2f}%")
    return test_rmse, test_mape

# Run evaluation on the test set with the best model
test_rmse, test_mape = evaluate_on_test_set(best_model, X_test_reshaped, y_test_3)

# Optional: Visualize predictions vs true values
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(y_test_3, label='True Values')
plt.plot(best_model.predict(X_test_reshaped), label='Predicted Values', alpha=0.7)
plt.xlabel("Samples")
plt.ylabel("Target Value")
plt.title("Predicted vs True Values on Test Set")
plt.legend()
plt.show()

# Optional: Save the best model
import joblib

joblib.dump(best_model, 'best_xgboost_model.pkl')
print("Best model saved as 'best_xgboost_model.pkl'")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Pastikan data actual_14_days tersedia (nilai aktual untuk 14 hari ke depan)
# Misalnya: actual_14_days = [nilai1, nilai2, ..., nilai14]

# Fungsi untuk melakukan forecasting iteratif 14 hari ke depan
def forecast_14_days(model, initial_input, forecast_horizon=14, scaler=None):
    predictions = []

    # Mulai dari input terakhir
    current_input = initial_input

    for _ in range(forecast_horizon):
        # Prediksi satu langkah ke depan
        prediction = model.predict(current_input.reshape(1, -1))
        
        # Simpan prediksi
        predictions.append(prediction[0])

        # Update input dengan menambahkan prediksi terbaru
        current_input = np.append(current_input[1:], prediction[0])

    # Jika scaler digunakan, kembalikan ke nilai aslinya
    if scaler is not None:
        predictions = scaler.inverse_transform(np.array(predictions).reshape(-1, 1)).flatten()

    return np.array(predictions)

# Ambil input terakhir dari data uji untuk prediksi 14 hari ke depan
initial_input = X_test_reshaped[-1]  # Mengambil window terakhir dari data uji

# Forecasting 14 hari ke depan
predicted_14_days = forecast_14_days(best_model, initial_input, forecast_horizon=14, scaler=scaler)

# Jika Anda memiliki data aktual 14 hari ke depan, masukkan di sini
# Contoh: actual_14_days = np.array([...])  # Data aktual untuk 14 hari ke depan

# Plot perbandingan prediksi dan nilai aktual
plt.figure(figsize=(12, 6))
plt.plot(range(1, 15), predicted_14_days, label="Predicted Values", marker='o')
try:
    plt.plot(range(1, 15), actual_14_days, label="Actual Values", marker='o')
except NameError:
    print("Warning: 'actual_14_days' not defined. Only predicted values will be shown.")
plt.xlabel("Days")
plt.ylabel("Target Value")
plt.title("14-Day Forecast: Predicted vs Actual Values")
plt.legend()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Misalkan kita memiliki daftar tanggal
dates = pd.date_range(start="2024-08-01", periods=50, freq="D")

# Data target yang sebenarnya untuk periode tertentu (ganti dengan data aktual)
target_values = np.random.rand(50) * 0.08  # Contoh, ganti dengan nilai aktual

# Prediksi median (50%) dan kuantil 10% dan 90% (interval kepercayaan)
predicted_median = np.random.rand(50) * 0.08  # Hasil prediksi median (contoh prediksi utama)
predicted_quantile_10 = predicted_median - np.random.rand(50) * 0.02  # Contoh kuantil 10%
predicted_quantile_90 = predicted_median + np.random.rand(50) * 0.02  # Contoh kuantil 90%

# Plot target aktual
plt.figure(figsize=(10, 6))
plt.plot(dates, target_values, label="Target", color="blue")

# Plot prediksi median
plt.plot(dates, predicted_median, label="Predicted Median", color="green")

# Plot interval kepercayaan
plt.fill_between(
    dates,
    predicted_quantile_10,
    predicted_quantile_90,
    color="green",
    alpha=0.3,
    label="Confidence Interval (0.1 - 0.9)"
)

# Pengaturan tambahan untuk grafik
plt.xlabel("Date")
plt.ylabel("Target Value")
plt.title("Predicted vs Actual with Confidence Intervals")
plt.legend()
plt.xticks(rotation=45)  # Memiringkan tanggal agar mudah dibaca
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Memuat data (misalkan Anda memiliki data hingga 15 Februari 2024)
dates = pd.date_range(start="2024-02-16", periods=50, freq="D")  # Contoh tanggal prediksi untuk 50 hari ke depan

# Data target aktual untuk periode tertentu (ganti dengan data aktual jika ada)
target_values = np.random.rand(50) * 0.08  # Contoh, ganti dengan nilai target aktual

# Prediksi median (50%) dan kuantil 10% dan 90% (interval kepercayaan)
# Contoh data prediksi, pastikan menggantinya dengan data prediksi sebenarnya
predicted_median = np.random.rand(50) * 0.08  # Contoh prediksi median
predicted_quantile_10 = predicted_median - np.random.rand(50) * 0.02  # Interval 10%
predicted_quantile_90 = predicted_median + np.random.rand(50) * 0.02  # Interval 90%

# Plot nilai target aktual
plt.figure(figsize=(10, 6))
plt.plot(dates, target_values, label="Target", color="blue")

# Plot prediksi median
plt.plot(dates, predicted_median, label="Predicted Median", color="green")

# Plot interval kepercayaan
plt.fill_between(
    dates,
    predicted_quantile_10,
    predicted_quantile_90,
    color="green",
    alpha=0.3,
    label="Confidence Interval (0.1 - 0.9)"
)

# Pengaturan tambahan untuk grafik
plt.xlabel("Date")
plt.ylabel("Target Value")
plt.title("Predicted vs Actual with Confidence Intervals")
plt.legend()
plt.xticks(rotation=45)  # Memiringkan tanggal agar mudah dibaca
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

np.random.seed(42)  # Angka 42 bisa diganti dengan nilai lain

# Tanggal untuk 14 hari ke depan dari 16 Februari 2024
dates = pd.date_range(start="2024-02-15", periods=14, freq="D")

# Data target aktual untuk 14 hari (ganti dengan data aktual jika ada)
target_values = np.random.rand(14) * 0.08  # Contoh, ganti dengan nilai target aktual

# Prediksi median (50%) dan kuantil 10% dan 90% (interval kepercayaan)
predicted_median = np.random.rand(14) * 0.08  # Contoh prediksi median
predicted_quantile_10 = predicted_median - np.random.rand(14) * 0.02  # Interval 10%
predicted_quantile_90 = predicted_median + np.random.rand(14) * 0.02  # Interval 90%

# Plot data target aktual
plt.figure(figsize=(10, 6))
plt.plot(dates, target_values, label="Target", color="blue")  # Menggunakan data target asli

# Plot prediksi median
plt.plot(dates, predicted_median, label="Predicted Median", color="green")  # Menggunakan hasil prediksi median dari model

# Plot interval kepercayaan (quantile)
plt.fill_between(
    dates,
    predicted_quantile_10,  # Hasil prediksi kuantil 10% dari model
    predicted_quantile_90,  # Hasil prediksi kuantil 90% dari model
    color="green",
    alpha=0.3,
    label="Confidence Interval (0.1 - 0.9)"
)

# Pengaturan tambahan untuk grafik
plt.xlabel("Date")
plt.ylabel("Target Value")
plt.title("14-Day Forecast: Predicted vs Actual with Confidence Intervals")
plt.legend()
plt.xticks(rotation=45)  # Memiringkan tanggal agar mudah dibaca
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)  # Angka 42 bisa diganti dengan nilai lain

# Contoh data sintetis untuk ilustrasi (misalkan dari 1 Januari hingga 15 Februari 2024)
dates = pd.date_range(start="2024-01-01", end="2024-02-15", freq="D")
data_values = np.random.rand(len(dates)) * 0.08  # Contoh nilai target, ganti dengan data aktual
data = pd.DataFrame({'date': dates, 'y': data_values})

# Pisahkan data menjadi bagian pelatihan dan target (14 hari terakhir sebagai target)
train_data = data.iloc[:-14]  # Data pelatihan tanpa 14 hari terakhir
target_data = data.iloc[-14:]  # 14 hari terakhir sebagai data target

# Cetak untuk memeriksa apakah pembagian sudah benar
print("Train Data:")
print(train_data.tail())  # Cek beberapa baris terakhir dari data pelatihan
print("\nTarget Data:")
print(target_data)  # Cek data target 14 hari

# Gunakan target_data['y'] sebagai nilai target untuk validasi prediksi
target_values = target_data['y'].values  # Data target untuk 14 hari ke depan

# Misalnya, buat prediksi median dan interval kepercayaan dari model (disini menggunakan data acak untuk ilustrasi)
np.random.seed(42)  # Untuk hasil acak yang konsisten
predicted_median = target_values + (np.random.rand(14) - 0.5) * 0.02  # Contoh prediksi median
predicted_quantile_10 = predicted_median - 0.01  # Interval 10%
predicted_quantile_90 = predicted_median + 0.01  # Interval 90%

# Plot data target aktual dan prediksi untuk 14 hari ke depan
plt.figure(figsize=(10, 6))
plt.plot(target_data['date'], target_values, label="Target", color="blue")

# Plot prediksi median
plt.plot(target_data['date'], predicted_median, label="Predicted Median", color="green")

# Plot interval kepercayaan
plt.fill_between(
    target_data['date'],
    predicted_quantile_10,
    predicted_quantile_90,
    color="green",
    alpha=0.3,
    label="Confidence Interval (0.1 - 0.9)"
)

# Pengaturan tambahan untuk grafik
plt.xlabel("Date")
plt.ylabel("Target Value")
plt.title("14-Day Forecast: Predicted vs Actual with Confidence Intervals")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)  # Seed untuk hasil yang konsisten

# Data tanggal untuk 14 hari ke depan dari 16 Februari 2024
forecast_start_date = pd.to_datetime("2024-02-16")
forecast_dates = pd.date_range(start=forecast_start_date, periods=14, freq="D")

# Data terakhir pada training data untuk window input terakhir (16 elemen terakhir)
window_size = 16  # Sesuaikan dengan window size yang dipakai saat melatih model
last_window_data = train_data['y'].values[-window_size:]  # Mengambil window terakhir dari data pelatihan

# Fungsi untuk melakukan prediksi iteratif 14 hari ke depan
def forecast_14_days(model, initial_input, forecast_horizon=14):
    predictions = []
    current_input = initial_input.copy()  # Salin data window terakhir sebagai input awal

    for _ in range(forecast_horizon):
        # Prediksi satu langkah ke depan
        prediction = model.predict(current_input.reshape(1, -1))
        predictions.append(prediction[0])  # Simpan prediksi

        # Update input untuk langkah berikutnya (geser window, tapi tetap menjaga ukuran 16)
        current_input = np.append(current_input[1:], prediction)

    return np.array(predictions)

# Forecast 14 hari ke depan
predicted_median = forecast_14_days(best_model, last_window_data)

# Contoh pembuatan interval prediksi dengan asumsi +/- 0.01 dari median
predicted_quantile_10 = predicted_median - 0.01
predicted_quantile_90 = predicted_median + 0.01

# Visualisasi hasil prediksi
plt.figure(figsize=(10, 6))
plt.plot(forecast_dates, predicted_median, label="Predicted Median", color="green", marker="o")

# Plot interval kepercayaan
plt.fill_between(
    forecast_dates,
    predicted_quantile_10,
    predicted_quantile_90,
    color="green",
    alpha=0.3,
    label="Confidence Interval (0.1 - 0.9)"
)

# Pengaturan tambahan untuk grafik
plt.xlabel("Date")
plt.ylabel("Target Value")
plt.title("14-Day Forecast Starting from February 16, 2024")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Step 5: Create a window-based dataset using scaled_data (3 Months - 24 jam)
def create_windows_xgboost3_32(data, window_size=32, forecast_horizon=224):
    X, y = [], []
    for i in range(len(data) - window_size - forecast_horizon + 1):
        X.append(data.iloc[i: i + window_size].values)
        y.append(data.iloc[i + window_size: i + window_size + forecast_horizon].values)
    return np.array(X), np.array(y)

In [ ]:
# Buat dataset berbasis window untuk setiap skenario (24 jam)
X_train_3, y_train_3 = create_windows_xgboost3_32(train_data_3_months)
X_train_6, y_train_6 = create_windows_xgboost3_32(train_data_6_months)
X_train_8, y_train_8 = create_windows_xgboost3_32(train_data_8_months)

X_val_3, y_val_3 = create_windows_xgboost3_32(val_data_3_months)
X_val_6, y_val_6 = create_windows_xgboost3_32(val_data_6_months)
X_val_8, y_val_8 = create_windows_xgboost3_32(val_data_8_months)

X_test_3, y_test_3 = create_windows_xgboost3_32(test_data_3_months)
X_test_6, y_test_6 = create_windows_xgboost3_32(test_data_6_months)
X_test_8, y_test_8 = create_windows_xgboost3_32(test_data_8_months)

# Cek hasilnya untuk melihat ukuran setiap dataset
print(f"Shape of X_train_3: {X_train_3.shape}, y_train_3: {y_train_3.shape}")
print(f"Shape of X_train_6: {X_train_6.shape}, y_train_6: {y_train_6.shape}")
print(f"Shape of X_train_8: {X_train_8.shape}, y_train_8: {y_train_8.shape}")
print(f"Shape of X_val_3: {X_val_3.shape}, y_val_3: {y_val_3.shape}")
print(f"Shape of X_val_6: {X_val_6.shape}, y_val_6: {y_val_6.shape}")
print(f"Shape of X_val_8: {X_val_8.shape}, y_val_8: {y_val_8.shape}")
print(f"Shape of X_test_3: {X_test_3.shape}, y_test_3: {y_test_3.shape}")
print(f"Shape of X_test_6: {X_test_6.shape}, y_test_6: {y_test_6.shape}")
print(f"Shape of X_test_8: {X_test_8.shape}, y_test_8: {y_test_8.shape}")


In [ ]:
# Reshape y_train, y_val, and y_test to 2D
y_train_reshaped = y_train_3.reshape(y_train_3.shape[0], -1)  # Shape (num_samples, 224)
y_val_reshaped = y_val_3.reshape(y_val_3.shape[0], -1)
y_test_reshaped = y_test_3.reshape(y_test_3.shape[0], -1)

# Reshape X_train, X_val, and X_test to 2D
X_train_reshaped = X_train_3.reshape(X_train_3.shape[0], -1)  # Shape (num_samples, window_size)
X_val_reshaped = X_val_3.reshape(X_val_3.shape[0], -1)
X_test_reshaped = X_test_3.reshape(X_test_3.shape[0], -1)

In [ ]:
from sklearn.model_selection import KFold
import itertools
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Define a parameter grid for tuning
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200, 300],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0],
    'gamma': [0, 0.1, 0.2]
}

# Flatten parameter grid for random search or exhaustive grid search
param_combinations = list(itertools.product(
    param_grid['max_depth'],
    param_grid['learning_rate'],
    param_grid['n_estimators'],
    param_grid['subsample'],
    param_grid['colsample_bytree'],
    param_grid['gamma']
))

# Function for cross-validation with XGBoost on multiple parameter configurations
def cross_val_xgboost(X_train, y_train, param_combinations, n_splits=5, scaler=None):
    best_rmse = float('inf')
    best_params = None

    # Perform cross-validation on each parameter combination
    for params in param_combinations:
        current_params = {
            'max_depth': params[0],
            'learning_rate': params[1],
            'n_estimators': params[2],
            'subsample': params[3],
            'colsample_bytree': params[4],
            'gamma': params[5]
        }
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
        rmse_scores = []
        mape_scores = []
        
        print(f"Evaluating parameters: {current_params}")

        for train_index, val_index in kf.split(X_train):
            X_train_cv, X_val_cv = X_train[train_index], X_train[val_index]
            y_train_cv, y_val_cv = y_train[train_index], y_train[val_index]
            
            model = xgb.XGBRegressor(**current_params, random_state=42)
            model.fit(X_train_cv, y_train_cv)
            
            y_val_pred = model.predict(X_val_cv)
            
            # If scaling was applied, inverse transform predictions and true values
            if scaler is not None:
                y_val_cv = scaler.inverse_transform(y_val_cv.reshape(-1, 1)).flatten()
                y_val_pred = scaler.inverse_transform(y_val_pred.reshape(-1, 1)).flatten()

            rmse = np.sqrt(mean_squared_error(y_val_cv, y_val_pred))
            mape = mean_absolute_percentage_error(y_val_cv, y_val_pred)

            rmse_scores.append(rmse)
            mape_scores.append(mape)
        
        avg_rmse = np.mean(rmse_scores)
        avg_mape = np.mean(mape_scores) * 100  # Convert to percentage

        print(f"Average RMSE: {avg_rmse:.4f}, Average MAPE: {avg_mape:.2f}% for parameters {current_params}")
        
        # Save the best parameters based on RMSE
        if avg_rmse < best_rmse:
            best_rmse = avg_rmse
            best_params = current_params

    print(f"\nBest RMSE: {best_rmse:.4f} with parameters: {best_params}")
    return best_params

# Run cross-validation to find the best parameters
best_params = cross_val_xgboost(X_train_reshaped, y_train_reshaped, param_combinations)

# Function to train and evaluate single model with best parameters
def train_and_evaluate_single_xgboost(best_params, X_train, y_train, X_val, y_val):
    model = xgb.XGBRegressor(**best_params, random_state=42)
    model.fit(X_train, y_train)

    y_val_pred = model.predict(X_val)

    val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    val_mape = mean_absolute_percentage_error(y_val, y_val_pred) * 100  # Convert to percentage

    return model, val_rmse, val_mape

# Train the best model with these parameters
best_model, val_rmse, val_mape = train_and_evaluate_single_xgboost(
    best_params, X_train_reshaped, y_train_reshaped, X_val_reshaped, y_val_reshaped
)

print(f"Validation RMSE: {val_rmse:.4f}, Validation MAPE: {val_mape:.2f}%")

In [ ]:
print(f"Shape of y_val: {y_val_3.shape}")
print(f"Shape of y_val_pred: {y_val_pred_3.shape}")


In [ ]:
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error  # Import MAPE here
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
# Step 3: Train an XGBoost Model
# XGBoost Model with Updated Hyperparameters
xgboost_model = xgb.XGBRegressor(
    n_estimators=1500,
    learning_rate=0.005,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    early_stopping_rounds=15,
    eval_metric="rmse"
)

xgboost_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=True
)

# Evaluate on validation and test sets
y_val_pred = xgboost_model.predict(X_val)
y_test_pred = xgboost_model.predict(X_test)

rmse_val = mean_squared_error(y_val, y_val_pred, squared=False)
mape_val = mean_absolute_percentage_error(y_val, y_val_pred) * 100
rmse_test = mean_squared_error(y_test, y_test_pred, squared=False)
mape_test = mean_absolute_percentage_error(y_test, y_test_pred) * 100

print(f"Validation RMSE: {rmse_val:.4f}, Validation MAPE: {mape_val:.2f}%")
print(f"Test RMSE: {rmse_test:.4f}, Test MAPE: {mape_test:.2f}%")

# Step 6: Visualize Predictions vs Actual
# Convert predictions to DataFrames for easier visualization
y_test_pred_series = pd.Series(y_test_pred, index=test_data_3_months.index[-len(y_test):])

plt.figure(figsize=(12, 6))
plt.plot(test_data_3_months.index[-len(y_test):], y_test, label='Actual', color='orange')
plt.plot(y_test_pred_series.index, y_test_pred_series, label='Predicted', color='green')
plt.xlabel("Time")
plt.ylabel("Scaled Wave Height (Hs)")
plt.title("XGBoost Forecast: Actual vs Predicted")
plt.legend()
plt.grid()
plt.show()

In [ ]:
import xgboost as xgb
reg = xgb.XGBRegressor(n_estimators=1000)
reg.fit(X_train_3, y_train_3, verbose = False)